# Spur Review: notebook app design brief

**Status (18 November):** Design draft with a captured-data interactive prototype below. The new specification and prototype supersede earlier wireframes where they differ. Native App mode implementation and live collectors are not part of this design delivery. The Tech Utility visual direction remains an assumption for this draft.

## Brief lock

| Decision | Proposed answer |
|---|---|
| Surface | Desktop Spur Notebook App mode. A native frontend cell contains the workspace; notebook cells hold the inspectable evidence pipeline. |
| Audience | Open-source contributors, maintainers and reviewing agents working from a local repository plus GitHub. |
| Primary job | Understand a PR's changed behavior, likely impact, architectural fit and relevant people, then produce a review grounded in that evidence. |
| Initial repository and case | Local DuckDB checkout linked to duckdb/duckdb; previously captured PR #28731. |
| Tone | Precise engineering tool. Dense lists, readable diffs, explicit evidence states and concise explanations. |
| Brand direction | Proposed **Tech Utility** from Open Design: light neutral surfaces, one restrained green action accent, system sans, mono source names and revision IDs. |
| Scale | One repository and one pinned PR review first. Repository switching and a paginated GitHub inbox fit the model. No full-history ingest prerequisite. |
| AI role | On-demand reviewer over the selected immutable evidence packet. Findings are suggestions with citations, assumptions and validation steps. |
| First implementation target | Read-only workspace with actual captured evidence, shared selection and source-backed detail views. Local validation and GitHub refresh are explicit backend jobs; AI generation uses a configured provider. |

**Approval:** [ ] Approve this scope and direction   [ ] Revise below

**Changes requested:**

## Visual alternatives

- [ ] **Tech Utility (recommended):** engineering density, light surfaces, familiar diff/list conventions.
- [ ] Modern Minimal: quieter blue accent, lower initial density.
- [ ] Editorial Monocle: reading-focused review narratives and serif headings.
- [ ] Warm Soft: softer onboarding and document-style review.
- [ ] Brutalist Experimental: strong grid and oversized typography.

The app preserves evidence quality in text and shapes as well as color. Motion is limited to selection, expansion and job progress. Proposed design dials: variance 3/10, motion 2/10, density 8/10, following the dashboard surface with reduced motion.

The later design artifact makes the proposed experience concrete for review. Native frontend integration follows a separate implementation plan and validation loop.


## Product choice: one review workspace

We considered three shapes:

| Shape | Strength | Cost |
|---|---|---|
| **Diff-led review workspace (recommended)** | Fits the reviewer's first question: what changed? Every selected file or symbol carries into impact, architecture, people and AI. | Needs a shared selection and evidence contract. |
| Graph-first explorer | Good for open-ended repository learning and dependency exploration. | Starts reviewers in a large graph before they know the change; weak review completion path. |
| AI-first conversation | Fast entry for experienced agents. | Hides missing coverage and makes source comparison harder; requires repeated navigation back to evidence. |

The recommended app combines an explicit review path with optional graph exploration and contextual AI. AI review consumes the same evidence shown to the human.

### Information architecture

```text
Spur Review
  Repository switcher
  Setup                  local checkout, GitHub mapping, runtime and providers
  Inbox                  Issues | Pull requests | Saved reviews
  Review #28731
    Overview             intent, scope, source health, tests/checks
    Changes + Impact     file/symbol diff and typed propagation paths
    Architecture         component membership and alignment rules
    People               participants, owners and expertise candidates
    AI Review            cited findings, questions and validation plan
  Evidence               sources, snapshots, queries and export
```

The PR header, comparison revisions and selected file/symbol remain visible across every review lens. Repository-wide exploration has a separate scope label and never silently replaces PR scope.

## Journey 1: attach a local open-source repository

**User outcome:** connect an existing checkout and understand which analysis is available.

1. Select an existing local Git directory. Validate its canonical root and read access.
2. Detect remotes. Confirm upstream repository and optional fork separately. For the example: upstream duckdb/duckdb, contributor fork kevintruong/duckdb.
3. Show branch, HEAD, dirty status, shallow/partial-clone status and whether required PR objects are available.
4. Select analysis scope and exclusions, including generated/vendor files, submodules and supported languages.
5. Link a saved GitHub connection or continue with public/cached metadata. The app receives a credential reference, never a raw token in a browser control.
6. Check the actual notebook runtime and required extension contracts. New notebooks may default to another runtime, so DuckDB 2.0 is a checked prerequisite.
7. Validate providers and show source status. User starts a bounded initial collection or opens the captured example.

### Setup wireframe

```text
Repository setup                                              [Validate]
Local checkout     /Users/kevintruong/project/duckdb            [Choose]
Upstream           duckdb/duckdb                               [Confirm]
Contributor fork   kevintruong/duckdb                          [Optional]

Source health
Git objects        Available / missing commit objects
Working tree       Clean / modified (review uses pinned commits)
GitHub             Connected / public / offline / permission missing
Runtime            Required DuckDB 2.0 | actual selected runtime
Source extraction  Spur Graph contract and language coverage
Git history        DuckTails load + bounded history/blame probe
Graph queries      DuckPGQ load + grammar/query probe / SQL fallback

[Save repository]  [Open captured DuckDB example]
```

Saving configuration does not switch branches, stash edits or fetch an entire history. Missing objects have an explicit fetch action, target refs and progress. A dirty checkout remains usable for a pinned PR review; working-tree review is a separately named comparison.

## Journey 2: collect GitHub context and open a PR

**User outcome:** find relevant changes and freeze enough context for review.

The Inbox has Issues and Pull requests tabs, repository/state/label/author filters, search, a PR/issue URL input and saved reviews. Every collected resource records pagination, fetched-at time, permission errors and completeness. Cached results stay visible while refresh runs.

Selecting a PR resolves its description, issue links, commits, changed files, checks, review submissions, requested reviewers and threads. Missing fields are unknown, not empty. Issues retain reporter, reproduction and related PR context.

### Inbox wireframe

```text
DuckDB   Inbox                                  [Paste Issue or PR URL]
[Pull requests] [Issues] [Saved reviews]          Search and filters

#28731  Preserve aggregate domains for proven singleton groups
        kevintruong | captured open state | 5 files | +603 / -3
        Linked issue #28615 | exact-key scope | [Open review]

#28615  DISTINCT ON loses join-key statistics
        ted-wq-x | captured issue context | [Read reproduction]

Collection status: captured snapshot / refresh in progress / incomplete
```

### Freeze a comparison

A review run stores upstream/fork identity, target-base SHA, merge-base SHA, head SHA, comparison semantics, GitHub resource snapshot and source blob identities. The default PR changed view compares merge base to head. Target-branch drift and a future merge-result experiment are separate contexts.

If the PR head moves, show **New revision available**. Refresh creates a new run. The old run and its notes remain readable. Source extraction, graph paths, component rules, people evidence and AI findings cannot mix run IDs.

## Journey 3: review Changes + Impact

**User outcome:** understand the changed behavior and follow specific reasons that other code may be affected.

### Main workspace wireframe

```text
DuckDB / PR #28731   Preserve aggregate domains...              [AI review]
Snapshot: merge base 629e395 -> head d29b26b     Coverage: partial
[Overview] [Changes + Impact] [Architecture] [People] [AI Review]

Changed files and symbols   | Diff / Impact graph             | Evidence
first_last_any.cpp          | [Diff] [Graph] [Path list]        | Selected:
  RewriteOrderedFirst      |                                  | ordered rewrite
relation_statistics...     | Source -> changed symbol         | Why this edge?
  HasUniqueGroups          |   -> statistics consumer         | Relation and direction
  HasBaseColumnSource      |   -> estimator                   | Base/head source anchors
  GetValuePreservingColumn |                                  | Provenance / confidence
  ExtractAggregation...    | [Expand callers] [Consumers]     | Missing resolution
Tests and configurations   | [History co-change: off]         | [Open exact source]
```

Three impact layers remain distinct:

- **Direct changes:** file hunks and normalized symbol changes, with rename, ambiguity and unsupported-file diagnostics.
- **Structural impact:** typed calls, imports, inheritance, callback registration and metadata/data-flow relationships supported by the provider. Every result explains a path from a changed seed.
- **Historical associations:** DuckTails co-change and nearby tests, opt-in with time window and sample policy. These are not causal dependencies.

Show the direction of each relation. Incoming callers and outgoing consumers answer different questions. A generic undirected neighborhood is an exploration view, not a proof of blast radius. Each expansion states its scope, query bound and incomplete resolution. The path list provides keyboard-accessible alternatives to the canvas.

The captured example contains five semantic function seeds and a source-inspected path from singleton identity through aggregate domains to join estimates. The six raw candidates include an overlapping template/function representation; the UI preserves ambiguity or normalization evidence instead of inflating the count. Actual plan/performance effects remain unmeasured.

## Journey 4: review architecture alignment

**User outcome:** identify the component boundary affected by a change and compare it with an explicit architecture baseline.

The Architecture view contains a component map, a changed dependency matrix and a list of rule evaluations. Selecting a component filters the same changed symbols and impact paths used elsewhere.

### Architecture wireframe

```text
Architecture      Baseline: approved rules @ base revision     [Inspect rules]

SQL planning -> Aggregate binding/rewrite -> Relational statistics
                         ^                       |
                 Unique-key proofs               v
                                           Join optimization

Change membership             Alignment result       Evidence
Ordered rewrite identity      Not evaluated          No approved policy yet
Unique grouping proof         Not evaluated          Proposed component mapping
Domain transfer               Not evaluated          Source-based membership

[Changed component edges] [Rules] [Unmapped items] [Exceptions]
```

A versioned architecture contract defines component IDs, path/symbol membership rules, allowed dependency directions, public interfaces and documented exceptions. Evaluate newly introduced or removed edges between base and head. A pre-existing dependency is not a newly introduced violation.

Statuses: **Aligned**, **Potential violation**, **Needs mapping**, **Exception**, **Not evaluated**. A detected rule violation requires a specific rule version, resolved edge and source citation. An unresolved edge produces an investigation item.

For DuckDB, the previous experiment supplies a proposed source-based component map, not an approved architecture policy. The first display must say **Alignment not evaluated**. AI-generated mappings are proposals until accepted. Policy edits inside the PR are themselves reviewed against the trusted baseline; they cannot silently approve their own new dependencies.

## Journey 5: identify relevant people

**User outcome:** understand who is involved, who has declared ownership and who may have useful technical context.

### People wireframe

```text
People            Scope: selected change / whole PR
[Current participants] [Declared owners] [Technical candidates]

lnkuiper   Statistics expertise candidate
Why: 108/116 lines in inspected base region; 22/28 sampled file commits
Review focus: singleton cardinality and total/current domain evidence
[Inspect blame] [Inspect commits] [Open GitHub profile]

kryonix    Aggregate rewrite expertise candidate
Why: 38/42 base-region lines; 7/37 sampled file commits
Review focus: ordered rewrite, overloads and serialized rebinding

Ownership: no CODEOWNERS found in captured base/head
Assignment: unknown in captured connector response
```

Roles have independent evidence: PR author, issue reporter, commenter/reviewer, requested reviewer, CODEOWNERS match and historical expertise. Do not collapse them into a single maintainer label or a synthetic score.

DuckTails uses blame and history at the comparison base so the PR author's new lines do not dominate. Show file/region, date window, sample cap, merge policy and identity-mapping evidence. Respect mailmap and separate bots. Shared history does not prove present responsibility or availability.

The first app suggests people and review questions. A later draft request can be reviewed by the human; it never contacts or assigns people automatically.

## Journey 6: AI review grounded in all three lenses

**User outcome:** receive specific, checkable review findings informed by impact, architecture and stakeholder context.

### AI Review wireframe

```text
AI Review             Run: same pinned PR snapshot             [Prepare packet]
Evidence readiness: Changes ready | Impact partial | Architecture unmapped
                    People historical + current assignment unknown
Provider: selected saved provider / not configured

Review scope       [Whole PR] [Selected component] [Selected symbol]
Review focus       Correctness | Compatibility | Tests | Architecture

[Generate review]   unavailable until provider and packet are ready

Finding / question                         Evidence and limits
Exact-key scope does not cover CTAS         PR description + linked issue
Ordered rewrite needs identity propagation Source path + new regression
Architecture alignment needs a baseline    No approved rule set

[Inspect cited source] [Inspect impact path] [Validation steps]
[Keep] [Dismiss] [Needs verification]        [Export draft review]
```

The example items above are review topics from captured evidence, not newly generated defect findings. The app starts with **AI review not run**.

### Review procedure

1. Build a versioned evidence packet from the selected run: intent/reproduction, diff, normalized changes, typed impact paths, architecture rules/evaluations, people context, tests/checks and known gaps.
2. Show what evidence is present and what is missing. A partial review is allowed with a visible scope; architecture approval cannot be claimed without a policy.
3. Resolve a saved AI provider/model and an explicit user-selected review action. Record model/provider version, prompt version, packet hash and execution status.
4. Analyze correctness first, then effect on dependent code, architectural boundaries, test gaps and unresolved assumptions. Read cited source slices, not only summary counts.
5. Emit findings with file/span, specific trigger and consequence, severity, confidence, supporting and counter-evidence, validation steps and test status. Keep questions and coverage gaps separate from defects.
6. Human accepts, dismisses or requests verification. Export a review packet or editable GitHub draft. Publishing remains a separate explicit action.

People evidence helps ask better review questions and identify expertise; it is not evidence that the code is correct. Graph reachability does not prove a bug. PR text, issue comments and source comments are review data, not agent instructions.

For large PRs, partition by coherent changed component and retrieve paths/source on demand. Preserve citations and contradictory evidence through synthesis. If execution, source coverage or the graph is unavailable, say so rather than report a clean review.

## Provider responsibilities and honest availability

| Layer | Responsibility | Verified state / first-build behavior |
|---|---|---|
| Local Git access | Resolve immutable objects and source blobs; preserve local edits | Existing DuckDB object database supports the captured case. No checkout mutation is required. |
| DuckTails | File/revision diffs, base-pinned history, blame and historical co-change | Loaded with Spur Graph in the prior DuckDB 2.0 experiment. Use catalog-discovered schemas. |
| Spur Graph | Source symbols, diagnostic file facts and notebook-cell-aware symbols/changes | Native extraction works. Reported resolution scope is single_file. Ordinary matching and overlapping C++ representations need hardening. |
| Repository linker | Join extracted facts into revision-correct cross-file edges and explicit unresolved records | A required capability gap. The captured source-inspected graph is labeled as curated evidence until automated links are verified. |
| DuckDB core | Versioned evidence tables, joins, aggregation and bounded recursive graph queries | Baseline analytical store and traversal fallback. |
| DuckPGQ adapter | Property-graph pattern/path queries and optional graph algorithms over verified tables | Loaded and queried alongside DuckTails and Spur Graph in the subsequent DuckDB 2.0 experimental notebook. This design notebook renders captured evidence and does not start a live graph provider. |
| Architecture evaluator | Versioned component membership and allowed-dependency rules | Proposed app/repository layer. No existing architecture policy is invented. |
| GitHub collector | Current issues, PRs, reviews, checks, participants and requests | Separate from local Git history. Preserve resource timestamps and pagination/permissions. |
| AI reviewer | Review packet, source retrieval, cited findings and validation questions | Configured provider required. No AI review has run for this design. |

DuckPGQ supports SQL/PGQ and path/graph queries. Its DuckDB 2.0 grammar must be activated per connection after loading, as a separate statement: `SET active_grammar_extensions = ['duckpgq'];`. A production probe must still check actual availability and a small query on its target runtime. The completed extension POC records the successful experimental probe. Graph algorithms can prioritize exploration; centrality is not a measure of correctness, blast severity or ownership. See [official extension reference](https://duckdb.org/community_extensions/extensions/duckpgq) and [grammar settings](https://duckpgq.org/documentation/settings/).

The optional graph provider operates on the same versioned node/edge tables as the SQL fallback. It does not extract missing C++ dependencies. Backend and provider failures are visible in the relevant view, not hidden behind an empty graph.

## Shared evidence contract and notebook data flow

### Identity

A review_run_id binds the repository identity, comparison kind, target base, merge base, head, GitHub context snapshot, source manifests, extractor/linker versions and architecture-policy version. Current metadata can refresh even when the source head has not changed; the resulting context snapshot and AI packet still receive a new version.

Each fact has evidence_id, origin, observed_at, source revision/hash, status and provenance. Edges also carry direction, relation, resolution method and supporting source spans. Notebook symbol references preserve cell identity, source hash and original source coordinates.

A field can be ready, partial, unavailable or failed. Valid empty results remain distinct from missing collection. Historical contributor evidence cannot fill a missing current-reviewer field.

### Data flow

```mermaid
flowchart TD
  C["User intents: repository, PR, scope, refresh"] --> R["Resolve and freeze review request"]
  R --> G["Collect GitHub resource snapshot"]
  R --> B["Read pinned Git blobs"]
  R --> H["DuckTails base history and blame"]
  B --> S["Spur Graph extraction and change normalization"]
  S --> L["Repository linking and unresolved facts"]
  L --> P["Bounded impact paths: SQL or DuckPGQ"]
  S --> A["Architecture mapping and rule evaluation"]
  P --> A
  H --> O["Identity and stakeholder evidence"]
  G --> O
  S --> V["Compose one versioned review view model"]
  G --> V
  P --> V
  A --> V
  O --> V
  V --> U["Native notebook frontend"]
  V --> K["Versioned AI evidence packet"]
  K --> F["Configured AI review job"]
  F --> W["Cited findings and human decisions"]
```

The interactive frontend submits backend-allowlisted intents. Browser controls do not read local files, obtain credentials or execute shell commands. User interaction is a command path; the evidence pipeline remains an acyclic DAG.

Use one producer cell per logical dataset/Arrow port. A composer consumes those independent ports and publishes one review view model. Native Deno reads the model through ports, rather than Python globals or four unrelated queries. Avoid multiple producer-to-consumer ports that form the notebook engine's parallel-edge problem.

The frontend declares its input bindings and emitted intents. App mode shows a native frontend surface; code and source diagnostics remain inspectable in Notebook mode. No multi-client or headless runtime is assumed.

### State integrity

A response from an older run is saved to that run and cannot overwrite the selected run. Every view model declares partial sources explicitly. A single shared selection contains repo, run, PR, comparison, file/symbol or component and active evidence item. Moving between tabs does not rerun collection or reset selection.

A failed collector retries independently. Canceling a job does not discard a previously readable snapshot. AI results bind to packet_hash and become stale when source, architecture policy or relevant context changes.

The human and agent use the same review_packet records and citations. Export includes the comparison, source coverage, query parameters and known gaps, not only prose findings.

## Proposed delivery sequence and acceptance cases

This is a design for approval, not a claim that the app or new providers are implemented.

### Thin vertical slices after approval

1. **Rendered design:** use the approved Open Design palette and layout. Produce a self-contained text/html artifact with an initial visible state, grounded in the captured DuckDB PR evidence. Verify the design with scripts off and inspect the rendered output. Clearly label sample navigation and actions that do not yet invoke a backend.
2. **Native review shell:** create the actual notebook frontend and versioned source ports. Load the captured PR as an explicit snapshot, with shared selection, diff/impact path list, component map, people detail and AI readiness. Run static doctor after structural writes and full doctor/cascade/output checks before reporting app readiness.
3. **Live repository and GitHub loop:** attach the local repo, validate providers, collect a paginated PR context and pinned source objects, then compute real DuckTails/Spur Graph evidence. Reuse cached blobs and surface partial linking. Verify that an upstream/fork mismatch cannot open the wrong PR context.
4. **Architecture and people enrichment:** edit/confirm a versioned component map, evaluate resolved dependency changes against rules, join declared ownership and current GitHub state with historical expertise.
5. **AI review:** configure a provider, prepare the visible evidence packet, run a user-triggered review and inspect every citation. Add exportable draft review decisions. Publication remains explicit.

### Acceptance cases

| Scenario | Expected behavior |
|---|---|
| Wrong notebook runtime | Setup identifies the mismatch before loading the extensions; no misleading empty data |
| Dirty or moved local checkout | Pinned PR source remains stable; working-tree comparison is separate |
| Shallow clone missing a required commit | Explain the missing object and offer a targeted fetch; retain cached context |
| Fork PR targets upstream | Resolve upstream, fork and merge-base identities independently |
| GitHub pagination/rate limit/permission failure | Resource completeness and refresh state are visible; cached rows remain usable |
| New PR head arrives during review | Create a new run; never mix old graph, new diff and old AI findings |
| Duplicate symbol names / template wrappers | No false no-op change counts; preserve ambiguity and normalization evidence |
| Binary, generated or unsupported source | Show explicit coverage/exclusion status rather than silently omit it |
| Callback or cross-file link is unresolved | Show an unresolved frontier and source; no complete blast-radius claim |
| DuckPGQ is unavailable | Keep SQL-backed evidence usable and explain the disabled provider |
| No architecture policy / unmapped symbol | Return Not evaluated / Needs mapping; AI asks a question instead of alleging a violation |
| A dependency already exists at base | Do not report it as a newly introduced architectural violation |
| Historical contributor has many commits | Show expertise evidence without assigning formal ownership or availability |
| Requested-reviewer field is missing | Show unknown, not no reviewers requested |
| AI provider missing or generation fails | Keep the evidence packet accessible; no fabricated findings or clean-review status |
| AI cites stale source or invents an evidence ID | Reject or flag the finding before presenting it as grounded |
| User selects a function then switches views | Preserve the exact run and selection in architecture, people and AI scope |
| Keyboard navigation / narrow viewport | Provide a path list, readable detail panel and stacked layout; no graph-only interaction |

Initial success is a reviewer completing this path on PR #28731: identify the five semantic changes, explain the guarded statistics-to-estimator path, see architecture alignment is not yet evaluated, inspect historical technical candidates, and generate or export a correctly scoped review packet. No performance or complete-coverage claim is implied.

### Design review checkpoints

Confirm the diff-led workspace, Tech Utility direction, read-only first scope and provider boundaries. Then validate the rendered layout against hierarchy, execution, specificity and restraint. Only after that start live integrations and extend the acceptance coverage.

Research references: [completed PR simulation](/Users/kevintruong/project/SpurCode/docs/research/2026-11-17-duckdb-pr-28731-review-simulation.md), [rebuilt-extension and app audit](/Users/kevintruong/project/SpurCode/docs/research/2026-11-17-spur-graph-explore-experience-audit.md), [DuckTails reference](https://duckdb.org/community_extensions/extensions/duck_tails), [DuckPGQ reference](https://duckdb.org/community_extensions/extensions/duckpgq).


# Updated interactive design, 18 November

The completed repository experiment changes the review model: changed files are the selection seeds, and all tracked files supply context. The design below demonstrates unchanged callers and wrapper bodies, a shared metadata contract, inherited test exclusions, CMake membership, historical expertise and an AI evidence packet.

This is a **reviewable design prototype**, not the native app implementation. It uses the prior brief's Tech Utility direction as a design assumption. Navigation and evidence selection are interactive; repository setup is a preview, and no GitHub collection, AI execution or publication is connected. The prototype uses a curated selection from the captured PR evidence, rather than running new graph queries.

The artifact uses shared stateless components, a server-rendered baseline and a bundled client island. Enable active content to try the interactions. The default design remains readable with scripts disabled. A complete UX specification lives at `/Users/kevintruong/project/SpurCode/docs/superpowers/specs/2026-11-18-spur-review-app-ux.md`.

In [5]:
// Design artifact only: captured evidence, no live app providers or AI calls.
import { h } from 'https://esm.sh/preact@10.24.3';
import { render as ssr } from 'https://esm.sh/preact-render-to-string@6.5.11';
import * as esbuild from 'npm:esbuild@0.24.0';
const viewSource="import { h } from 'https://esm.sh/preact@10.24.3';\nexport const initial = () => ({screen:'review',lens:'Changes + context',file:'stats',symbol:'ExtractAggregationStatsInternal',question:'incoming',mode:'map',evidence:'native:wrapper',candidates:true,packet:false,notice:''});\nconst el=(tag,props,...children)=>h(tag,props,...children.flat());\nconst btn=(text,fn,props={})=>el('button',{type:'button',onClick:fn,...props},text);\nconst badge=(text,kind='')=>el('span',{class:'badge '+kind},text);\nconst mono=text=>el('span',{class:'mono'},text);\nconst label=q=>({incoming:'Incoming clients',dependencies:'Dependencies',contract:'Shared contracts',config:'Tests + config'}[q]);\nconst quality=q=>({native_bound_syntax:'Syntax-bound',source_reference_candidate:'Reference candidate',callback_dispatch_candidate:'Possible callback',shared_contract_reference:'Shared contract',build_declaration:'Build declaration',config_value:'Declared config'}[q]||q);\n\nfunction FileList({s,d,act}){\n return el('aside',{class:'files', 'aria-label':'Changed files and symbols'},\n  el('div',{class:'pane-title'},el('strong',{},'Changed files'),badge('5')),\n  el('p',{class:'small muted'},'Select a change to explore repository context.'),\n  el('nav',{},d.files.map(f=>el('div',{class:'file-group'},\n   btn(el('span',{},el('span',{class:'file-name'},f.name),el('span',{class:'file-status'},f.status)),()=>act('file',f.id),{class:'file '+(s.file===f.id?'selected':''),'aria-current':s.file===f.id?'true':undefined,title:f.path}),\n   s.file===f.id&&f.symbols.map(name=>btn(el('span',{},name),()=>act('symbol',name),{class:'symbol '+(s.symbol===name?'selected':''),title:name,'aria-pressed':s.symbol===name}))\n  ))),\n  el('div',{class:'file-footer'},el('strong',{},'Context includes unchanged files'),el('p',{class:'small muted'},'Tracked repository catalog. Semantic linking remains partial.'),btn('Inspect coverage',()=>act('coverage'),{class:'text-button'})));\n}\n\nfunction Graph({s,d,act}){\n const c=d.contexts[s.question],filtered=c.edges.filter(e=>s.candidates||e.kind!=='candidate'),ids=new Set(filtered.flatMap(e=>[e.a,e.b]));\n return el('div',{class:'graph-region',role:'region','aria-label':'Context map. Equivalent relationships are listed below.',tabIndex:0},\n el('svg',{viewBox:'0 0 786 296',role:'img','aria-label':c.title},\n el('defs',{},el('marker',{id:'arrow',viewBox:'0 0 8 8',refX:'7',refY:'4',markerWidth:'6',markerHeight:'6',orient:'auto-start-reverse'},el('path',{d:'M0 0 L8 4 L0 8',fill:'#7c8d84'}))),\n filtered.map(e=>{const a=c.nodes.find(n=>n.id===e.a),b=c.nodes.find(n=>n.id===e.b);const reverse=a.x>b.x;const x1=a.x+(reverse?0:210),x2=b.x+(reverse?210:0),y1=a.y+43,y2=b.y+43;return el('g',{},\n  el('path',{d:`M${x1} ${y1} C${(x1+x2)/2} ${y1},${(x1+x2)/2} ${y2},${x2} ${y2}`,class:'edge '+e.kind,'marker-end':'url(#arrow)'}),\n  el('text',{x:(x1+x2)/2,y:(y1+y2)/2-8,'text-anchor':'middle',class:'edge-label'},e.label==='API reference'?'API ref.':e.label));}),\n c.nodes.filter(n=>ids.has(n.id)).map(n=>el('g',{class:'graph-node',onClick:()=>act('evidence',n.evidence)},\n  el('rect',{x:n.x,y:n.y,width:210,height:86,rx:5,class:n.title===s.symbol?'selected-node':n.state==='Modified'||n.state==='Added'?'changed-node':'neutral-node'}),\n  el('foreignObject',{x:n.x+10,y:n.y+10,width:190,height:68},el('div',{xmlns:'http://www.w3.org/1999/xhtml'},el('div',{class:'node-title'},n.title),el('div',{class:'node-sub'},n.sub),el('div',{class:'node-state'},n.state)))))));\n}\n\nfunction Paths({s,d,act}){\n const c=d.contexts[s.question],rows=c.rows.filter(r=>s.candidates||r.kind!=='candidate');\n return el('div',{class:'paths'},el('div',{class:'section-line'},el('strong',{},'Review paths'),el('span',{class:'small muted'},'Select a path to inspect its witness')),\n rows.length? rows.map(r=>btn(el('span',{},el('span',{class:'path-title'},r.title),el('span',{class:'path-body'},r.body),badge(r.kind==='candidate'?'Candidate':r.kind==='syntax'?'Syntax-bound':r.kind==='contract'?'Shared contract':'Declared config',r.kind)),()=>act('evidence',r.evidence),{class:'path '+(s.evidence===r.evidence?'active':''),'aria-pressed':s.evidence===r.evidence})):el('p',{},'No paths in this filtered selection. Candidate relationships are hidden.'),\n !s.candidates&&el('p',{class:'small muted'},'Candidate relationships hidden. This does not imply no impact.'));\n}\n\nfunction Changes({s,d,act}){\n const f=d.files.find(f=>f.id===s.file),c=d.contexts[s.question];\n return el('section',{class:'main-view'},\n  el('div',{class:'selection-heading'},el('div',{},el('div',{class:'eyebrow'},'SELECTED CHANGE'),el('h2',{},s.symbol||f.name),el('div',{class:'small mono path-wrap'},f.path)),badge(f.status,'change')),\n  el('div',{class:'view-switch',role:'group','aria-label':'Context presentation'},['map','paths','diff'].map(m=>btn({map:'Context map',paths:'Path list',diff:'Source diff'}[m],()=>act('mode',m),{class:s.mode===m?'chosen':'','aria-pressed':s.mode===m}))),\n  s.mode==='diff'?el('div',{class:'diff-view'},el('div',{class:'section-line'},el('strong',{},'Frozen Git diff'),mono('629e395 \u2192 d29b26b')),el('pre',{},f.diff.split('\\n').map(line=>el('span',{class:line.startsWith('+')?'addition':line.startsWith('-')?'deletion':line.startsWith('@@')?'hunk':''},line+'\\n')))):\n  el('div',{},el('div',{class:'question-tabs',role:'group','aria-label':'Context question'},['incoming','dependencies','contract','config'].map(q=>btn(label(q),()=>act('question',q),{'aria-pressed':s.question===q,class:s.question===q?'chosen':''}))),\n   el('div',{class:'context-heading'},el('h3',{},c.title),el('p',{class:'muted'},c.subtitle)),\n   el('div',{class:'filter-line'},el('label',{},el('input',{type:'checkbox',checked:s.candidates,onChange:e=>act('candidates',e.currentTarget.checked)}),' Include candidates'),el('span',{class:'small muted'},'Captured PR neighborhood \u00b7 frozen head')),\n   s.mode==='map'&&el(Graph,{s,d,act}),\n   el('div',{class:'scope-note'},el('strong',{},'Evidence limit. '),c.caveat),el(Paths,{s,d,act})));\n}\n\nfunction Architecture({s,d,act}){\n return el('section',{class:'main-view'},el('div',{class:'eyebrow'},'SAME PR SNAPSHOT'),el('h2',{},'Where the change sits'),el('p',{class:'muted'},'Build-declared components and the contracts between them.'),\n el('div',{class:'scope-note'},el('strong',{},'Alignment not evaluated. '),'No approved architecture policy is attached to this run.'),\n el('div',{class:'architecture-flow'},el('div',{},'Aggregate functions',mono('duckdb_aggr_distr')),el('span',{},'\u2192'),el('div',{},'Shared identity contract',mono('single_value_identity')),el('span',{},'\u2190'),el('div',{},'Statistics / execution',mono('Readers in unchanged code'))),\n el('p',{class:'small muted'},'Arrows show contract participation, not newly introduced architectural dependencies.'),\n el('table',{class:'component-table'},el('thead',{},el('tr',{},el('th',{},'Component'),el('th',{},'Captured members'),el('th',{},'Evidence'))),el('tbody',{},d.components.map(c=>el('tr',{},el('td',{},el('strong',{},c.caption),el('div',{class:'small mono'},c.target)),el('td',{},c.members.map(p=>el('div',{class:'small path-wrap'},p))),el('td',{},btn('Inspect CMake',()=>act('evidence',c.evidence),{class:'text-button'})))))),\n el('div',{class:'next-step'},el('h3',{},'Review the boundary before judging alignment'),el('p',{},'Confirm component membership, then add a versioned policy for allowed dependencies and exceptions. Existing base dependencies remain distinct from new edges.')));\n}\n\nfunction People({s,d,act}){\n const candidates=d.people.filter(p=>p.github_login);\n return el('section',{class:'main-view'},el('div',{class:'eyebrow'},'PARTICIPATION AND EXPERTISE'),el('h2',{},'Who can add context?'),el('p',{class:'muted'},'Historical knowledge is separate from current review responsibility.'),\n el('div',{class:'people-status'},el('div',{},el('strong',{},'Current reviewers'),el('p',{},'Unknown in captured response')),el('div',{},el('strong',{},'Declared ownership'),el('p',{},'No CODEOWNERS found in captured base/head'))),\n el('h3',{},'Historical expertise candidates'),candidates.map(p=>el('article',{class:'person'},el('div',{class:'person-initial'},p.github_login[0].toUpperCase()),el('div',{},el('h3',{},p.github_login),el('p',{},p.author_name),el('p',{class:'small mono path-wrap'},p.file_path),el('p',{},`${p.base_region_lines} inspected base-region lines; ${p.history_commits} sampled file commits.`),el('p',{class:'small muted'},'Base-pinned history and blame. This is not an assignment, ownership claim or availability signal.'),el('details',{},el('summary',{},'Inspect historical evidence'),el('p',{class:'small path-wrap'},'Identity witness: ',p.evidence_url),el('p',{class:'small'},'Source: stakeholder-candidates.json. Sampling scope is inspectable in the original history/blame receipts.'))))),\n el('div',{class:'next-step'},el('strong',{},'Useful review questions'),el('p',{},'Statistics: does the uniqueness proof preserve total and current domain evidence?'),el('p',{},'Aggregate rewrite: is identity metadata preserved across rewrite and rebinding?')));\n}\n\nfunction AI({s,d,act}){\n return el('section',{class:'main-view'},el('div',{class:'eyebrow'},'EVIDENCE BEFORE GENERATION'),el('h2',{},'AI review'),badge('Not run'),el('p',{class:'muted'},'Prepare one packet from the same changes, context, components and people evidence.'),\n el('table',{class:'readiness'},el('tbody',{},[['Changes','Frozen Git diff and semantic seeds'],['Context','Partial linking; candidates retained'],['Architecture','Build membership ready; policy missing'],['People','Historical evidence; current requests unknown'],['Execution','DuckDB test suite not run'],['AI provider','Not configured in this design prototype']].map(([a,b])=>el('tr',{},el('th',{},a),el('td',{},b))))),\n btn('Preview evidence packet',()=>act('packet',!s.packet),{class:'primary','aria-expanded':s.packet}),\n s.packet&&el('section',{class:'packet',id:'packet'},el('h3',{},'Packet preview'),el('p',{},'Design preview over captured evidence. No AI job has run.'),el('p',{class:'small mono'},'Source: 629e395 \u2192 d29b26b'),el('h4',{},'Review questions, not generated defect findings'),\n d.contexts.contract.rows.slice(0,2).map(r=>btn(el('span',{},r.title,el('span',{class:'path-body'},'Inspect supporting source before drawing a conclusion.')),()=>act('evidence',r.evidence),{class:'path'})),\n el('h4',{},'Known limits travel with the packet'),el('p',{},'C++ bindings are incomplete. Contract proximity is not runtime propagation. Architecture has no approved policy. People evidence is historical.'),el('p',{class:'small muted'},'Production findings require citations, counter-evidence, validation steps and a human disposition.')));\n}\n\nfunction Inspector({s,d,act}){\n const e=d.evidence[s.evidence];\n return el('aside',{class:'inspector','aria-label':'Selected evidence'},el('div',{class:'pane-title'},el('strong',{},'Evidence'),badge('Frozen head')),\n el('h3',{},quality(e.evidence_class)),el('p',{class:'relation-name'},e.relation.replaceAll('_',' ')),\n el('div',{class:'evidence-location'},el('div',{class:'small mono path-wrap'},e.witness_path),el('strong',{},'Line '+e.witness_line)),\n el('p',{class:'small muted'},e.location||'Exact source witness'),\n el('pre',{class:'source'},e.lines.map(l=>el('span',{class:l.n===e.witness_line?'current-line':''},el('span',{class:'line-number'},l.n),el('span',{},l.text),'\\n'))),\n el('h4',{},'What this establishes'),el('p',{},e.caveat|| (e.evidence_class==='build_declaration'?'Declared build membership. Architecture policy alignment has not been evaluated.':e.evidence_class==='config_value'?'Declared configuration relationship. This is not an executed test result.':'A retained source witness for this relationship.')),\n e.registration_witnesses&&el('details',{},el('summary',{},'4 callback registrations'),e.registration_witnesses.map(w=>el('div',{class:'registration'},mono(w.path+':'+w.line),el('pre',{},w.text)))),\n el('details',{class:'provenance'},el('summary',{},'Provenance and query scope'),el('dl',{},el('dt',{},'Revision'),el('dd',{class:'mono path-wrap'},e.revision),el('dt',{},'Evidence ID'),el('dd',{class:'mono'},e.id),el('dt',{},'Receipt'),el('dd',{class:'mono path-wrap'},e.artifact+' ['+e.artifact_index+']'),e.witness_blob_oid&&[el('dt',{},'Source blob'),el('dd',{class:'mono path-wrap'},e.witness_blob_oid)]),el('p',{class:'small muted'},'Graph identities and source anchors remain scoped to the originating experiment.')),\n btn('Copy reference',()=>act('copy',e.id),{class:'secondary'}),el('p',{class:'small muted'},'Source and path selection share this review run.'));\n}\n\nfunction Setup({s,d,act}){\n return el('main',{class:'setup-page'},el('div',{class:'eyebrow'},'REPOSITORY SETUP \u00b7 DESIGN PREVIEW'),el('h2',{},'Bring your repository into the review'),el('p',{class:'muted'},'Connect an existing checkout and inspect which evidence is available.'),\n el('label',{class:'field'},'Local repository',el('input',{readOnly:true,value:'/Users/kevintruong/project/duckdb'})),el('label',{class:'field'},'Upstream',el('input',{readOnly:true,value:'duckdb/duckdb'})),el('label',{class:'field'},'Saved GitHub connection',el('input',{readOnly:true,value:'@github-pat'})),\n el('div',{class:'setup-status'},[['Source objects','Captured comparison available'],['Working tree','Review reads pinned commits'],['Runtime','DuckDB 2.0 experiment captured'],['Providers','DuckTails / Spur Graph / DuckPGQ probed'],['Context catalog','18,342 tracked head files'],['Linking','Partial C++ resolution']].map(([a,b])=>el('div',{},el('strong',{},a),el('span',{},b)))),\n el('p',{class:'small muted'},'Fields are a setup preview. This artifact does not save settings, fetch objects or validate a runtime.'),btn('Open captured PR',()=>act('screen','review'),{class:'primary'}));\n}\nfunction Inbox({s,d,act}){\n return el('main',{class:'setup-page'},el('div',{class:'eyebrow'},'DUCKDB / CAPTURED INBOX'),el('h2',{},'Pull requests'),el('p',{class:'muted'},'A saved example is available. Live GitHub collection is not wired in this artifact.'),btn(el('span',{},el('strong',{},'#28731 Preserve aggregate domains for proven singleton groups'),el('span',{class:'path-body'},'5 changed files \u00b7 pinned source snapshot \u00b7 partial context')),()=>act('screen','review'),{class:'inbox-row'}));\n}\n\nexport function View({s,d,act=()=>{}}){\n return el('div',{class:'app'},el('header',{class:'topbar'},el('div',{class:'brand'},el('span',{class:'brand-mark'},'S'),el('strong',{},'Spur Graph'),el('span',{class:'muted'},'Review')),el('nav',{'aria-label':'App navigation'},['review','inbox','setup'].map(name=>btn({review:'Review',inbox:'Inbox',setup:'Repository'}[name],()=>act('screen',name),{'aria-current':s.screen===name?'page':undefined,class:s.screen===name?'chosen':''}))),badge('Interactive design')),\n el('div',{class:'prototype-note'},'Captured evidence from PR #28731. Navigation is interactive; collection, AI execution and publication are not connected.'),\n s.screen==='setup'?el(Setup,{s,d,act}):s.screen==='inbox'?el(Inbox,{s,d,act}):el('div',{},\n  el('section',{class:'pr-header'},el('div',{},el('div',{class:'eyebrow'},'DUCKDB / PULL REQUEST #28731'),el('h1',{},'Preserve aggregate domains for proven singleton groups'),el('div',{class:'snapshot'},mono('629e395 \u2192 d29b26b'),el('span',{},'5 changed files'),el('span',{},'Partial linking'))),btn('Review packet',()=>act('lens','AI review'),{class:'primary'})),\n  el('nav',{class:'lens-tabs','aria-label':'Review lens'},['Changes + context','Architecture','People','AI review'].map(l=>btn(l,()=>act('lens',l),{'aria-current':s.lens===l?'page':undefined,class:s.lens===l?'chosen':''}))),\n  el('div',{class:'workspace'},el(FileList,{s,d,act}),s.lens==='Architecture'?el(Architecture,{s,d,act}):s.lens==='People'?el(People,{s,d,act}):s.lens==='AI review'?el(AI,{s,d,act}):el(Changes,{s,d,act}),el(Inspector,{s,d,act}))),\n el('footer',{class:'app-footer'},el('span',{},'Selection persists across review lenses.'),el('span',{},'Solid: syntax / declarations \u00b7 Dashed: candidates \u00b7 Dotted: shared contracts')),\n el('div',{class:'live',role:'status','aria-live':'polite'},s.notice||''),el('noscript',{},el('div',{class:'scripts-note'},'Static design preview. Enable active content in the notebook to try navigation and evidence selection.')));\n}\n";
const css=":root{font-family:-apple-system,BlinkMacSystemFont,\"Segoe UI\",sans-serif;color:#1b2723;background:#f6f7f6;font-size:14px;line-height:1.5;--green:#176b48;--line:#d8dfda;--muted:#53615b}*{box-sizing:border-box}body{margin:0}button,input{font:inherit}button{cursor:pointer;color:inherit;background:white;border:1px solid var(--line);border-radius:5px;padding:7px 12px;text-align:left;line-height:1.4}button:hover{background:#eff2ef}button:focus-visible,input:focus-visible,summary:focus-visible,[tabindex]:focus-visible{outline:3px solid #176b48;outline-offset:3px}.app{max-width:1920px;margin:auto;background:white;border:1px solid var(--line)}.topbar{display:flex;align-items:center;justify-content:space-between;gap:16px;padding:11px 22px;border-bottom:1px solid var(--line)}.brand,.topbar nav,.snapshot,.view-switch,.question-tabs,.lens-tabs,.filter-line,.section-line,.pane-title{display:flex;align-items:center;gap:12px}.brand-mark{display:grid;place-items:center;width:27px;height:27px;border:1px solid #87998f;border-radius:5px;font-weight:800;font-size:16px}.topbar nav button,.view-switch button,.lens-tabs button,.question-tabs button{border-color:transparent;background:transparent}.topbar .chosen,.view-switch .chosen,.question-tabs .chosen{background:#eff2ef;border-color:var(--line)}.badge{display:inline-block;font-size:11px;line-height:1.4;padding:3px 7px;border:1px solid var(--line);border-radius:4px;white-space:nowrap;color:#53615b;background:#f6f7f6}.badge.change{background:#e6f2e9;color:var(--green);border-color:#b5d1bf}.badge.candidate{color:#805416;border-style:dashed}.badge.contract{border-style:dotted}.prototype-note{font-size:12px;color:#53615b;background:#f6f7f6;padding:7px 22px;border-bottom:1px solid var(--line)}.pr-header{padding:22px 24px 18px;display:flex;align-items:center;justify-content:space-between;gap:24px}h1,h2,h3,h4,p{margin:0}h1{font-size:22px;letter-spacing:-.55px;font-weight:650;line-height:1.3;margin:5px 0 10px}h2{font-size:19px;font-weight:650;line-height:1.35;overflow-wrap:anywhere}h3{font-size:15px;font-weight:650}h4{font-size:13px;margin:18px 0 6px}p{margin:7px 0 12px}.eyebrow{font-size:10px;letter-spacing:1.2px;font-weight:700;color:var(--muted);margin-bottom:7px}.muted,.snapshot{color:var(--muted)}.small{font-size:12px}.mono,pre,.node-title{font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace}.path-wrap{overflow-wrap:anywhere;word-break:break-word}.snapshot{font-size:12px;gap:20px;flex-wrap:wrap}.primary{background:var(--green);border-color:var(--green);color:white;font-weight:600;white-space:nowrap}.primary:hover{background:#105238}.secondary{margin-top:16px}.lens-tabs{gap:4px;padding:0 24px;border-bottom:1px solid var(--line)}.lens-tabs button{border-radius:0;padding:11px 14px;margin-bottom:-1px;border-bottom:2px solid transparent}.lens-tabs .chosen{font-weight:650;border-bottom-color:var(--green);color:var(--green)}.workspace{display:grid;grid-template-columns:234px minmax(0,1fr) 310px;min-height:670px}.files{border-right:1px solid var(--line);padding:17px 12px;background:#fafbfa}.pane-title{justify-content:space-between;margin-bottom:12px}.files .small{padding:0 5px}.file-group{margin-top:10px}.file{display:block;width:100%;border-color:transparent;background:transparent;padding:9px 8px}.file-name{font-size:12px;font-weight:600;display:block;overflow-wrap:anywhere}.file-status{font-size:10px;color:var(--muted);display:block;margin-top:4px}.file.selected{background:#e6f2e9;border-color:#b5d1bf}.symbol{display:block;width:calc(100% - 12px);margin:2px 0 2px 12px;padding:6px 7px;font-family:ui-monospace,monospace;font-size:11px;background:transparent;border-color:transparent;overflow-wrap:anywhere}.symbol.selected{background:#e8ece8;border-color:var(--line)}.file-footer{margin-top:28px;border-top:1px solid var(--line);padding-top:15px;font-size:12px}.text-button{padding:4px 0;border:none;background:transparent;text-decoration:underline;text-underline-offset:3px;color:var(--green);font-size:12px}.main-view{padding:20px;min-width:0}.selection-heading{display:flex;justify-content:space-between;gap:12px;align-items:start;min-height:74px}.selection-heading .small{margin-top:6px;color:var(--muted)}.view-switch{gap:3px;padding:10px 0 13px;margin-bottom:13px;border-bottom:1px solid var(--line);font-size:12px}.question-tabs{gap:3px;flex-wrap:wrap;font-size:11px}.question-tabs button{padding:6px 8px}.context-heading{margin:18px 0 6px}.context-heading p{font-size:12px}.filter-line{justify-content:space-between;margin:10px 0;font-size:12px;flex-wrap:wrap}.filter-line label{display:flex;gap:6px;align-items:center}input[type=checkbox]{accent-color:var(--green)}.graph-region{background:#fafbfa;border:1px solid var(--line);border-radius:5px;overflow:auto;min-height:210px}.graph-region svg{display:block;width:100%;min-width:560px}.edge{stroke:#7c8d84;stroke-width:1.6;fill:none}.edge.candidate{stroke-dasharray:6 4}.edge.contract{stroke-dasharray:2 4}.edge-label{font-size:9px;fill:#53615b;paint-order:stroke;stroke:#fafbfa;stroke-width:5px}.changed-node{fill:#e6f2e9;stroke:#478b67;stroke-width:1.5}.neutral-node{fill:white;stroke:#acb8af;stroke-width:1}.node-title{font-size:11px;line-height:1.35;overflow-wrap:anywhere;font-weight:600}.node-sub{font-size:9px;color:#53615b;margin-top:4px;overflow-wrap:anywhere}.node-state{font-size:9px;margin-top:4px;font-weight:600}.graph-node{cursor:pointer}.graph-node:hover rect{stroke:#176b48;stroke-width:2.5}.scope-note{font-size:12px;color:#53615b;background:#f2f4f1;border:1px solid var(--line);border-radius:4px;padding:10px 12px;margin:12px 0 18px}.section-line{justify-content:space-between;align-items:baseline;margin-bottom:8px;flex-wrap:wrap;gap:8px}.paths{margin-top:10px}.path{display:block;width:100%;padding:11px 12px;margin-bottom:7px;background:white;border:1px solid var(--line)}.path.active{border-color:#7a9987;background:#f5f8f5}.path-title{font-size:12px;font-weight:600;display:block}.path-body{font-size:12px;display:block;color:var(--muted);margin:4px 0 7px}.inspector{border-left:1px solid var(--line);padding:17px 16px;min-width:0;background:#fcfdfb}.inspector h3{margin-top:20px}.relation-name{font-size:12px;color:#53615b;overflow-wrap:anywhere}.evidence-location{padding-top:14px;margin-top:15px;border-top:1px solid var(--line)}.evidence-location strong{font-size:11px}.source{background:#f1f3f0;font-size:11px;line-height:1.8;padding:8px 0;white-space:pre-wrap;overflow-wrap:anywhere;word-break:break-word;max-height:270px;overflow:auto;border:1px solid var(--line);border-radius:4px}.source>span{display:block}.source .line-number{display:inline-block;width:36px;color:#53615b;text-align:right;padding-right:8px;user-select:none}.source .current-line{background:#dfeae1}.inspector p{font-size:12px}.provenance{border-top:1px solid var(--line);padding-top:14px;margin-top:20px;font-size:12px}summary{cursor:pointer;font-weight:600}dt{font-size:11px;color:var(--muted);margin-top:9px}dd{font-size:11px;margin:2px 0 0}.registration{margin-top:10px;font-size:11px;overflow-wrap:anywhere}.registration pre{white-space:pre-wrap}.app-footer{padding:10px 22px;background:#f6f7f6;border-top:1px solid var(--line);font-size:11px;color:var(--muted);display:flex;justify-content:space-between;gap:12px;flex-wrap:wrap}.scripts-note{padding:10px 22px;font-size:12px;background:#fff7e8}.live:empty{display:none}.live{padding:10px 22px;font-size:12px;background:#e6f2e9}.architecture-flow{display:flex;gap:14px;align-items:center;border:1px solid var(--line);padding:18px 12px;margin-top:20px;font-size:12px}.architecture-flow>div{flex:1}.architecture-flow .mono{display:block;font-size:10px;margin-top:6px;overflow-wrap:anywhere}table{border-collapse:collapse;width:100%;margin:18px 0;font-size:12px}th,td{text-align:left;vertical-align:top;padding:12px 8px;border-bottom:1px solid var(--line)}th{font-weight:600;color:#53615b}.component-table td:first-child{width:34%}.component-table .small{font-size:10px}.next-step{border-top:1px solid var(--line);margin-top:25px;padding-top:16px;font-size:12px}.people-status{display:grid;grid-template-columns:1fr 1fr;gap:20px;background:#f6f7f6;padding:16px;margin:20px 0;font-size:12px}.person{display:flex;gap:14px;padding:20px 0;border-bottom:1px solid var(--line)}.person-initial{display:grid;place-items:center;width:35px;height:35px;background:#eff2ef;border:1px solid var(--line);border-radius:50%;flex-shrink:0;font-size:13px;font-weight:600}.person p{font-size:12px}.person details{font-size:12px}.packet{padding:16px;background:#fafbfa;border:1px solid var(--line);margin-top:20px;font-size:12px}.readiness th{width:25%}.setup-page{max-width:800px;padding:36px 28px 60px;margin:auto}.setup-page h2{font-size:25px}.field{display:block;font-size:12px;font-weight:600;margin:20px 0}.field input{display:block;width:100%;padding:10px;margin-top:7px;border:1px solid var(--line);border-radius:4px;background:#fafbfa;color:#53615b}.setup-status{margin:25px 0;border-top:1px solid var(--line)}.setup-status>div{display:flex;justify-content:space-between;padding:12px 0;border-bottom:1px solid var(--line);font-size:12px;gap:20px}.inbox-row{padding:20px;width:100%;margin-top:25px}.diff-view{font-size:12px}.diff-view pre{font-size:11px;white-space:pre-wrap;word-break:break-word;max-height:630px;overflow:auto;background:#fafbfa;border:1px solid var(--line);padding:10px}.diff-view pre span{display:block}.addition{background:#e6f2e9}.deletion{background:#faece9}.hunk{color:#245d77;background:#eaf0f3}@media(min-width:1600px){.workspace{grid-template-columns:260px minmax(0,1fr) 350px}.main-view{padding:24px}}@media(max-width:1100px){.workspace{grid-template-columns:205px minmax(0,1fr)}.inspector{grid-column:1/-1;border-left:0;border-top:1px solid var(--line);display:grid;grid-template-columns:minmax(0,1fr) minmax(0,1fr);column-gap:24px}.inspector .pane-title{grid-column:1/-1}.inspector .source{grid-column:1/-1}.inspector .provenance{grid-column:1/-1}.topbar{padding:10px 15px}.pr-header{padding:18px}.main-view{padding:16px}}@media(max-width:700px){.workspace{display:block}.files{border-right:0;border-bottom:1px solid var(--line)}.files nav{display:grid;grid-template-columns:1fr 1fr;gap:4px}.file-footer{margin-top:15px}.symbol{display:none}.pr-header{display:block}.pr-header>.primary{margin-top:12px}.topbar{flex-wrap:wrap;gap:9px}.brand{gap:7px}.topbar nav{gap:3px}.topbar button{padding:6px 8px;font-size:12px}.lens-tabs{padding:0 8px;gap:0;flex-wrap:wrap}.lens-tabs button{font-size:12px;padding:10px 9px}.inspector{display:block}.pr-header h1{font-size:21px}.snapshot{gap:10px}.people-status{grid-template-columns:1fr}.component-table td,.component-table th{padding:10px 4px}.architecture-flow{gap:8px}.prototype-note{padding:8px 15px}.selection-heading{min-height:unset;margin-bottom:15px}}@media(prefers-reduced-motion:reduce){*{scroll-behavior:auto!important;transition:none!important}}\n\n.selected-node{fill:#e6f2e9;stroke:#176b48;stroke-width:2.5}.component-table .mono{overflow-wrap:anywhere}\n";
const data={"head": "d29b26bb92ac940a0807be0f43cc385b31623839", "base": "629e395a854d72dcb1f414e1a0451c68ac0e9f1f", "files": [{"id": "stats", "name": "relation_statistics_helper.cpp", "path": "src/optimizer/relation_statistics/relation_statistics_helper.cpp", "status": "Modified", "symbols": ["ExtractAggregationStatsInternal", "HasUniqueGroups", "HasBaseColumnSource", "GetValuePreservingColumn"], "diff": "diff --git a/src/optimizer/relation_statistics/relation_statistics_helper.cpp b/src/optimizer/relation_statistics/relation_statistics_helper.cpp\nindex 99c3862b89c..6eac019d1e3 100644\n--- a/src/optimizer/relation_statistics/relation_statistics_helper.cpp\n+++ b/src/optimizer/relation_statistics/relation_statistics_helper.cpp\n@@ -2,6 +2,8 @@\n #include \"duckdb/optimizer/relation_statistics/relation_statistics_extractor.hpp\"\n \n #include \"duckdb/common/types/value_map.hpp\"\n+#include \"duckdb/optimizer/apply_functional_dependencies.hpp\"\n+#include \"duckdb/planner/expression/bound_cast_expression.hpp\"\n #include \"duckdb/planner/expression/bound_comparison_expression.hpp\"\n #include \"duckdb/planner/expression/bound_constant_expression.hpp\"\n #include \"duckdb/planner/expression/list.hpp\"\n@@ -450,6 +452,109 @@ idx_t RelationStatisticsHelper::EstimateGroupCardinality(LogicalOperator &input,\n \treturn estimate;\n }\n \n+static optional<ColumnBinding> GetValuePreservingColumn(const Expression &expression) {\n+\tif (expression.GetExpressionClass() == ExpressionClass::BOUND_COLUMN_REF) {\n+\t\treturn expression.Cast<BoundColumnRefExpression>().Binding();\n+\t}\n+\tif (!BoundCastExpression::IsCast(expression)) {\n+\t\treturn {};\n+\t}\n+\tauto &cast = expression.Cast<BoundFunctionExpression>();\n+\tif (!BoundCastExpression::IsDefaultCast(cast) || BoundCastExpression::IsTryCast(cast)) {\n+\t\treturn {};\n+\t}\n+\tauto source_type = BoundCastExpression::SourceType(cast);\n+\tauto &target_type = BoundCastExpression::TargetType(cast);\n+\t// Successful strict integral casts do not merge distinct input values.\n+\tif (source_type != target_type && !(source_type.IsIntegral() && target_type.IsIntegral()) &&\n+\t    !BoundCastExpression::CastIsInvertibleForEquality(source_type, target_type)) {\n+\t\treturn {};\n+\t}\n+\treturn GetValuePreservingColumn(BoundCastExpression::Child(cast));\n+}\n+\n+static bool HasBaseColumnSource(LogicalOperator &op, ColumnBinding binding) {\n+\tswitch (op.type) {\n+\tcase LogicalOperatorType::LOGICAL_GET: {\n+\t\tif (!op.children.empty()) {\n+\t\t\treturn false;\n+\t\t}\n+\t\tfor (auto &column : op.GetColumnBindings()) {\n+\t\t\tif (column == binding) {\n+\t\t\t\treturn true;\n+\t\t\t}\n+\t\t}\n+\t\treturn false;\n+\t}\n+\tcase LogicalOperatorType::LOGICAL_PROJECTION: {\n+\t\tif (op.children.size() != 1) {\n+\t\t\treturn false;\n+\t\t}\n+\t\tauto bindings = op.GetColumnBindings();\n+\t\tfor (idx_t i = 0; i < bindings.size(); i++) {\n+\t\t\tif (bindings[i] != binding) {\n+\t\t\t\tcontinue;\n+\t\t\t}\n+\t\t\tauto child_binding = GetValuePreservingColumn(*op.expressions[i]);\n+\t\t\treturn child_binding && HasBaseColumnSource(*op.children[0], *child_binding);\n+\t\t}\n+\t\treturn false;\n+\t}\n+\tcase LogicalOperatorType::LOGICAL_FILTER:\n+\tcase LogicalOperatorType::LOGICAL_ORDER_BY:\n+\tcase LogicalOperatorType::LOGICAL_LIMIT:\n+\tcase LogicalOperatorType::LOGICAL_TOP_N:\n+\t\treturn op.children.size() == 1 && HasBaseColumnSource(*op.children[0], binding);\n+\tdefault:\n+\t\treturn false;\n+\t}\n+}\n+\n+static bool HasUniqueGroups(LogicalOperator &op, const column_binding_set_t &groups) {\n+\tif (groups.empty()) {\n+\t\treturn false;\n+\t}\n+\tswitch (op.type) {\n+\tcase LogicalOperatorType::LOGICAL_PROJECTION: {\n+\t\tif (op.children.size() != 1) {\n+\t\t\treturn false;\n+\t\t}\n+\t\tcolumn_binding_set_t child_groups;\n+\t\tauto bindings = op.GetColumnBindings();\n+\t\tfor (idx_t i = 0; i < bindings.size(); i++) {\n+\t\t\tif (groups.count(bindings[i]) == 0) {\n+\t\t\t\tcontinue;\n+\t\t\t}\n+\t\t\tauto binding = GetValuePreservingColumn(*op.expressions[i]);\n+\t\t\tif (binding) {\n+\t\t\t\tchild_groups.insert(*binding);\n+\t\t\t}\n+\t\t}\n+\t\treturn HasUniqueGroups(*op.children[0], child_groups);\n+\t}\n+\tcase LogicalOperatorType::LOGICAL_FILTER:\n+\tcase LogicalOperatorType::LOGICAL_ORDER_BY:\n+\tcase LogicalOperatorType::LOGICAL_LIMIT:\n+\tcase LogicalOperatorType::LOGICAL_TOP_N:\n+\t\treturn op.children.size() == 1 && HasUniqueGroups(*op.children[0], groups);\n+\tdefault:\n+\t\tbreak;\n+\t}\n+\tfor (auto &key : ApplyFunctionalDependencies::GetUniqueColumnCombinations(op)) {\n+\t\tbool covered = !key.empty();\n+\t\tfor (auto &binding : key) {\n+\t\t\tif (groups.count(binding) == 0) {\n+\t\t\t\tcovered = false;\n+\t\t\t\tbreak;\n+\t\t\t}\n+\t\t}\n+\t\tif (covered) {\n+\t\t\treturn true;\n+\t\t}\n+\t}\n+\treturn false;\n+}\n+\n static optional<RelationStats> ExtractAggregationStatsInternal(LogicalAggregate &aggregate,\n                                                                const RelationStats &child_stats,\n                                                                RelationStatsExtractor &extractor,\n@@ -510,6 +615,20 @@ static optional<RelationStats> ExtractAggregationStatsInternal(LogicalAggregate\n \t\t        ? 1\n \t\t        : RelationStatisticsHelper::EstimateDistinctCardinality(cardinality_counts, child_stats.cardinality);\n \t}\n+\tbool has_unique_groups = false;\n+\tif (preserves_group_evidence && !aggregate.groups.empty() && aggregate.children.size() == 1) {\n+\t\tcolumn_binding_set_t groups;\n+\t\tfor (auto &group : aggregate.groups) {\n+\t\t\tauto binding = GetValuePreservingColumn(*group);\n+\t\t\tif (binding) {\n+\t\t\t\tgroups.insert(*binding);\n+\t\t\t}\n+\t\t}\n+\t\thas_unique_groups = HasUniqueGroups(*aggregate.children[0], groups);\n+\t\tif (has_unique_groups) {\n+\t\t\tresult.cardinality = child_stats.cardinality;\n+\t\t}\n+\t}\n \tfor (idx_t group_idx = 0; group_idx < aggregate.groups.size(); group_idx++) {\n \t\tauto &group = *aggregate.groups[group_idx];\n \t\tDistinctCount distinct_count(result.cardinality, DistinctCountSource::CARDINALITY);\n@@ -536,9 +655,24 @@ static optional<RelationStats> ExtractAggregationStatsInternal(LogicalAggregate\n \t\tresult.columns.emplace_back(bindings[result.columns.size()], distinct_count, current_domain, Identifier(),\n \t\t                            semi_anti_join_domain_evidence);\n \t}\n-\tfor (idx_t aggregate_idx = 0; aggregate_idx < aggregate.expressions.size(); aggregate_idx++) {\n-\t\tresult.columns.emplace_back(bindings[result.columns.size()],\n-\t\t                            DistinctCount(result.cardinality, DistinctCountSource::CARDINALITY), Identifier());\n+\tfor (auto &expression : aggregate.expressions) {\n+\t\tDistinctCount total_domain(result.cardinality, DistinctCountSource::CARDINALITY);\n+\t\tDistinctCount current_domain = total_domain;\n+\t\tif (has_unique_groups && expression->GetExpressionClass() == ExpressionClass::BOUND_AGGREGATE) {\n+\t\t\tauto &bound_aggregate = expression->Cast<BoundAggregateExpression>();\n+\t\t\tauto &agg_children = bound_aggregate.GetChildren();\n+\t\t\tif (bound_aggregate.Function().HasSingleValueIdentity() && !bound_aggregate.GetFilter() &&\n+\t\t\t    !agg_children.empty() && expression->GetReturnType() == agg_children[0]->GetReturnType() &&\n+\t\t\t    agg_children[0]->GetExpressionClass() == ExpressionClass::BOUND_COLUMN_REF) {\n+\t\t\t\tauto binding = agg_children[0]->Cast<BoundColumnRefExpression>().Binding();\n+\t\t\t\tauto child_column = child_columns.Find(binding);\n+\t\t\t\tif (child_column && HasBaseColumnSource(*aggregate.children[0], binding)) {\n+\t\t\t\t\ttotal_domain = child_column->total_domain;\n+\t\t\t\t\tcurrent_domain = child_column->current_domain;\n+\t\t\t\t}\n+\t\t\t}\n+\t\t}\n+\t\tresult.columns.emplace_back(bindings[result.columns.size()], total_domain, current_domain, Identifier());\n \t}\n \tfor (idx_t grouping_idx = 0; grouping_idx < aggregate.grouping_functions.size(); grouping_idx++) {\n \t\tauto grouping_count = MinValue<idx_t>(result.cardinality, MaxValue<idx_t>(aggregate.grouping_sets.size(), 1));\n"}, {"id": "rewrite", "name": "first_last_any.cpp", "path": "src/function/aggregate/distributive/first_last_any.cpp", "status": "Modified", "symbols": ["RewriteOrderedFirst"], "diff": "diff --git a/src/function/aggregate/distributive/first_last_any.cpp b/src/function/aggregate/distributive/first_last_any.cpp\nindex f36a4e84705..0ba568cb72b 100644\n--- a/src/function/aggregate/distributive/first_last_any.cpp\n+++ b/src/function/aggregate/distributive/first_last_any.cpp\n@@ -53,6 +53,8 @@ unique_ptr<Expression> RewriteOrderedFirst(AggregateRewriteInput &input) {\n \tconst auto &function = entry.functions.GetFunctionByArguments(input.context, child_types);\n \tauto result = binder.BindAggregateFunction(function, std::move(children), std::move(aggregate->GetFilterMutable()),\n \t                                           aggregate->GetAggregateType());\n+\t// The encoded sort key is non-NULL, so a singleton still returns its input value.\n+\tresult->FunctionMutable().SetSingleValueIdentity(aggregate->Function().HasSingleValueIdentity());\n \treturn unique_ptr_cast<BoundAggregateExpression, Expression>(std::move(result));\n }\n \n"}, {"id": "shared", "name": "stats_dependent_tests.json", "path": "test/configs/stats_dependent_tests.json", "status": "Modified", "symbols": [], "diff": "diff --git a/test/configs/stats_dependent_tests.json b/test/configs/stats_dependent_tests.json\nindex 0db774e9b95..7457b6bce55 100644\n--- a/test/configs/stats_dependent_tests.json\n+++ b/test/configs/stats_dependent_tests.json\n@@ -37,6 +37,7 @@\n         \"test/sql/storage/list_length_stats_checkpoint.test\",\n         \"test/sql/storage/list_length_stats_nested_optimistic.test\",\n         \"test/sql/optimizer/statistics/list_length_stats_propagation.test\",\n+        \"test/sql/optimizer/statistics/distinct_on_stats_propagation.test\",\n         \"test/sql/storage/list_length_stats_old_version.test\",\n         \"test/sql/types/nested/array/array_statistics.test\",\n         \"test/sql/vacuum/test_analyze.test\",\n"}, {"id": "export", "name": "verify_aggregate_state_export.json", "path": "test/configs/verify_aggregate_state_export.json", "status": "Modified", "symbols": [], "diff": "diff --git a/test/configs/verify_aggregate_state_export.json b/test/configs/verify_aggregate_state_export.json\nindex 7c7c58f3252..e59f36a3806 100644\n--- a/test/configs/verify_aggregate_state_export.json\n+++ b/test/configs/verify_aggregate_state_export.json\n@@ -47,6 +47,7 @@\n         \"test/sql/copy/parquet/parquet_stats.test\",\n         \"test/sql/copy/parquet/read_single_parquet_file.test\",\n         \"test/sql/optimizer/aggregate_stats_propagation.test\",\n+        \"test/sql/optimizer/statistics/distinct_on_stats_propagation.test\",\n         \"test/sql/optimizer/scalar_aggregate_merge_files_plan.test\",\n         \"test/sql/optimizer/scalar_aggregate_merge_plan.test\",\n         \"test/sql/optimizer/scalar_aggregate_merge_tpcds_plan.test\",\n"}, {"id": "test", "name": "distinct_on_stats_propagation.test", "path": "test/sql/optimizer/statistics/distinct_on_stats_propagation.test", "status": "Added", "symbols": [], "diff": "diff --git a/test/sql/optimizer/statistics/distinct_on_stats_propagation.test b/test/sql/optimizer/statistics/distinct_on_stats_propagation.test\nnew file mode 100644\nindex 00000000000..97430e2de99\n--- /dev/null\n+++ b/test/sql/optimizer/statistics/distinct_on_stats_propagation.test\n@@ -0,0 +1,462 @@\n+# name: test/sql/optimizer/statistics/distinct_on_stats_propagation.test\n+# description: Aggregate domains require exact grouping keys and single-value identity\n+# group: [statistics]\n+\n+# Exact plan estimates depend on sampled distinct counts.\n+require vector_size 2048\n+\n+# HLL can estimate one group per input row even when grouping removes a row.\n+statement ok\n+CREATE TABLE false_singletons AS\n+SELECT CASE WHEN i = 32 THEN 0 ELSE i END * 2018::BIGINT AS g,\n+       CASE WHEN i = 32 THEN 9 ELSE i % 2 END::INT AS x FROM range(33) t(i);\n+\n+statement ok\n+CREATE TABLE false_singletons_right AS SELECT * FROM false_singletons;\n+\n+statement ok\n+ANALYZE;\n+\n+statement ok\n+SET explain_output = 'optimized_only';\n+\n+query II\n+EXPLAIN SELECT count(*)\n+FROM (SELECT first(x ORDER BY x) AS k FROM false_singletons GROUP BY g) l\n+JOIN (SELECT first(x ORDER BY x) AS k FROM false_singletons_right GROUP BY g) r ON l.k = r.k;\n+----\n+logical_opt\t<REGEX>:(?s).*Comparison Join[^~]*~33 rows.*\n+\n+query I\n+SELECT count(*)\n+FROM (SELECT first(x ORDER BY x) AS k FROM false_singletons GROUP BY g) l\n+JOIN (SELECT first(x ORDER BY x) AS k FROM false_singletons_right GROUP BY g) r ON l.k = r.k;\n+----\n+512\n+\n+# Positive fixture with a declared key. Unconstrained inputs are checked separately.\n+statement ok\n+CREATE TABLE nodes(id INTEGER PRIMARY KEY, k INTEGER);\n+\n+statement ok\n+INSERT INTO nodes SELECT i::INT, i::INT FROM range(100000) t(i);\n+\n+statement ok\n+CREATE TABLE edges AS SELECT i::INT AS eid, (i % 10000)::INT AS k FROM range(100000) t(i);\n+\n+statement ok\n+ANALYZE;\n+\n+statement ok\n+SET explain_output = 'optimized_only';\n+\n+# DISTINCT ON lowers to Aggregate(GROUP BY id, first(k)): the first() output carries k's\n+# domain (100000 distinct) instead of an aggregate-cardinality guess, so the join\n+# denominator is max(100000, 10000) and the estimate matches the DISTINCT k control\n+query II\n+EXPLAIN SELECT count(*) FROM (SELECT DISTINCT ON (id) k FROM nodes) s JOIN edges e ON s.k = e.k;\n+----\n+logical_opt\t<REGEX>:(?s).*Comparison Join[^~]*~100,000 rows.*\n+\n+# a handwritten first() gets the same domain forwarding\n+query II\n+EXPLAIN SELECT count(*) FROM (SELECT first(k) AS k FROM nodes GROUP BY id) s JOIN edges e ON s.k = e.k;\n+----\n+logical_opt\t<REGEX>:(?s).*Comparison Join[^~]*~100,000 rows.*\n+\n+# DISTINCT ON with an ORDER BY inside the subquery keeps the forwarded domain\n+query II\n+EXPLAIN SELECT count(*) FROM (SELECT DISTINCT ON (id) k FROM nodes ORDER BY id, k) s JOIN edges e ON s.k = e.k;\n+----\n+logical_opt\t<REGEX>:(?s).*Comparison Join[^~]*~100,000 rows.*\n+\n+# the DISTINCT k control was already exact and stays exact\n+query II\n+EXPLAIN SELECT count(*) FROM (SELECT DISTINCT k FROM nodes) s JOIN edges e ON s.k = e.k;\n+----\n+logical_opt\t<REGEX>:(?s).*Comparison Join[^~]*~100,000 rows.*\n+\n+# the statistics change does not change results\n+query I\n+SELECT count(*) FROM (SELECT DISTINCT ON (id) k FROM nodes) s JOIN edges e ON s.k = e.k;\n+----\n+100000\n+\n+query I\n+SELECT count(*) FROM (SELECT DISTINCT ON (id) k FROM nodes ORDER BY id, k) s JOIN edges e ON s.k = e.k;\n+----\n+100000\n+\n+# List-producing overloads create combinations outside their first child's domain.\n+statement ok\n+CREATE TABLE bit_pairs_left AS\n+SELECT g::INT AS g, p::INT AS p, ((g >> p) & 1)::INT AS x\n+FROM range(4) g_values(g), range(2) p_values(p);\n+\n+statement ok\n+CREATE TABLE bit_pairs_right AS SELECT * FROM bit_pairs_left;\n+\n+statement ok\n+ANALYZE;\n+\n+query II\n+EXPLAIN SELECT count(*)\n+FROM (SELECT arg_min(x, p, 2) AS k FROM bit_pairs_left GROUP BY g) l\n+JOIN (SELECT arg_min(x, p, 2) AS k FROM bit_pairs_right GROUP BY g) r ON l.k = r.k;\n+----\n+logical_opt\t<REGEX>:(?s).*Comparison Join[^~]*~4 rows.*\n+\n+query II\n+EXPLAIN SELECT count(*)\n+FROM (SELECT arg_max(x, p, 2) AS k FROM bit_pairs_left GROUP BY g) l\n+JOIN (SELECT arg_max(x, p, 2) AS k FROM bit_pairs_right GROUP BY g) r ON l.k = r.k;\n+----\n+logical_opt\t<REGEX>:(?s).*Comparison Join[^~]*~4 rows.*\n+\n+# The return type also differs when the input is already a list.\n+query II\n+EXPLAIN SELECT count(*)\n+FROM (SELECT arg_min([x], p, 2) AS k FROM bit_pairs_left GROUP BY g) l\n+JOIN (SELECT arg_min([x], p, 2) AS k FROM bit_pairs_right GROUP BY g) r ON l.k = r.k;\n+----\n+logical_opt\t<REGEX>:(?s).*Comparison Join[^~]*~4 rows.*\n+\n+# Reducing groups retain the aggregate cardinality fallback.\n+query II\n+EXPLAIN SELECT count(*)\n+FROM (SELECT first(x ORDER BY p) AS k FROM bit_pairs_left GROUP BY g) l\n+JOIN (SELECT first(x ORDER BY p) AS k FROM bit_pairs_right GROUP BY g) r ON l.k = r.k;\n+----\n+logical_opt\t<REGEX>:(?s).*Comparison Join[^~]*~4 rows.*\n+\n+# List-producing overloads remain excluded even with certified singleton groups.\n+statement ok\n+CREATE TABLE unique_list_values(id INTEGER PRIMARY KEY, x INTEGER, p INTEGER);\n+\n+statement ok\n+INSERT INTO unique_list_values SELECT i::INT, (i % 2)::INT, NULL FROM range(8) t(i);\n+\n+statement ok\n+CREATE TABLE unique_list_values_right(id INTEGER PRIMARY KEY, x INTEGER, p INTEGER);\n+\n+statement ok\n+INSERT INTO unique_list_values_right SELECT * FROM unique_list_values;\n+\n+statement ok\n+ANALYZE;\n+\n+# The scalar control forwards the domain for the same unique groups.\n+query II\n+EXPLAIN SELECT count(*)\n+FROM (SELECT first(x) AS k FROM unique_list_values GROUP BY id) l\n+JOIN (SELECT first(x) AS k FROM unique_list_values_right GROUP BY id) r ON l.k = r.k;\n+----\n+logical_opt\t<REGEX>:(?s).*Comparison Join[^~]*~32 rows.*\n+\n+# An aggregate filter can remove values even when every group has one row.\n+query II\n+EXPLAIN SELECT count(*)\n+FROM (SELECT first(x) FILTER (WHERE id % 2 = 0) AS k FROM unique_list_values GROUP BY id) l\n+JOIN (SELECT first(x) FILTER (WHERE id % 2 = 0) AS k FROM unique_list_values_right GROUP BY id) r ON l.k = r.k;\n+----\n+logical_opt\t<REGEX>:(?s).*Comparison Join[^~]*~8 rows.*\n+\n+query I\n+SELECT count(*)\n+FROM (SELECT first(x) FILTER (WHERE id % 2 = 0) AS k FROM unique_list_values GROUP BY id) l\n+JOIN (SELECT first(x) FILTER (WHERE id % 2 = 0) AS k FROM unique_list_values_right GROUP BY id) r ON l.k = r.k;\n+----\n+16\n+\n+query II\n+EXPLAIN SELECT count(*)\n+FROM (SELECT arg_min(x, id, 2) AS k FROM unique_list_values GROUP BY id) l\n+JOIN (SELECT arg_min(x, id, 2) AS k FROM unique_list_values_right GROUP BY id) r ON l.k = r.k;\n+----\n+logical_opt\t<REGEX>:(?s).*Comparison Join[^~]*~8 rows.*\n+\n+query II\n+EXPLAIN SELECT count(*)\n+FROM (SELECT arg_max(x, id, 2) AS k FROM unique_list_values GROUP BY id) l\n+JOIN (SELECT arg_max(x, id, 2) AS k FROM unique_list_values_right GROUP BY id) r ON l.k = r.k;\n+----\n+logical_opt\t<REGEX>:(?s).*Comparison Join[^~]*~8 rows.*\n+\n+query I\n+SELECT count(*)\n+FROM (SELECT arg_min(x, id, 2) AS k FROM unique_list_values GROUP BY id) l\n+JOIN (SELECT arg_min(x, id, 2) AS k FROM unique_list_values_right GROUP BY id) r ON l.k = r.k;\n+----\n+32\n+\n+query I\n+SELECT count(*)\n+FROM (SELECT arg_max(x, id, 2) AS k FROM unique_list_values GROUP BY id) l\n+JOIN (SELECT arg_max(x, id, 2) AS k FROM unique_list_values_right GROUP BY id) r ON l.k = r.k;\n+----\n+32\n+\n+query I\n+SELECT count(*)\n+FROM (SELECT arg_min(x, p, 2) AS k FROM bit_pairs_left GROUP BY g) l\n+JOIN (SELECT arg_min(x, p, 2) AS k FROM bit_pairs_right GROUP BY g) r ON l.k = r.k;\n+----\n+4\n+\n+query I\n+SELECT count(*)\n+FROM (SELECT arg_max(x, p, 2) AS k FROM bit_pairs_left GROUP BY g) l\n+JOIN (SELECT arg_max(x, p, 2) AS k FROM bit_pairs_right GROUP BY g) r ON l.k = r.k;\n+----\n+4\n+\n+query I\n+SELECT count(*)\n+FROM (SELECT first(x ORDER BY p) AS k FROM bit_pairs_left GROUP BY g) l\n+JOIN (SELECT first(x ORDER BY p) AS k FROM bit_pairs_right GROUP BY g) r ON l.k = r.k;\n+----\n+8\n+\n+# Aliases retain the bound single-value identity property.\n+query II\n+EXPLAIN SELECT count(*) FROM (SELECT arbitrary(k) AS k FROM nodes GROUP BY id) s JOIN edges e ON s.k = e.k;\n+----\n+logical_opt\t<REGEX>:(?s).*Comparison Join[^~]*~100,000 rows.*\n+\n+query II\n+EXPLAIN SELECT count(*) FROM (SELECT last(k) AS k FROM nodes GROUP BY id) s JOIN edges e ON s.k = e.k;\n+----\n+logical_opt\t<REGEX>:(?s).*Comparison Join[^~]*~100,000 rows.*\n+\n+query II\n+EXPLAIN SELECT count(*) FROM (SELECT any_value(k) AS k FROM nodes GROUP BY id) s JOIN edges e ON s.k = e.k;\n+----\n+logical_opt\t<REGEX>:(?s).*Comparison Join[^~]*~100,000 rows.*\n+\n+# A unique grouping key does not stop arg_min from rejecting a NULL selector.\n+query II\n+EXPLAIN SELECT count(*)\n+FROM (SELECT arg_min(x, p) AS k FROM unique_list_values GROUP BY id) l\n+JOIN (SELECT arg_min(x, p) AS k FROM unique_list_values_right GROUP BY id) r ON l.k = r.k;\n+----\n+logical_opt\t<REGEX>:(?s).*Comparison Join[^~]*~8 rows.*\n+\n+query II\n+EXPLAIN SELECT count(*)\n+FROM (SELECT min_by(x, p) AS k FROM unique_list_values GROUP BY id) l\n+JOIN (SELECT min_by(x, p) AS k FROM unique_list_values_right GROUP BY id) r ON l.k = r.k;\n+----\n+logical_opt\t<REGEX>:(?s).*Comparison Join[^~]*~8 rows.*\n+\n+query II\n+EXPLAIN SELECT count(*)\n+FROM (SELECT max_by(x, p) AS k FROM unique_list_values GROUP BY id) l\n+JOIN (SELECT max_by(x, p) AS k FROM unique_list_values_right GROUP BY id) r ON l.k = r.k;\n+----\n+logical_opt\t<REGEX>:(?s).*Comparison Join[^~]*~8 rows.*\n+\n+query I\n+SELECT count(*)\n+FROM (SELECT arg_min(x, p) AS k FROM unique_list_values GROUP BY id) l\n+JOIN (SELECT arg_min(x, p) AS k FROM unique_list_values_right GROUP BY id) r ON l.k = r.k;\n+----\n+0\n+\n+# An identical CTAS input has no certified key and retains the fallback.\n+statement ok\n+CREATE TABLE unconstrained_values AS SELECT * FROM unique_list_values;\n+\n+statement ok\n+CREATE TABLE unconstrained_values_right AS SELECT * FROM unique_list_values;\n+\n+statement ok\n+ANALYZE;\n+\n+query II\n+EXPLAIN SELECT count(*)\n+FROM (SELECT first(x) AS k FROM unconstrained_values GROUP BY id) l\n+JOIN (SELECT first(x) AS k FROM unconstrained_values_right GROUP BY id) r ON l.k = r.k;\n+----\n+logical_opt\t<REGEX>:(?s).*Comparison Join[^~]*~8 rows.*\n+\n+query I\n+SELECT count(*)\n+FROM (SELECT first(x) AS k FROM unconstrained_values GROUP BY id) l\n+JOIN (SELECT first(x) AS k FROM unconstrained_values_right GROUP BY id) r ON l.k = r.k;\n+----\n+32\n+\n+# Non-null immediate UNIQUE keys provide the same proof as a primary key.\n+statement ok\n+CREATE TABLE nonnull_unique(id INTEGER UNIQUE NOT NULL, x INTEGER);\n+\n+statement ok\n+INSERT INTO nonnull_unique SELECT id, x FROM unique_list_values;\n+\n+statement ok\n+CREATE TABLE nullable_unique(id INTEGER UNIQUE, x INTEGER);\n+\n+statement ok\n+INSERT INTO nullable_unique SELECT CASE WHEN id = 7 THEN NULL ELSE id END, x FROM unique_list_values;\n+\n+statement ok\n+CREATE TABLE composite_unique(a INTEGER NOT NULL, b INTEGER NOT NULL, x INTEGER, UNIQUE(a, b));\n+\n+statement ok\n+INSERT INTO composite_unique SELECT id % 2, id // 2, x FROM unique_list_values;\n+\n+statement ok\n+ANALYZE;\n+\n+query II\n+EXPLAIN (FORMAT JSON) SELECT count(*)\n+FROM (SELECT first(x) AS k FROM nonnull_unique GROUP BY id) l\n+JOIN (SELECT first(x) AS k FROM nonnull_unique GROUP BY id) r ON l.k = r.k;\n+----\n+logical_opt\t<REGEX>:(?s).*\"Join Type\": \"INNER\",\\s*\"Conditions\": \"\\(k = k\\)\",\\s*\"Estimated Cardinality\": \"32\".*\n+\n+# Nullable keys do not certify singleton groups.\n+query II\n+EXPLAIN (FORMAT JSON) SELECT count(*)\n+FROM (SELECT first(x) AS k FROM nullable_unique GROUP BY id) l\n+JOIN (SELECT first(x) AS k FROM nullable_unique GROUP BY id) r ON l.k = r.k;\n+----\n+logical_opt\t<REGEX>:(?s).*\"Join Type\": \"INNER\",\\s*\"Conditions\": \"\\(k = k\\)\",\\s*\"Estimated Cardinality\": \"7\".*\n+\n+# A composite key must be completely covered by the grouping expressions.\n+query II\n+EXPLAIN (FORMAT JSON) SELECT count(*)\n+FROM (SELECT first(x) AS k FROM composite_unique GROUP BY a, b) l\n+JOIN (SELECT first(x) AS k FROM composite_unique GROUP BY a, b) r ON l.k = r.k;\n+----\n+logical_opt\t<REGEX>:(?s).*\"Join Type\": \"INNER\",\\s*\"Conditions\": \"\\(k = k\\)\",\\s*\"Estimated Cardinality\": \"32\".*\n+\n+query II\n+EXPLAIN (FORMAT JSON) SELECT count(*)\n+FROM (SELECT first(x) AS k FROM composite_unique GROUP BY a) l\n+JOIN (SELECT first(x) AS k FROM composite_unique GROUP BY a) r ON l.k = r.k;\n+----\n+logical_opt\t<REGEX>:(?s).*\"Join Type\": \"INNER\",\\s*\"Conditions\": \"\\(k = k\\)\",\\s*\"Estimated Cardinality\": \"2\".*\n+\n+# Casting a key to BOOLEAN can merge distinct key values.\n+query II\n+EXPLAIN SELECT count(*)\n+FROM (SELECT first(x) AS k FROM unique_list_values GROUP BY CAST(id AS BOOLEAN)) l\n+JOIN (SELECT first(x) AS k FROM unique_list_values_right GROUP BY CAST(id AS BOOLEAN)) r ON l.k = r.k;\n+----\n+logical_opt\t<REGEX>:(?s).*Comparison Join[^~]*~4 rows.*\n+\n+# TRY_CAST can turn distinct key values into NULL and cannot certify uniqueness.\n+query II\n+EXPLAIN SELECT count(*)\n+FROM (SELECT first(x) AS k FROM unique_list_values GROUP BY TRY_CAST(id AS TINYINT)) l\n+JOIN (SELECT first(x) AS k FROM unique_list_values_right GROUP BY TRY_CAST(id AS TINYINT)) r ON l.k = r.k;\n+----\n+logical_opt\t<REGEX>:(?s).*Comparison Join[^~]*~4 rows.*\n+\n+# Joins can multiply a key's rows and must invalidate the singleton proof.\n+query II\n+EXPLAIN SELECT count(*)\n+FROM (SELECT first(v.x) AS k FROM unique_list_values v CROSS JOIN range(2) t(i) GROUP BY v.id) l\n+JOIN (SELECT first(v.x) AS k FROM unique_list_values_right v CROSS JOIN range(2) t(i) GROUP BY v.id) r ON l.k = r.k;\n+----\n+logical_opt\t<REGEX>:(?s).*Comparison Join[^~]*~8 rows.*\n+\n+query I\n+SELECT count(*)\n+FROM (SELECT first(v.x) AS k FROM unique_list_values v CROSS JOIN range(2) t(i) GROUP BY v.id) l\n+JOIN (SELECT first(v.x) AS k FROM unique_list_values_right v CROSS JOIN range(2) t(i) GROUP BY v.id) r ON l.k = r.k;\n+----\n+32\n+\n+# The ordered rewrite must retain identity on the bound expression only.\n+query II\n+EXPLAIN SELECT count(*)\n+FROM (SELECT first(x ORDER BY p) AS k FROM unique_list_values GROUP BY id) l\n+JOIN (SELECT last(x ORDER BY p) AS k FROM unique_list_values_right GROUP BY id) r ON l.k = r.k;\n+----\n+logical_opt\t<REGEX>:(?s).*Comparison Join[^~]*~32 rows.*\n+\n+query II\n+EXPLAIN SELECT count(*)\n+FROM (SELECT any_value(x ORDER BY p) AS k FROM unique_list_values GROUP BY id) l\n+JOIN (SELECT any_value(x ORDER BY p) AS k FROM unique_list_values_right GROUP BY id) r ON l.k = r.k;\n+----\n+logical_opt\t<REGEX>:(?s).*Comparison Join[^~]*~32 rows.*\n+\n+query II\n+EXPLAIN SELECT count(*)\n+FROM (SELECT arg_min_null(x, p) AS k FROM unique_list_values GROUP BY id) l\n+JOIN (SELECT arg_max_null(x, p) AS k FROM unique_list_values_right GROUP BY id) r ON l.k = r.k;\n+----\n+logical_opt\t<REGEX>:(?s).*Comparison Join[^~]*~8 rows.*\n+\n+# Every group selects the common value from an otherwise larger input domain.\n+statement ok\n+CREATE TABLE biased_nodes AS\n+SELECT g::INT AS g, p::INT AS p, CASE WHEN p = 0 THEN 0 ELSE 2 * g + 1 END::INT AS k\n+FROM range(4) g_values(g), range(2) p_values(p);\n+\n+statement ok\n+CREATE TABLE biased_edges AS SELECT ((i % 3) * 3)::INT AS k FROM range(12) t(i);\n+\n+statement ok\n+ANALYZE;\n+\n+query II\n+EXPLAIN SELECT count(*)\n+FROM (SELECT DISTINCT ON (g) k FROM biased_nodes ORDER BY g, p) s\n+JOIN biased_edges e ON s.k = e.k;\n+----\n+logical_opt\t<REGEX>:(?s).*Comparison Join[^~]*~16 rows.*\n+\n+query I\n+SELECT count(*)\n+FROM (SELECT DISTINCT ON (g) k FROM biased_nodes ORDER BY g, p) s\n+JOIN biased_edges e ON s.k = e.k;\n+----\n+16\n+\n+# Computed CTE keys remain ineligible even when each group contains one row.\n+statement ok\n+CREATE TABLE ids_left AS SELECT i::INT AS id FROM range(8) t(i);\n+\n+statement ok\n+CREATE TABLE ids_right AS SELECT i::INT AS id FROM range(4) t(i);\n+\n+statement ok\n+ANALYZE;\n+\n+query II\n+EXPLAIN (FORMAT JSON)\n+WITH lhs AS MATERIALIZED (SELECT id, id % 2 AS k FROM ids_left),\n+     rhs AS MATERIALIZED (SELECT id % 2 AS k FROM ids_right)\n+SELECT count(*) FROM (SELECT first(k) AS k FROM lhs GROUP BY id) s JOIN rhs r ON s.k = r.k;\n+----\n+logical_opt\t<REGEX>:(?s).*\"Join Type\": \"INNER\",\\s*\"Conditions\": \"\\(k = k\\)\",\\s*\"Estimated Cardinality\": \"8\".*\n+\n+query I\n+WITH lhs AS MATERIALIZED (SELECT id, id % 2 AS k FROM ids_left),\n+     rhs AS MATERIALIZED (SELECT id % 2 AS k FROM ids_right)\n+SELECT count(*) FROM (SELECT first(k) AS k FROM lhs GROUP BY id) s JOIN rhs r ON s.k = r.k;\n+----\n+16\n+\n+# Temporary tables support deferred constraints with v1 storage.\n+# Keep this last so earlier cases can still exercise forced restarts.\n+statement ok\n+CREATE TEMP TABLE deferred_unique(id INTEGER PRIMARY KEY DEFERRED, x INTEGER);\n+\n+statement ok\n+INSERT INTO deferred_unique SELECT id, x FROM unique_list_values;\n+\n+statement ok\n+ANALYZE deferred_unique;\n+\n+# Deferred keys do not certify singleton groups.\n+query II\n+EXPLAIN (FORMAT JSON) SELECT count(*)\n+FROM (SELECT first(x) AS k FROM deferred_unique GROUP BY id) l\n+JOIN (SELECT first(x) AS k FROM deferred_unique GROUP BY id) r ON l.k = r.k;\n+----\n+logical_opt\t<REGEX>:(?s).*\"Join Type\": \"INNER\",\\s*\"Conditions\": \"\\(k = k\\)\",\\s*\"Estimated Cardinality\": \"8\".*\n"}], "contexts": {"incoming": {"title": "Who reaches this change?", "subtitle": "Unchanged clients, an unchanged wrapper and the modified implementation.", "caveat": "External API references are candidates. The wrapper-to-internal leg is syntax-bound.", "nodes": [{"id": "client", "title": "Relation manager", "sub": "relation_manager.cpp", "state": "Unchanged", "x": 20, "y": 38, "evidence": "context:255"}, {"id": "extractor", "title": "Statistics extractor", "sub": "relation_statistics_extractor.cpp", "state": "Unchanged", "x": 20, "y": 171, "evidence": "context:257"}, {"id": "wrapper", "title": "ExtractAggregationStats", "sub": "Two overloads inside modified file", "state": "Unchanged body", "x": 278, "y": 102, "evidence": "native:wrapper"}, {"id": "internal", "title": "ExtractAggregationStatsInternal", "sub": "relation_statistics_helper.cpp", "state": "Modified", "x": 536, "y": 102, "evidence": "native:helpers"}], "edges": [{"a": "client", "b": "wrapper", "label": "API reference", "evidence": "context:255", "kind": "candidate"}, {"a": "extractor", "b": "wrapper", "label": "API reference", "evidence": "context:257", "kind": "candidate"}, {"a": "wrapper", "b": "internal", "label": "Calls", "evidence": "native:wrapper", "kind": "syntax"}], "rows": [{"title": "Join-order client \u2192 public wrapper \u2192 changed internal", "body": "The first leg references an overloaded API. The second has a retained syntax binding.", "evidence": "context:255", "kind": "candidate"}, {"title": "Unchanged wrapper inside a modified file", "body": "Identical body bytes; moved from base lines 554 / 562 to head 688 / 696.", "evidence": "native:wrapper", "kind": "syntax"}, {"title": "Changed internal \u2192 new uniqueness proof", "body": "Follow the new helper without treating syntax matching as compiler resolution.", "evidence": "native:helpers", "kind": "syntax"}]}, "dependencies": {"title": "What does this change depend on?", "subtitle": "A bounded view of retained syntax bindings from the selected statistics change.", "caveat": "These are syntax matches. C++ receiver, qualifier and overload resolution still have known gaps.", "nodes": [{"id": "internal", "title": "ExtractAggregationStatsInternal", "sub": "Modified statistics implementation", "state": "Modified", "x": 20, "y": 104, "evidence": "native:helpers"}, {"id": "unique", "title": "HasUniqueGroups", "sub": "New singleton-group proof", "state": "Added", "x": 278, "y": 38, "evidence": "native:helpers"}, {"id": "source", "title": "HasBaseColumnSource", "sub": "New base-column proof", "state": "Added", "x": 278, "y": 171, "evidence": "native:HasBaseColumnSource"}, {"id": "value", "title": "GetValuePreservingColumn", "sub": "New expression proof", "state": "Added", "x": 536, "y": 104, "evidence": "native:GetValuePreservingColumn"}], "edges": [{"a": "internal", "b": "unique", "label": "Calls", "evidence": "native:helpers", "kind": "syntax"}, {"a": "internal", "b": "source", "label": "Calls", "evidence": "native:HasBaseColumnSource", "kind": "syntax"}, {"a": "internal", "b": "value", "label": "Calls", "evidence": "native:GetValuePreservingColumn", "kind": "syntax"}], "rows": [{"title": "Internal implementation \u2192 HasUniqueGroups", "body": "Retained native syntax path. Open its source and binding receipt.", "evidence": "native:helpers", "kind": "syntax"}, {"title": "Statistics reader \u2192 aggregate identity contract", "body": "The statistics guard reads the unchanged metadata API.", "evidence": "context:276", "kind": "contract"}, {"title": "Repository dependency binding receipt", "body": "Cross-file shared Rust graph candidate; function anchor is not an exact call site.", "evidence": "context:319", "kind": "syntax"}]}, "contract": {"title": "Which shared contracts deserve review?", "subtitle": "The changed producer and statistics reader share metadata with unchanged code.", "caveat": "Shared contract participation does not establish runtime impact on the recursive-CTE consumer.", "nodes": [{"id": "rewrite", "title": "RewriteOrderedFirst", "sub": "first_last_any.cpp", "state": "Modified", "x": 20, "y": 38, "evidence": "context:268"}, {"id": "stats", "title": "Statistics helper", "sub": "relation_statistics_helper.cpp", "state": "Modified", "x": 20, "y": 171, "evidence": "context:276"}, {"id": "identity", "title": "single_value_identity", "sub": "aggregate_function.hpp", "state": "Unchanged", "x": 278, "y": 102, "evidence": "context:268"}, {"id": "cte", "title": "Recursive CTE consumer", "sub": "physical_recursive_cte_keyed.cpp", "state": "Unchanged", "x": 536, "y": 102, "evidence": "context:267"}], "edges": [{"a": "rewrite", "b": "identity", "label": "Writes", "evidence": "context:268", "kind": "contract"}, {"a": "stats", "b": "identity", "label": "Reads", "evidence": "context:276", "kind": "contract"}, {"a": "cte", "b": "identity", "label": "Reads", "evidence": "context:267", "kind": "contract"}], "rows": [{"title": "Changed rewrite \u2192 shared identity contract", "body": "The rewrite propagates HasSingleValueIdentity into the replacement aggregate.", "evidence": "context:268", "kind": "contract"}, {"title": "Unchanged recursive-CTE consumer \u2192 same contract", "body": "A concrete review neighbor. Runtime propagation from this PR is not established.", "evidence": "context:267", "kind": "contract"}, {"title": "Unchanged dispatcher \u2192 possible rewrite callback", "body": "Inspect one dispatcher and all four registration witnesses.", "evidence": "context:266", "kind": "candidate"}]}, "config": {"title": "Where does test selection change?", "subtitle": "Two unchanged configurations inherit a changed shared skip list.", "caveat": "Declared exclusions only. This experiment did not execute the DuckDB test suite.", "nodes": [{"id": "disable", "title": "disable_optimizer.json", "sub": "Inherits shared list", "state": "Unchanged", "x": 20, "y": 38, "evidence": "context:237"}, {"id": "serializer", "title": "verify_serializer.json", "sub": "Inherits shared list", "state": "Unchanged", "x": 20, "y": 171, "evidence": "context:249"}, {"id": "shared", "title": "stats_dependent_tests.json", "sub": "Shared skip list", "state": "Modified", "x": 278, "y": 102, "evidence": "context:243"}, {"id": "test", "title": "distinct_on_stats_propagation.test", "sub": "New statistics regression", "state": "Added", "x": 536, "y": 102, "evidence": "context:243"}], "edges": [{"a": "disable", "b": "shared", "label": "Inherits", "evidence": "context:237", "kind": "config"}, {"a": "serializer", "b": "shared", "label": "Inherits", "evidence": "context:249", "kind": "config"}, {"a": "shared", "b": "test", "label": "Excludes", "evidence": "context:243", "kind": "config"}], "rows": [{"title": "disable_optimizer \u2192 shared skip list \u2192 new test", "body": "Unchanged configuration inherits the newly declared exclusion.", "evidence": "context:237", "kind": "config"}, {"title": "verify_serializer \u2192 shared skip list \u2192 new test", "body": "A second unchanged configuration inherits the exclusion.", "evidence": "context:249", "kind": "config"}, {"title": "verify_aggregate_state_export \u2192 new test", "body": "The modified execution configuration directly excludes the test.", "evidence": "context:246", "kind": "config"}]}}, "evidence": {"context:255": {"source_path": "src/optimizer/join_order/relation_manager.cpp", "target_path": "src/optimizer/relation_statistics/relation_statistics_helper.cpp", "relation": "references_aggregate_stats_api", "evidence_class": "source_reference_candidate", "source_changed": false, "target_changed": true, "witness_path": "src/optimizer/join_order/relation_manager.cpp", "witness_line": 623, "witness_text": "auto operator_stats = RelationStatisticsHelper::ExtractAggregationStats(aggr, child_stats, extractor, true);", "witness_blob_oid": "722f7dbbc7feb7af0ad422d9cc27d496f1dc42ec", "revision": "d29b26bb92ac940a0807be0f43cc385b31623839", "target_api": "RelationStatisticsHelper::ExtractAggregationStats", "caveat": "Source reference; overload/declaration-to-definition binding is not compiler-proven", "id": "context:255", "location": "Exact source witness", "lines": [{"n": 621, "text": "\t\tauto &aggr = op->Cast<LogicalAggregate>();"}, {"n": 622, "text": "\t\tRelationStatsExtractor extractor(context, optimizer.GetSourceStats());"}, {"n": 623, "text": "\t\tauto operator_stats = RelationStatisticsHelper::ExtractAggregationStats(aggr, child_stats, extractor, true);"}, {"n": 624, "text": "\t\tif (!operator_stats) {"}, {"n": 625, "text": "\t\t\tstats_complete = false;"}, {"n": 626, "text": "\t\t\treturn false;"}], "artifact": "relationships.json", "artifact_index": 255}, "context:257": {"source_path": "src/optimizer/relation_statistics/relation_statistics_extractor.cpp", "target_path": "src/optimizer/relation_statistics/relation_statistics_helper.cpp", "relation": "references_aggregate_stats_api", "evidence_class": "source_reference_candidate", "source_changed": false, "target_changed": true, "witness_path": "src/optimizer/relation_statistics/relation_statistics_extractor.cpp", "witness_line": 124, "witness_text": "return RelationStatisticsHelper::ExtractAggregationStats(op.Cast<LogicalAggregate>(), child_stats[0], *this);", "witness_blob_oid": "f8f7a1eda024525a3456505fae6e7b9a82487ac7", "revision": "d29b26bb92ac940a0807be0f43cc385b31623839", "target_api": "RelationStatisticsHelper::ExtractAggregationStats", "caveat": "Source reference; overload/declaration-to-definition binding is not compiler-proven", "id": "context:257", "location": "Exact source witness", "lines": [{"n": 122, "text": "\t}"}, {"n": 123, "text": "\tif (op.type == LogicalOperatorType::LOGICAL_AGGREGATE_AND_GROUP_BY && child_stats.size() == 1) {"}, {"n": 124, "text": "\t\treturn RelationStatisticsHelper::ExtractAggregationStats(op.Cast<LogicalAggregate>(), child_stats[0], *this);"}, {"n": 125, "text": "\t}"}, {"n": 126, "text": "\treturn RelationStatisticsHelper::ExtractOperatorStats(op, context, child_stats);"}, {"n": 127, "text": "}"}], "artifact": "relationships.json", "artifact_index": 257}, "native:wrapper": {"id": "native:wrapper", "relation": "Wrapper calls changed internal helper", "evidence_class": "native_bound_syntax", "witness_path": "src/optimizer/relation_statistics/relation_statistics_helper.cpp", "witness_line": 693, "revision": "d29b26bb92ac940a0807be0f43cc385b31623839", "witness_text": "\t// Join reconstruction has already published the selected input cardinalities.\n\tRelationStatsExtractor extractor(context);\n\treturn ExtractAggregationStatsInternal(aggregate, child_stats, extractor, true);\n}\n\noptional<RelationStats> RelationStatisticsHelper::ExtractAggregationStats(LogicalAggregate &aggregate,", "lines": [{"n": 691, "text": "\t// Join reconstruction has already published the selected input cardinalities."}, {"n": 692, "text": "\tRelationStatsExtractor extractor(context);"}, {"n": 693, "text": "\treturn ExtractAggregationStatsInternal(aggregate, child_stats, extractor, true);"}, {"n": 694, "text": "}"}, {"n": 695, "text": ""}, {"n": 696, "text": "optional<RelationStats> RelationStatisticsHelper::ExtractAggregationStats(LogicalAggregate &aggregate,"}], "caveat": "Syntax binding from the retained two-file graph. The wrapper body is unchanged; compiler resolution is not claimed.", "artifact": "native-incoming-paths.json", "artifact_index": 6, "location": "Source context anchor; exact graph path is in the retained receipt"}, "native:helpers": {"id": "native:helpers", "relation": "Internal helper calls uniqueness proof", "evidence_class": "native_bound_syntax", "witness_path": "src/optimizer/relation_statistics/relation_statistics_helper.cpp", "witness_line": 627, "revision": "d29b26bb92ac940a0807be0f43cc385b31623839", "witness_text": "\t\t\t}\n\t\t}\n\t\thas_unique_groups = HasUniqueGroups(*aggregate.children[0], groups);\n\t\tif (has_unique_groups) {\n\t\t\tresult.cardinality = child_stats.cardinality;\n\t\t}", "lines": [{"n": 625, "text": "\t\t\t}"}, {"n": 626, "text": "\t\t}"}, {"n": 627, "text": "\t\thas_unique_groups = HasUniqueGroups(*aggregate.children[0], groups);"}, {"n": 628, "text": "\t\tif (has_unique_groups) {"}, {"n": 629, "text": "\t\t\tresult.cardinality = child_stats.cardinality;"}, {"n": 630, "text": "\t\t}"}], "caveat": "Syntax binding, not compiler proof. Open the frozen path record for graph-scoped identities.", "artifact": "native-incoming-paths.json", "artifact_index": 3, "location": "Source context anchor; exact graph path is in the retained receipt"}, "native:HasBaseColumnSource": {"id": "native:HasBaseColumnSource", "relation": "Internal implementation calls HasBaseColumnSource", "evidence_class": "native_bound_syntax", "witness_path": "src/optimizer/relation_statistics/relation_statistics_helper.cpp", "witness_line": 669, "revision": "d29b26bb92ac940a0807be0f43cc385b31623839", "witness_text": "\t\t\t\tauto binding = agg_children[0]->Cast<BoundColumnRefExpression>().Binding();\n\t\t\t\tauto child_column = child_columns.Find(binding);\n\t\t\t\tif (child_column && HasBaseColumnSource(*aggregate.children[0], binding)) {\n\t\t\t\t\ttotal_domain = child_column->total_domain;\n\t\t\t\t\tcurrent_domain = child_column->current_domain;\n\t\t\t\t}", "lines": [{"n": 667, "text": "\t\t\t\tauto binding = agg_children[0]->Cast<BoundColumnRefExpression>().Binding();"}, {"n": 668, "text": "\t\t\t\tauto child_column = child_columns.Find(binding);"}, {"n": 669, "text": "\t\t\t\tif (child_column && HasBaseColumnSource(*aggregate.children[0], binding)) {"}, {"n": 670, "text": "\t\t\t\t\ttotal_domain = child_column->total_domain;"}, {"n": 671, "text": "\t\t\t\t\tcurrent_domain = child_column->current_domain;"}, {"n": 672, "text": "\t\t\t\t}"}], "caveat": "Retained syntax binding, not compiler resolution.", "artifact": "native-incoming-paths.json", "artifact_index": 0, "location": "Source span with retained native path"}, "native:GetValuePreservingColumn": {"id": "native:GetValuePreservingColumn", "relation": "Internal implementation calls GetValuePreservingColumn", "evidence_class": "native_bound_syntax", "witness_path": "src/optimizer/relation_statistics/relation_statistics_helper.cpp", "witness_line": 622, "revision": "d29b26bb92ac940a0807be0f43cc385b31623839", "witness_text": "\t\tcolumn_binding_set_t groups;\n\t\tfor (auto &group : aggregate.groups) {\n\t\t\tauto binding = GetValuePreservingColumn(*group);\n\t\t\tif (binding) {\n\t\t\t\tgroups.insert(*binding);\n\t\t\t}", "lines": [{"n": 620, "text": "\t\tcolumn_binding_set_t groups;"}, {"n": 621, "text": "\t\tfor (auto &group : aggregate.groups) {"}, {"n": 622, "text": "\t\t\tauto binding = GetValuePreservingColumn(*group);"}, {"n": 623, "text": "\t\t\tif (binding) {"}, {"n": 624, "text": "\t\t\t\tgroups.insert(*binding);"}, {"n": 625, "text": "\t\t\t}"}], "caveat": "Retained syntax binding, not compiler resolution.", "artifact": "native-incoming-paths.json", "artifact_index": 10, "location": "Source span with retained native path"}, "context:268": {"source_path": "src/function/aggregate/distributive/first_last_any.cpp", "target_path": "src/include/duckdb/function/aggregate_function.hpp", "relation": "writes_shared_identity_contract", "evidence_class": "shared_contract_reference", "source_changed": true, "target_changed": false, "witness_path": "src/function/aggregate/distributive/first_last_any.cpp", "witness_line": 57, "witness_text": "result->FunctionMutable().SetSingleValueIdentity(aggregate->Function().HasSingleValueIdentity());", "witness_blob_oid": "0ba568cb72b05ba00c313dfd8e6d58d158e6b656", "revision": "d29b26bb92ac940a0807be0f43cc385b31623839", "contract": "AggregateFunction.properties.single_value_identity", "contract_definition": {"path": "src/include/duckdb/function/aggregate_function.hpp", "field_line": 353, "reader_line": 401, "writer_line": 402}, "caveat": "Shared contract participation; not a proven runtime flow between these files", "id": "context:268", "location": "Exact source witness", "lines": [{"n": 55, "text": "\t                                           aggregate->GetAggregateType());"}, {"n": 56, "text": "\t// The encoded sort key is non-NULL, so a singleton still returns its input value."}, {"n": 57, "text": "\tresult->FunctionMutable().SetSingleValueIdentity(aggregate->Function().HasSingleValueIdentity());"}, {"n": 58, "text": "\treturn unique_ptr_cast<BoundAggregateExpression, Expression>(std::move(result));"}, {"n": 59, "text": "}"}, {"n": 60, "text": ""}], "artifact": "relationships.json", "artifact_index": 268}, "context:276": {"source_path": "src/optimizer/relation_statistics/relation_statistics_helper.cpp", "target_path": "src/include/duckdb/function/aggregate_function.hpp", "relation": "reads_shared_identity_contract", "evidence_class": "shared_contract_reference", "source_changed": true, "target_changed": false, "witness_path": "src/optimizer/relation_statistics/relation_statistics_helper.cpp", "witness_line": 664, "witness_text": "if (bound_aggregate.Function().HasSingleValueIdentity() && !bound_aggregate.GetFilter() &&", "witness_blob_oid": "6eac019d1e36990aae2d28be02ad8657892cb8d5", "revision": "d29b26bb92ac940a0807be0f43cc385b31623839", "contract": "AggregateFunction.properties.single_value_identity", "contract_definition": {"path": "src/include/duckdb/function/aggregate_function.hpp", "field_line": 353, "reader_line": 401, "writer_line": 402}, "caveat": "Shared contract participation; not a proven runtime flow between these files", "id": "context:276", "location": "Exact source witness", "lines": [{"n": 662, "text": "\t\t\tauto &bound_aggregate = expression->Cast<BoundAggregateExpression>();"}, {"n": 663, "text": "\t\t\tauto &agg_children = bound_aggregate.GetChildren();"}, {"n": 664, "text": "\t\t\tif (bound_aggregate.Function().HasSingleValueIdentity() && !bound_aggregate.GetFilter() &&"}, {"n": 665, "text": "\t\t\t    !agg_children.empty() && expression->GetReturnType() == agg_children[0]->GetReturnType() &&"}, {"n": 666, "text": "\t\t\t    agg_children[0]->GetExpressionClass() == ExpressionClass::BOUND_COLUMN_REF) {"}, {"n": 667, "text": "\t\t\t\tauto binding = agg_children[0]->Cast<BoundColumnRefExpression>().Binding();"}], "artifact": "relationships.json", "artifact_index": 276}, "context:267": {"source_path": "src/execution/operator/set/physical_recursive_cte_keyed.cpp", "target_path": "src/include/duckdb/function/aggregate_function.hpp", "relation": "reads_shared_identity_contract", "evidence_class": "shared_contract_reference", "source_changed": false, "target_changed": false, "witness_path": "src/execution/operator/set/physical_recursive_cte_keyed.cpp", "witness_line": 42, "witness_text": "aggregate.Function().HasSingleValueIdentity() && !children.empty() &&", "witness_blob_oid": "69a0f6255be6e0485ba15440a261a67ea3db85e1", "revision": "d29b26bb92ac940a0807be0f43cc385b31623839", "contract": "AggregateFunction.properties.single_value_identity", "contract_definition": {"path": "src/include/duckdb/function/aggregate_function.hpp", "field_line": 353, "reader_line": 401, "writer_line": 402}, "caveat": "Shared contract participation; not a proven runtime flow between these files", "id": "context:267", "location": "Exact source witness", "lines": [{"n": 40, "text": "\t\tauto &children = aggregate.GetChildren();"}, {"n": 41, "text": "\t\tresult.can_reuse_new_group_candidates ="}, {"n": 42, "text": "\t\t    aggregate.Function().HasSingleValueIdentity() && !children.empty() &&"}, {"n": 43, "text": "\t\t    children[0]->GetExpressionClass() == ExpressionClass::BOUND_REF &&"}, {"n": 44, "text": "\t\t    children[0]->Cast<BoundReferenceExpression>().Index() == op.payload_idx[payload_idx];"}, {"n": 45, "text": "\t}"}], "artifact": "relationships.json", "artifact_index": 267}, "context:266": {"source_path": "src/optimizer/aggregate_rewrite.cpp", "target_path": "src/function/aggregate/distributive/first_last_any.cpp", "relation": "dispatches_possible_registered_rewrite", "evidence_class": "callback_dispatch_candidate", "source_changed": false, "target_changed": true, "witness_path": "src/optimizer/aggregate_rewrite.cpp", "witness_line": 58, "witness_text": "auto rewrite = input.aggregate.Function().GetDirectRewriteCallback()(input);", "witness_blob_oid": "862ba08cef7ef508a700d24350465dda41a770f7", "revision": "d29b26bb92ac940a0807be0f43cc385b31623839", "registration_witnesses": [{"path": "src/function/aggregate/distributive/first_last_any.cpp", "line": 492, "symbol": "RewriteOrderedFirst", "text": "function.SetDirectRewriteCallback(RewriteOrderedFirst<LAST, SKIP_NULLS>);"}, {"path": "src/function/aggregate/distributive/first_last_any.cpp", "line": 517, "symbol": "RewriteOrderedFirst", "text": "function.SetDirectRewriteCallback(RewriteOrderedFirst<LAST, SKIP_NULLS>);"}, {"path": "src/function/aggregate/distributive/first_last_any.cpp", "line": 542, "symbol": "RewriteOrderedFirst", "text": "fun.SetDirectRewriteCallback(RewriteOrderedFirst<false, false>);"}, {"path": "src/function/aggregate/distributive/first_last_any.cpp", "line": 551, "symbol": "RewriteOrderedFirst", "text": "fun.SetDirectRewriteCallback(RewriteOrderedFirst<true, false>);"}], "caveat": "Possible callback flow from registration and dispatcher; not a direct bound call or runtime trace", "id": "context:266", "location": "Exact source witness", "lines": [{"n": 56, "text": "\t\treturn nullptr;"}, {"n": 57, "text": "\t}"}, {"n": 58, "text": "\tauto rewrite = input.aggregate.Function().GetDirectRewriteCallback()(input);"}, {"n": 59, "text": "\tif (!rewrite) {"}, {"n": 60, "text": "\t\treturn nullptr;"}, {"n": 61, "text": "\t}"}], "artifact": "relationships.json", "artifact_index": 266}, "context:243": {"source_path": "test/configs/stats_dependent_tests.json", "target_path": "test/sql/optimizer/statistics/distinct_on_stats_propagation.test", "relation": "skips_test_in_mode", "evidence_class": "config_value", "source_changed": true, "target_changed": true, "witness_path": "test/configs/stats_dependent_tests.json", "witness_line": 40, "witness_text": "\"test/sql/optimizer/statistics/distinct_on_stats_propagation.test\",", "witness_blob_oid": "7457b6bce5591a0984135f9d551b794441f64609", "revision": "d29b26bb92ac940a0807be0f43cc385b31623839", "reason": "Statistics are not set without optimizers.", "on_init": null, "id": "context:243", "location": "Exact source witness", "lines": [{"n": 38, "text": "        \"test/sql/storage/list_length_stats_nested_optimistic.test\","}, {"n": 39, "text": "        \"test/sql/optimizer/statistics/list_length_stats_propagation.test\","}, {"n": 40, "text": "        \"test/sql/optimizer/statistics/distinct_on_stats_propagation.test\","}, {"n": 41, "text": "        \"test/sql/storage/list_length_stats_old_version.test\","}, {"n": 42, "text": "        \"test/sql/types/nested/array/array_statistics.test\","}, {"n": 43, "text": "        \"test/sql/vacuum/test_analyze.test\","}], "artifact": "relationships.json", "artifact_index": 243}, "context:237": {"source_path": "test/configs/disable_optimizer.json", "target_path": "test/configs/stats_dependent_tests.json", "relation": "inherits_skip_tests", "evidence_class": "config_value", "source_changed": false, "target_changed": true, "witness_path": "test/configs/disable_optimizer.json", "witness_line": 5, "witness_text": "\"inherit_skip_tests\": \"test/configs/stats_dependent_tests.json\",", "witness_blob_oid": "9e2330e8854ff5de38dcdc0a491b5c17885ee6ca", "revision": "d29b26bb92ac940a0807be0f43cc385b31623839", "id": "context:237", "location": "Exact source witness", "lines": [{"n": 3, "text": "  \"on_init\": \"SET debug_disable_optimizer=true;\","}, {"n": 4, "text": "  \"skip_compiled\": \"true\","}, {"n": 5, "text": "  \"inherit_skip_tests\": \"test/configs/stats_dependent_tests.json\","}, {"n": 6, "text": "  \"skip_tests\": ["}, {"n": 7, "text": "    {"}, {"n": 8, "text": "      \"reason\": \"Checks the plan of an executed query through the profiler output.\","}], "artifact": "relationships.json", "artifact_index": 237}, "context:249": {"source_path": "test/configs/verify_serializer.json", "target_path": "test/configs/stats_dependent_tests.json", "relation": "inherits_skip_tests", "evidence_class": "config_value", "source_changed": false, "target_changed": true, "witness_path": "test/configs/verify_serializer.json", "witness_line": 5, "witness_text": "\"inherit_skip_tests\": \"test/configs/stats_dependent_tests.json\",", "witness_blob_oid": "e410713085d6a974b1c326d9adda9ba15e7a1104", "revision": "d29b26bb92ac940a0807be0f43cc385b31623839", "id": "context:249", "location": "Exact source witness", "lines": [{"n": 3, "text": "  \"on_init\": \"SET debug_verify_serializer=true\","}, {"n": 4, "text": "  \"skip_compiled\": \"true\","}, {"n": 5, "text": "  \"inherit_skip_tests\": \"test/configs/stats_dependent_tests.json\","}, {"n": 6, "text": "  \"skip_tests\": ["}, {"n": 7, "text": "    {"}, {"n": 8, "text": "      \"reason\": \"Measures CTE restrictions using planner dependencies deliberately discarded by serialization\","}], "artifact": "relationships.json", "artifact_index": 249}, "context:246": {"source_path": "test/configs/verify_aggregate_state_export.json", "target_path": "test/sql/optimizer/statistics/distinct_on_stats_propagation.test", "relation": "skips_test_in_mode", "evidence_class": "config_value", "source_changed": true, "target_changed": true, "witness_path": "test/configs/verify_aggregate_state_export.json", "witness_line": 50, "witness_text": "\"test/sql/optimizer/statistics/distinct_on_stats_propagation.test\",", "witness_blob_oid": "e59f36a380661403a63cf36bc89c0712b8e00cb3", "revision": "d29b26bb92ac940a0807be0f43cc385b31623839", "reason": "Explain/profiling output is different due to verification", "on_init": "SET debug_verify_aggregate_state_export=true;", "id": "context:246", "location": "Exact source witness", "lines": [{"n": 48, "text": "        \"test/sql/copy/parquet/read_single_parquet_file.test\","}, {"n": 49, "text": "        \"test/sql/optimizer/aggregate_stats_propagation.test\","}, {"n": 50, "text": "        \"test/sql/optimizer/statistics/distinct_on_stats_propagation.test\","}, {"n": 51, "text": "        \"test/sql/optimizer/scalar_aggregate_merge_files_plan.test\","}, {"n": 52, "text": "        \"test/sql/optimizer/scalar_aggregate_merge_plan.test\","}, {"n": 53, "text": "        \"test/sql/optimizer/scalar_aggregate_merge_tpcds_plan.test\","}], "artifact": "relationships.json", "artifact_index": 246}, "context:319": {"source_path": "src/optimizer/relation_statistics/relation_statistics_helper.cpp", "target_path": "src/include/duckdb/parser/base_expression.hpp", "relation": "calls_dependency_in_shared_syntax_graph", "evidence_class": "native_bound_syntax", "source_changed": true, "target_changed": false, "witness_path": "src/optimizer/relation_statistics/relation_statistics_helper.cpp", "witness_line": 558, "witness_text": "src/optimizer/relation_statistics/relation_statistics_helper.cpp :: ExtractAggregationStatsInternal -> src/include/duckdb/parser/base_expression.hpp :: GetExpressionClass [bind=Some(\"method_crate_singleton\"), cross_file=true]", "witness_blob_oid": "6eac019d1e36990aae2d28be02ad8657892cb8d5", "revision": "d29b26bb92ac940a0807be0f43cc385b31623839", "source_symbol": "ExtractAggregationStatsInternal", "target_symbol": "GetExpressionClass", "changed_symbol_seed": true, "bind_method": "Some(\"method_crate_singleton\")", "evidence_artifact": "docs/research/artifacts/2026-11-17-cpp-extraction-gap-lab/rust-full-repository-seed-witnesses.txt", "evidence_artifact_line": 52, "witness_location": "Core receipt is the binding witness; repository line is function start when unique, not the call site", "id": "context:319", "location": "Exact source witness", "lines": [{"n": 556, "text": "}"}, {"n": 557, "text": ""}, {"n": 558, "text": "static optional<RelationStats> ExtractAggregationStatsInternal(LogicalAggregate &aggregate,"}, {"n": 559, "text": "                                                               const RelationStats &child_stats,"}, {"n": 560, "text": "                                                               RelationStatsExtractor &extractor,"}, {"n": 561, "text": "                                                               bool use_operator_cardinality) {"}], "artifact": "relationships.json", "artifact_index": 319}, "context:114": {"source_path": "src/function/aggregate/distributive/first_last_any.cpp", "target_path": "src/function/aggregate/distributive/CMakeLists.txt", "relation": "listed_in_build_target", "evidence_class": "build_declaration", "source_changed": true, "target_changed": false, "witness_path": "src/function/aggregate/distributive/CMakeLists.txt", "witness_line": 1, "witness_text": "add_library_unity(duckdb_aggr_distr OBJECT count.cpp decimal_avg.cpp\n                  first_last_any.cpp minmax.cpp)", "witness_blob_oid": "d7e51e1411270b494afa438d1328aa169ae66ae6", "revision": "d29b26bb92ac940a0807be0f43cc385b31623839", "component": "duckdb_aggr_distr", "id": "context:114", "location": "Exact source witness", "lines": [{"n": 1, "text": "add_library_unity(duckdb_aggr_distr OBJECT count.cpp decimal_avg.cpp"}, {"n": 2, "text": "                  first_last_any.cpp minmax.cpp)"}, {"n": 3, "text": "set(ALL_OBJECT_FILES"}, {"n": 4, "text": "    ${ALL_OBJECT_FILES} $<TARGET_OBJECTS:duckdb_aggr_distr>"}], "artifact": "relationships.json", "artifact_index": 114}, "context:157": {"source_path": "src/optimizer/relation_statistics/relation_statistics_extractor.cpp", "target_path": "src/optimizer/relation_statistics/CMakeLists.txt", "relation": "listed_in_build_target", "evidence_class": "build_declaration", "source_changed": false, "target_changed": false, "witness_path": "src/optimizer/relation_statistics/CMakeLists.txt", "witness_line": 1, "witness_text": "add_library_unity(\n  duckdb_optimizer_relation_statistics\n  OBJECT\n  relation_statistics.cpp\n  relation_statistics_extractor.cpp\n  relation_statistics_helper.cpp\n  relation_statistics_operator.cpp\n  relation_statistics_scan.cpp)", "witness_blob_oid": "94d04dd6cf092a944afb6ee89389ef475422af69", "revision": "d29b26bb92ac940a0807be0f43cc385b31623839", "component": "duckdb_optimizer_relation_statistics", "id": "context:157", "location": "Exact source witness", "lines": [{"n": 1, "text": "add_library_unity("}, {"n": 2, "text": "  duckdb_optimizer_relation_statistics"}, {"n": 3, "text": "  OBJECT"}, {"n": 4, "text": "  relation_statistics.cpp"}], "artifact": "relationships.json", "artifact_index": 157}, "context:156": {"source_path": "src/optimizer/join_order/relation_manager.cpp", "target_path": "src/optimizer/join_order/CMakeLists.txt", "relation": "listed_in_build_target", "evidence_class": "build_declaration", "source_changed": false, "target_changed": false, "witness_path": "src/optimizer/join_order/CMakeLists.txt", "witness_line": 1, "witness_text": "add_library_unity(\n  duckdb_optimizer_join_order\n  OBJECT\n  query_graph.cpp\n  filter_info.cpp\n  join_predicate.cpp\n  join_relation_set.cpp\n  join_node.cpp\n  join_order_operator.cpp\n  join_order_optimizer.cpp\n  join_order_source_stats.cpp\n  cardinality_estimator.cpp\n  cost_model.cpp\n  plan_enumerator.cpp\n  relation_manager.cpp\n  query_graph_manager.cpp)", "witness_blob_oid": "94eb6ce5fe48fb20cc162b06b09e39ee44cffcf5", "revision": "d29b26bb92ac940a0807be0f43cc385b31623839", "component": "duckdb_optimizer_join_order", "id": "context:156", "location": "Exact source witness", "lines": [{"n": 1, "text": "add_library_unity("}, {"n": 2, "text": "  duckdb_optimizer_join_order"}, {"n": 3, "text": "  OBJECT"}, {"n": 4, "text": "  query_graph.cpp"}], "artifact": "relationships.json", "artifact_index": 156}, "context:147": {"source_path": "src/optimizer/aggregate_rewrite.cpp", "target_path": "src/optimizer/CMakeLists.txt", "relation": "listed_in_build_target", "evidence_class": "build_declaration", "source_changed": false, "target_changed": false, "witness_path": "src/optimizer/CMakeLists.txt", "witness_line": 10, "witness_text": "add_library_unity(\n  duckdb_optimizer\n  OBJECT\n  aggregate_rewrite.cpp\n  aggregate_function_rewriter.cpp\n  aggregate_reuse.cpp\n  apply_functional_dependencies.cpp\n  materialized_aggregate_reuse.cpp\n  aggregate_rewrite_helper.cpp\n  build_probe_side_optimizer.cpp\n  column_binding_replacer.cpp\n  column_lifetime_analyzer.cpp\n  constant_or_null_simplification.cpp\n  empty_result_pullup.cpp\n  common_aggregate_optimizer.cpp\n  common_subplan_optimizer.cpp\n  compressed_materialization.cpp\n  cse_optimizer.cpp\n  cte_filter_pusher.cpp\n  cte_join_filter_collector.cpp\n  cte_join_filter_pusher.cpp\n  cte_runtime_filter_pushdown.cpp\n  cte_inlining.cpp\n  deliminator.cpp\n  duplicate_eliminated_domain_optimizer.cpp\n  multi_stage_aggregate_rewriter.cpp\n  expression_heuristics.cpp\n  expression_rewriter.cpp\n  filter_statistics.cpp\n  filter_combiner.cpp\n  filter_pullup.cpp\n  filter_pushdown.cpp\n  grouping_sets_optimizer.cpp\n  in_clause_rewriter.cpp\n  join_filter_pushdown_optimizer.cpp\n  runtime_filter_cast.cpp\n  late_materialization.cpp\n  late_materialization_helper.cpp\n  optimizer.cpp\n  outer_join_simplification.cpp\n  partial_aggregate_pushdown.cpp\n  partitioned_execution.cpp\n  join_elimination.cpp\n  join_build_properties.cpp\n  recursive_join_planning.cpp\n  regex_range_filter.cpp\n  remove_duplicate_groups.cpp\n  remove_redundant_order.cpp\n  remove_redundant_order_keys.cpp\n  remove_unused_columns.cpp\n  row_group_pruner.cpp\n  statistics_propagator.cpp\n  limit_pushdown.cpp\n  topn_optimizer.cpp\n  topn_window_elimination.cpp\n  unnest_rewriter.cpp\n  window_join_pushdown.cpp\n  window_self_join.cpp\n  row_number_rewriter.cpp\n  sampling_pushdown.cpp\n  scalar_aggregate_merger.cpp\n  type_pushdown.cpp\n  scalar_fn_pushdown.cpp\n  self_join_aggregate.cpp\n  projection_pullup.cpp\n  projection_placement.cpp\n  remote_pushdown_optimizer.cpp\n  disjunctive_join_rewriter.cpp)", "witness_blob_oid": "f408695f8f87cac0b83cf931d5c2db9119b1d1f9", "revision": "d29b26bb92ac940a0807be0f43cc385b31623839", "component": "duckdb_optimizer", "id": "context:147", "location": "Exact source witness", "lines": [{"n": 8, "text": "add_subdirectory(statistics)"}, {"n": 9, "text": ""}, {"n": 10, "text": "add_library_unity("}, {"n": 11, "text": "  duckdb_optimizer"}, {"n": 12, "text": "  OBJECT"}, {"n": 13, "text": "  aggregate_rewrite.cpp"}], "artifact": "relationships.json", "artifact_index": 147}, "context:105": {"source_path": "src/execution/operator/set/physical_recursive_cte_keyed.cpp", "target_path": "src/execution/operator/set/CMakeLists.txt", "relation": "listed_in_build_target", "evidence_class": "build_declaration", "source_changed": false, "target_changed": false, "witness_path": "src/execution/operator/set/CMakeLists.txt", "witness_line": 1, "witness_text": "add_library_unity(\n  duckdb_operator_set\n  OBJECT\n  physical_union.cpp\n  physical_recursive_cte.cpp\n  physical_recursive_cte_delta.cpp\n  physical_recursive_cte_keyed.cpp\n  physical_recursive_cte_runtime.cpp\n  physical_recursive_cte_state.cpp\n  physical_recursive_cte_union.cpp\n  physical_cte.cpp)", "witness_blob_oid": "70700bfd2a47745adbb578f91d761c0e84708379", "revision": "d29b26bb92ac940a0807be0f43cc385b31623839", "component": "duckdb_operator_set", "id": "context:105", "location": "Exact source witness", "lines": [{"n": 1, "text": "add_library_unity("}, {"n": 2, "text": "  duckdb_operator_set"}, {"n": 3, "text": "  OBJECT"}, {"n": 4, "text": "  physical_union.cpp"}], "artifact": "relationships.json", "artifact_index": 105}, "context:221": {"source_path": "test/optimizer/relation_statistics.cpp", "target_path": "test/optimizer/CMakeLists.txt", "relation": "listed_in_build_target", "evidence_class": "build_declaration", "source_changed": false, "target_changed": false, "witness_path": "test/optimizer/CMakeLists.txt", "witness_line": 1, "witness_text": "add_library_unity(\n  test_optimizer\n  OBJECT\n  aggregate_rewrite.cpp\n  aggregate_statistics.cpp\n  bound_function_identity.cpp\n  build_probe_side_optimizer.cpp\n  column_binding_rewrite.cpp\n  common_subplan.cpp\n  common_subplan_nonserializable.cpp\n  dynamic_filter_groups.cpp\n  expression_type_serialization.cpp\n  filter_pushdown_alias.cpp\n  function_invocation_serialization.cpp\n  join_filter_optimizer_hooks.cpp\n  join_order_conflict_detector.cpp\n  join_order_enumeration.cpp\n  logical_plan_verification.cpp\n  multi_file_metadata.cpp\n  multi_file_repeatability.cpp\n  relation_statistics.cpp\n  remove_unused_columns.cpp\n  runtime_filter_schedule.cpp\n  source_cardinality_bounds.cpp\n  unchecked_cast.cpp\n  union_alls.cpp)", "witness_blob_oid": "58d6dae3ffd2a0a5a88fd8a1283f355976edded7", "revision": "d29b26bb92ac940a0807be0f43cc385b31623839", "component": "test_optimizer", "id": "context:221", "location": "Exact source witness", "lines": [{"n": 1, "text": "add_library_unity("}, {"n": 2, "text": "  test_optimizer"}, {"n": 3, "text": "  OBJECT"}, {"n": 4, "text": "  aggregate_rewrite.cpp"}], "artifact": "relationships.json", "artifact_index": 221}}, "components": [{"target": "duckdb_aggr_distr", "caption": "Aggregate functions", "members": ["src/function/aggregate/distributive/first_last_any.cpp", "src/function/aggregate/distributive/minmax.cpp"], "evidence": "context:114"}, {"target": "duckdb_optimizer_relation_statistics", "caption": "Relation statistics", "members": ["src/optimizer/relation_statistics/relation_statistics_extractor.cpp", "src/optimizer/relation_statistics/relation_statistics_helper.cpp", "src/optimizer/relation_statistics/relation_statistics_operator.cpp"], "evidence": "context:157"}, {"target": "duckdb_optimizer_join_order", "caption": "Join ordering", "members": ["src/optimizer/join_order/relation_manager.cpp"], "evidence": "context:156"}, {"target": "duckdb_optimizer", "caption": "Optimizer", "members": ["src/optimizer/aggregate_rewrite.cpp"], "evidence": "context:147"}, {"target": "duckdb_operator_set", "caption": "Set execution", "members": ["src/execution/operator/set/physical_recursive_cte_keyed.cpp"], "evidence": "context:105"}, {"target": "test_optimizer", "caption": "Optimizer tests", "members": ["test/optimizer/relation_statistics.cpp"], "evidence": "context:221"}], "people": [{"file_path": "src/function/aggregate/distributive/first_last_any.cpp", "author_name": "Denis Hirn", "base_region_lines": 38, "github_login": "kryonix", "evidence_url": "https://github.com/duckdb/duckdb/commit/3a5d0c26c254ee4521b504e03a0f1315b821fcdc", "history_commits": 7, "role": "historical_expertise_candidate", "formal_owner_verified": false}, {"file_path": "src/function/aggregate/distributive/first_last_any.cpp", "author_name": "Mytherin", "base_region_lines": 2, "github_login": null, "evidence_url": null, "history_commits": 13, "role": "historical_expertise_candidate", "formal_owner_verified": false}, {"file_path": "src/function/aggregate/distributive/first_last_any.cpp", "author_name": "Pedro Holanda", "base_region_lines": 1, "github_login": null, "evidence_url": null, "history_commits": 1, "role": "historical_expertise_candidate", "formal_owner_verified": false}, {"file_path": "src/function/aggregate/distributive/first_last_any.cpp", "author_name": "Maxxen", "base_region_lines": 1, "github_login": null, "evidence_url": null, "history_commits": 1, "role": "historical_expertise_candidate", "formal_owner_verified": false}, {"file_path": "src/optimizer/relation_statistics/relation_statistics_helper.cpp", "author_name": "Laurens Kuiper", "base_region_lines": 108, "github_login": "lnkuiper", "evidence_url": "https://github.com/duckdb/duckdb/commit/255d752c99831484aa675d496d3934767c5ac0cb", "history_commits": 22, "role": "historical_expertise_candidate", "formal_owner_verified": false}, {"file_path": "src/optimizer/relation_statistics/relation_statistics_helper.cpp", "author_name": "xproblematique", "base_region_lines": 5, "github_login": null, "evidence_url": null, "history_commits": 1, "role": "historical_expertise_candidate", "formal_owner_verified": false}, {"file_path": "src/optimizer/relation_statistics/relation_statistics_helper.cpp", "author_name": "Robinho-MR", "base_region_lines": 3, "github_login": null, "evidence_url": null, "history_commits": 3, "role": "historical_expertise_candidate", "formal_owner_verified": false}], "input_refs": ["2026-11-18-pr-repository-context-map/relationships.json", "2026-11-18-pr-repository-context-map/native-incoming-paths.json", "2026-11-17-review-extensions-poc/stakeholder-candidates.json"]};
const clientSource="\nimport { h, hydrate } from 'https://esm.sh/preact@10.24.3';\nimport { useState } from 'https://esm.sh/preact@10.24.3/hooks';\nimport { View, initial } from './view.ts';\nconst d={\"head\": \"d29b26bb92ac940a0807be0f43cc385b31623839\", \"base\": \"629e395a854d72dcb1f414e1a0451c68ac0e9f1f\", \"files\": [{\"id\": \"stats\", \"name\": \"relation_statistics_helper.cpp\", \"path\": \"src/optimizer/relation_statistics/relation_statistics_helper.cpp\", \"status\": \"Modified\", \"symbols\": [\"ExtractAggregationStatsInternal\", \"HasUniqueGroups\", \"HasBaseColumnSource\", \"GetValuePreservingColumn\"], \"diff\": \"diff --git a/src/optimizer/relation_statistics/relation_statistics_helper.cpp b/src/optimizer/relation_statistics/relation_statistics_helper.cpp\\nindex 99c3862b89c..6eac019d1e3 100644\\n--- a/src/optimizer/relation_statistics/relation_statistics_helper.cpp\\n+++ b/src/optimizer/relation_statistics/relation_statistics_helper.cpp\\n@@ -2,6 +2,8 @@\\n #include \\\"duckdb/optimizer/relation_statistics/relation_statistics_extractor.hpp\\\"\\n \\n #include \\\"duckdb/common/types/value_map.hpp\\\"\\n+#include \\\"duckdb/optimizer/apply_functional_dependencies.hpp\\\"\\n+#include \\\"duckdb/planner/expression/bound_cast_expression.hpp\\\"\\n #include \\\"duckdb/planner/expression/bound_comparison_expression.hpp\\\"\\n #include \\\"duckdb/planner/expression/bound_constant_expression.hpp\\\"\\n #include \\\"duckdb/planner/expression/list.hpp\\\"\\n@@ -450,6 +452,109 @@ idx_t RelationStatisticsHelper::EstimateGroupCardinality(LogicalOperator &input,\\n \\treturn estimate;\\n }\\n \\n+static optional<ColumnBinding> GetValuePreservingColumn(const Expression &expression) {\\n+\\tif (expression.GetExpressionClass() == ExpressionClass::BOUND_COLUMN_REF) {\\n+\\t\\treturn expression.Cast<BoundColumnRefExpression>().Binding();\\n+\\t}\\n+\\tif (!BoundCastExpression::IsCast(expression)) {\\n+\\t\\treturn {};\\n+\\t}\\n+\\tauto &cast = expression.Cast<BoundFunctionExpression>();\\n+\\tif (!BoundCastExpression::IsDefaultCast(cast) || BoundCastExpression::IsTryCast(cast)) {\\n+\\t\\treturn {};\\n+\\t}\\n+\\tauto source_type = BoundCastExpression::SourceType(cast);\\n+\\tauto &target_type = BoundCastExpression::TargetType(cast);\\n+\\t// Successful strict integral casts do not merge distinct input values.\\n+\\tif (source_type != target_type && !(source_type.IsIntegral() && target_type.IsIntegral()) &&\\n+\\t    !BoundCastExpression::CastIsInvertibleForEquality(source_type, target_type)) {\\n+\\t\\treturn {};\\n+\\t}\\n+\\treturn GetValuePreservingColumn(BoundCastExpression::Child(cast));\\n+}\\n+\\n+static bool HasBaseColumnSource(LogicalOperator &op, ColumnBinding binding) {\\n+\\tswitch (op.type) {\\n+\\tcase LogicalOperatorType::LOGICAL_GET: {\\n+\\t\\tif (!op.children.empty()) {\\n+\\t\\t\\treturn false;\\n+\\t\\t}\\n+\\t\\tfor (auto &column : op.GetColumnBindings()) {\\n+\\t\\t\\tif (column == binding) {\\n+\\t\\t\\t\\treturn true;\\n+\\t\\t\\t}\\n+\\t\\t}\\n+\\t\\treturn false;\\n+\\t}\\n+\\tcase LogicalOperatorType::LOGICAL_PROJECTION: {\\n+\\t\\tif (op.children.size() != 1) {\\n+\\t\\t\\treturn false;\\n+\\t\\t}\\n+\\t\\tauto bindings = op.GetColumnBindings();\\n+\\t\\tfor (idx_t i = 0; i < bindings.size(); i++) {\\n+\\t\\t\\tif (bindings[i] != binding) {\\n+\\t\\t\\t\\tcontinue;\\n+\\t\\t\\t}\\n+\\t\\t\\tauto child_binding = GetValuePreservingColumn(*op.expressions[i]);\\n+\\t\\t\\treturn child_binding && HasBaseColumnSource(*op.children[0], *child_binding);\\n+\\t\\t}\\n+\\t\\treturn false;\\n+\\t}\\n+\\tcase LogicalOperatorType::LOGICAL_FILTER:\\n+\\tcase LogicalOperatorType::LOGICAL_ORDER_BY:\\n+\\tcase LogicalOperatorType::LOGICAL_LIMIT:\\n+\\tcase LogicalOperatorType::LOGICAL_TOP_N:\\n+\\t\\treturn op.children.size() == 1 && HasBaseColumnSource(*op.children[0], binding);\\n+\\tdefault:\\n+\\t\\treturn false;\\n+\\t}\\n+}\\n+\\n+static bool HasUniqueGroups(LogicalOperator &op, const column_binding_set_t &groups) {\\n+\\tif (groups.empty()) {\\n+\\t\\treturn false;\\n+\\t}\\n+\\tswitch (op.type) {\\n+\\tcase LogicalOperatorType::LOGICAL_PROJECTION: {\\n+\\t\\tif (op.children.size() != 1) {\\n+\\t\\t\\treturn false;\\n+\\t\\t}\\n+\\t\\tcolumn_binding_set_t child_groups;\\n+\\t\\tauto bindings = op.GetColumnBindings();\\n+\\t\\tfor (idx_t i = 0; i < bindings.size(); i++) {\\n+\\t\\t\\tif (groups.count(bindings[i]) == 0) {\\n+\\t\\t\\t\\tcontinue;\\n+\\t\\t\\t}\\n+\\t\\t\\tauto binding = GetValuePreservingColumn(*op.expressions[i]);\\n+\\t\\t\\tif (binding) {\\n+\\t\\t\\t\\tchild_groups.insert(*binding);\\n+\\t\\t\\t}\\n+\\t\\t}\\n+\\t\\treturn HasUniqueGroups(*op.children[0], child_groups);\\n+\\t}\\n+\\tcase LogicalOperatorType::LOGICAL_FILTER:\\n+\\tcase LogicalOperatorType::LOGICAL_ORDER_BY:\\n+\\tcase LogicalOperatorType::LOGICAL_LIMIT:\\n+\\tcase LogicalOperatorType::LOGICAL_TOP_N:\\n+\\t\\treturn op.children.size() == 1 && HasUniqueGroups(*op.children[0], groups);\\n+\\tdefault:\\n+\\t\\tbreak;\\n+\\t}\\n+\\tfor (auto &key : ApplyFunctionalDependencies::GetUniqueColumnCombinations(op)) {\\n+\\t\\tbool covered = !key.empty();\\n+\\t\\tfor (auto &binding : key) {\\n+\\t\\t\\tif (groups.count(binding) == 0) {\\n+\\t\\t\\t\\tcovered = false;\\n+\\t\\t\\t\\tbreak;\\n+\\t\\t\\t}\\n+\\t\\t}\\n+\\t\\tif (covered) {\\n+\\t\\t\\treturn true;\\n+\\t\\t}\\n+\\t}\\n+\\treturn false;\\n+}\\n+\\n static optional<RelationStats> ExtractAggregationStatsInternal(LogicalAggregate &aggregate,\\n                                                                const RelationStats &child_stats,\\n                                                                RelationStatsExtractor &extractor,\\n@@ -510,6 +615,20 @@ static optional<RelationStats> ExtractAggregationStatsInternal(LogicalAggregate\\n \\t\\t        ? 1\\n \\t\\t        : RelationStatisticsHelper::EstimateDistinctCardinality(cardinality_counts, child_stats.cardinality);\\n \\t}\\n+\\tbool has_unique_groups = false;\\n+\\tif (preserves_group_evidence && !aggregate.groups.empty() && aggregate.children.size() == 1) {\\n+\\t\\tcolumn_binding_set_t groups;\\n+\\t\\tfor (auto &group : aggregate.groups) {\\n+\\t\\t\\tauto binding = GetValuePreservingColumn(*group);\\n+\\t\\t\\tif (binding) {\\n+\\t\\t\\t\\tgroups.insert(*binding);\\n+\\t\\t\\t}\\n+\\t\\t}\\n+\\t\\thas_unique_groups = HasUniqueGroups(*aggregate.children[0], groups);\\n+\\t\\tif (has_unique_groups) {\\n+\\t\\t\\tresult.cardinality = child_stats.cardinality;\\n+\\t\\t}\\n+\\t}\\n \\tfor (idx_t group_idx = 0; group_idx < aggregate.groups.size(); group_idx++) {\\n \\t\\tauto &group = *aggregate.groups[group_idx];\\n \\t\\tDistinctCount distinct_count(result.cardinality, DistinctCountSource::CARDINALITY);\\n@@ -536,9 +655,24 @@ static optional<RelationStats> ExtractAggregationStatsInternal(LogicalAggregate\\n \\t\\tresult.columns.emplace_back(bindings[result.columns.size()], distinct_count, current_domain, Identifier(),\\n \\t\\t                            semi_anti_join_domain_evidence);\\n \\t}\\n-\\tfor (idx_t aggregate_idx = 0; aggregate_idx < aggregate.expressions.size(); aggregate_idx++) {\\n-\\t\\tresult.columns.emplace_back(bindings[result.columns.size()],\\n-\\t\\t                            DistinctCount(result.cardinality, DistinctCountSource::CARDINALITY), Identifier());\\n+\\tfor (auto &expression : aggregate.expressions) {\\n+\\t\\tDistinctCount total_domain(result.cardinality, DistinctCountSource::CARDINALITY);\\n+\\t\\tDistinctCount current_domain = total_domain;\\n+\\t\\tif (has_unique_groups && expression->GetExpressionClass() == ExpressionClass::BOUND_AGGREGATE) {\\n+\\t\\t\\tauto &bound_aggregate = expression->Cast<BoundAggregateExpression>();\\n+\\t\\t\\tauto &agg_children = bound_aggregate.GetChildren();\\n+\\t\\t\\tif (bound_aggregate.Function().HasSingleValueIdentity() && !bound_aggregate.GetFilter() &&\\n+\\t\\t\\t    !agg_children.empty() && expression->GetReturnType() == agg_children[0]->GetReturnType() &&\\n+\\t\\t\\t    agg_children[0]->GetExpressionClass() == ExpressionClass::BOUND_COLUMN_REF) {\\n+\\t\\t\\t\\tauto binding = agg_children[0]->Cast<BoundColumnRefExpression>().Binding();\\n+\\t\\t\\t\\tauto child_column = child_columns.Find(binding);\\n+\\t\\t\\t\\tif (child_column && HasBaseColumnSource(*aggregate.children[0], binding)) {\\n+\\t\\t\\t\\t\\ttotal_domain = child_column->total_domain;\\n+\\t\\t\\t\\t\\tcurrent_domain = child_column->current_domain;\\n+\\t\\t\\t\\t}\\n+\\t\\t\\t}\\n+\\t\\t}\\n+\\t\\tresult.columns.emplace_back(bindings[result.columns.size()], total_domain, current_domain, Identifier());\\n \\t}\\n \\tfor (idx_t grouping_idx = 0; grouping_idx < aggregate.grouping_functions.size(); grouping_idx++) {\\n \\t\\tauto grouping_count = MinValue<idx_t>(result.cardinality, MaxValue<idx_t>(aggregate.grouping_sets.size(), 1));\\n\"}, {\"id\": \"rewrite\", \"name\": \"first_last_any.cpp\", \"path\": \"src/function/aggregate/distributive/first_last_any.cpp\", \"status\": \"Modified\", \"symbols\": [\"RewriteOrderedFirst\"], \"diff\": \"diff --git a/src/function/aggregate/distributive/first_last_any.cpp b/src/function/aggregate/distributive/first_last_any.cpp\\nindex f36a4e84705..0ba568cb72b 100644\\n--- a/src/function/aggregate/distributive/first_last_any.cpp\\n+++ b/src/function/aggregate/distributive/first_last_any.cpp\\n@@ -53,6 +53,8 @@ unique_ptr<Expression> RewriteOrderedFirst(AggregateRewriteInput &input) {\\n \\tconst auto &function = entry.functions.GetFunctionByArguments(input.context, child_types);\\n \\tauto result = binder.BindAggregateFunction(function, std::move(children), std::move(aggregate->GetFilterMutable()),\\n \\t                                           aggregate->GetAggregateType());\\n+\\t// The encoded sort key is non-NULL, so a singleton still returns its input value.\\n+\\tresult->FunctionMutable().SetSingleValueIdentity(aggregate->Function().HasSingleValueIdentity());\\n \\treturn unique_ptr_cast<BoundAggregateExpression, Expression>(std::move(result));\\n }\\n \\n\"}, {\"id\": \"shared\", \"name\": \"stats_dependent_tests.json\", \"path\": \"test/configs/stats_dependent_tests.json\", \"status\": \"Modified\", \"symbols\": [], \"diff\": \"diff --git a/test/configs/stats_dependent_tests.json b/test/configs/stats_dependent_tests.json\\nindex 0db774e9b95..7457b6bce55 100644\\n--- a/test/configs/stats_dependent_tests.json\\n+++ b/test/configs/stats_dependent_tests.json\\n@@ -37,6 +37,7 @@\\n         \\\"test/sql/storage/list_length_stats_checkpoint.test\\\",\\n         \\\"test/sql/storage/list_length_stats_nested_optimistic.test\\\",\\n         \\\"test/sql/optimizer/statistics/list_length_stats_propagation.test\\\",\\n+        \\\"test/sql/optimizer/statistics/distinct_on_stats_propagation.test\\\",\\n         \\\"test/sql/storage/list_length_stats_old_version.test\\\",\\n         \\\"test/sql/types/nested/array/array_statistics.test\\\",\\n         \\\"test/sql/vacuum/test_analyze.test\\\",\\n\"}, {\"id\": \"export\", \"name\": \"verify_aggregate_state_export.json\", \"path\": \"test/configs/verify_aggregate_state_export.json\", \"status\": \"Modified\", \"symbols\": [], \"diff\": \"diff --git a/test/configs/verify_aggregate_state_export.json b/test/configs/verify_aggregate_state_export.json\\nindex 7c7c58f3252..e59f36a3806 100644\\n--- a/test/configs/verify_aggregate_state_export.json\\n+++ b/test/configs/verify_aggregate_state_export.json\\n@@ -47,6 +47,7 @@\\n         \\\"test/sql/copy/parquet/parquet_stats.test\\\",\\n         \\\"test/sql/copy/parquet/read_single_parquet_file.test\\\",\\n         \\\"test/sql/optimizer/aggregate_stats_propagation.test\\\",\\n+        \\\"test/sql/optimizer/statistics/distinct_on_stats_propagation.test\\\",\\n         \\\"test/sql/optimizer/scalar_aggregate_merge_files_plan.test\\\",\\n         \\\"test/sql/optimizer/scalar_aggregate_merge_plan.test\\\",\\n         \\\"test/sql/optimizer/scalar_aggregate_merge_tpcds_plan.test\\\",\\n\"}, {\"id\": \"test\", \"name\": \"distinct_on_stats_propagation.test\", \"path\": \"test/sql/optimizer/statistics/distinct_on_stats_propagation.test\", \"status\": \"Added\", \"symbols\": [], \"diff\": \"diff --git a/test/sql/optimizer/statistics/distinct_on_stats_propagation.test b/test/sql/optimizer/statistics/distinct_on_stats_propagation.test\\nnew file mode 100644\\nindex 00000000000..97430e2de99\\n--- /dev/null\\n+++ b/test/sql/optimizer/statistics/distinct_on_stats_propagation.test\\n@@ -0,0 +1,462 @@\\n+# name: test/sql/optimizer/statistics/distinct_on_stats_propagation.test\\n+# description: Aggregate domains require exact grouping keys and single-value identity\\n+# group: [statistics]\\n+\\n+# Exact plan estimates depend on sampled distinct counts.\\n+require vector_size 2048\\n+\\n+# HLL can estimate one group per input row even when grouping removes a row.\\n+statement ok\\n+CREATE TABLE false_singletons AS\\n+SELECT CASE WHEN i = 32 THEN 0 ELSE i END * 2018::BIGINT AS g,\\n+       CASE WHEN i = 32 THEN 9 ELSE i % 2 END::INT AS x FROM range(33) t(i);\\n+\\n+statement ok\\n+CREATE TABLE false_singletons_right AS SELECT * FROM false_singletons;\\n+\\n+statement ok\\n+ANALYZE;\\n+\\n+statement ok\\n+SET explain_output = 'optimized_only';\\n+\\n+query II\\n+EXPLAIN SELECT count(*)\\n+FROM (SELECT first(x ORDER BY x) AS k FROM false_singletons GROUP BY g) l\\n+JOIN (SELECT first(x ORDER BY x) AS k FROM false_singletons_right GROUP BY g) r ON l.k = r.k;\\n+----\\n+logical_opt\\t<REGEX>:(?s).*Comparison Join[^~]*~33 rows.*\\n+\\n+query I\\n+SELECT count(*)\\n+FROM (SELECT first(x ORDER BY x) AS k FROM false_singletons GROUP BY g) l\\n+JOIN (SELECT first(x ORDER BY x) AS k FROM false_singletons_right GROUP BY g) r ON l.k = r.k;\\n+----\\n+512\\n+\\n+# Positive fixture with a declared key. Unconstrained inputs are checked separately.\\n+statement ok\\n+CREATE TABLE nodes(id INTEGER PRIMARY KEY, k INTEGER);\\n+\\n+statement ok\\n+INSERT INTO nodes SELECT i::INT, i::INT FROM range(100000) t(i);\\n+\\n+statement ok\\n+CREATE TABLE edges AS SELECT i::INT AS eid, (i % 10000)::INT AS k FROM range(100000) t(i);\\n+\\n+statement ok\\n+ANALYZE;\\n+\\n+statement ok\\n+SET explain_output = 'optimized_only';\\n+\\n+# DISTINCT ON lowers to Aggregate(GROUP BY id, first(k)): the first() output carries k's\\n+# domain (100000 distinct) instead of an aggregate-cardinality guess, so the join\\n+# denominator is max(100000, 10000) and the estimate matches the DISTINCT k control\\n+query II\\n+EXPLAIN SELECT count(*) FROM (SELECT DISTINCT ON (id) k FROM nodes) s JOIN edges e ON s.k = e.k;\\n+----\\n+logical_opt\\t<REGEX>:(?s).*Comparison Join[^~]*~100,000 rows.*\\n+\\n+# a handwritten first() gets the same domain forwarding\\n+query II\\n+EXPLAIN SELECT count(*) FROM (SELECT first(k) AS k FROM nodes GROUP BY id) s JOIN edges e ON s.k = e.k;\\n+----\\n+logical_opt\\t<REGEX>:(?s).*Comparison Join[^~]*~100,000 rows.*\\n+\\n+# DISTINCT ON with an ORDER BY inside the subquery keeps the forwarded domain\\n+query II\\n+EXPLAIN SELECT count(*) FROM (SELECT DISTINCT ON (id) k FROM nodes ORDER BY id, k) s JOIN edges e ON s.k = e.k;\\n+----\\n+logical_opt\\t<REGEX>:(?s).*Comparison Join[^~]*~100,000 rows.*\\n+\\n+# the DISTINCT k control was already exact and stays exact\\n+query II\\n+EXPLAIN SELECT count(*) FROM (SELECT DISTINCT k FROM nodes) s JOIN edges e ON s.k = e.k;\\n+----\\n+logical_opt\\t<REGEX>:(?s).*Comparison Join[^~]*~100,000 rows.*\\n+\\n+# the statistics change does not change results\\n+query I\\n+SELECT count(*) FROM (SELECT DISTINCT ON (id) k FROM nodes) s JOIN edges e ON s.k = e.k;\\n+----\\n+100000\\n+\\n+query I\\n+SELECT count(*) FROM (SELECT DISTINCT ON (id) k FROM nodes ORDER BY id, k) s JOIN edges e ON s.k = e.k;\\n+----\\n+100000\\n+\\n+# List-producing overloads create combinations outside their first child's domain.\\n+statement ok\\n+CREATE TABLE bit_pairs_left AS\\n+SELECT g::INT AS g, p::INT AS p, ((g >> p) & 1)::INT AS x\\n+FROM range(4) g_values(g), range(2) p_values(p);\\n+\\n+statement ok\\n+CREATE TABLE bit_pairs_right AS SELECT * FROM bit_pairs_left;\\n+\\n+statement ok\\n+ANALYZE;\\n+\\n+query II\\n+EXPLAIN SELECT count(*)\\n+FROM (SELECT arg_min(x, p, 2) AS k FROM bit_pairs_left GROUP BY g) l\\n+JOIN (SELECT arg_min(x, p, 2) AS k FROM bit_pairs_right GROUP BY g) r ON l.k = r.k;\\n+----\\n+logical_opt\\t<REGEX>:(?s).*Comparison Join[^~]*~4 rows.*\\n+\\n+query II\\n+EXPLAIN SELECT count(*)\\n+FROM (SELECT arg_max(x, p, 2) AS k FROM bit_pairs_left GROUP BY g) l\\n+JOIN (SELECT arg_max(x, p, 2) AS k FROM bit_pairs_right GROUP BY g) r ON l.k = r.k;\\n+----\\n+logical_opt\\t<REGEX>:(?s).*Comparison Join[^~]*~4 rows.*\\n+\\n+# The return type also differs when the input is already a list.\\n+query II\\n+EXPLAIN SELECT count(*)\\n+FROM (SELECT arg_min([x], p, 2) AS k FROM bit_pairs_left GROUP BY g) l\\n+JOIN (SELECT arg_min([x], p, 2) AS k FROM bit_pairs_right GROUP BY g) r ON l.k = r.k;\\n+----\\n+logical_opt\\t<REGEX>:(?s).*Comparison Join[^~]*~4 rows.*\\n+\\n+# Reducing groups retain the aggregate cardinality fallback.\\n+query II\\n+EXPLAIN SELECT count(*)\\n+FROM (SELECT first(x ORDER BY p) AS k FROM bit_pairs_left GROUP BY g) l\\n+JOIN (SELECT first(x ORDER BY p) AS k FROM bit_pairs_right GROUP BY g) r ON l.k = r.k;\\n+----\\n+logical_opt\\t<REGEX>:(?s).*Comparison Join[^~]*~4 rows.*\\n+\\n+# List-producing overloads remain excluded even with certified singleton groups.\\n+statement ok\\n+CREATE TABLE unique_list_values(id INTEGER PRIMARY KEY, x INTEGER, p INTEGER);\\n+\\n+statement ok\\n+INSERT INTO unique_list_values SELECT i::INT, (i % 2)::INT, NULL FROM range(8) t(i);\\n+\\n+statement ok\\n+CREATE TABLE unique_list_values_right(id INTEGER PRIMARY KEY, x INTEGER, p INTEGER);\\n+\\n+statement ok\\n+INSERT INTO unique_list_values_right SELECT * FROM unique_list_values;\\n+\\n+statement ok\\n+ANALYZE;\\n+\\n+# The scalar control forwards the domain for the same unique groups.\\n+query II\\n+EXPLAIN SELECT count(*)\\n+FROM (SELECT first(x) AS k FROM unique_list_values GROUP BY id) l\\n+JOIN (SELECT first(x) AS k FROM unique_list_values_right GROUP BY id) r ON l.k = r.k;\\n+----\\n+logical_opt\\t<REGEX>:(?s).*Comparison Join[^~]*~32 rows.*\\n+\\n+# An aggregate filter can remove values even when every group has one row.\\n+query II\\n+EXPLAIN SELECT count(*)\\n+FROM (SELECT first(x) FILTER (WHERE id % 2 = 0) AS k FROM unique_list_values GROUP BY id) l\\n+JOIN (SELECT first(x) FILTER (WHERE id % 2 = 0) AS k FROM unique_list_values_right GROUP BY id) r ON l.k = r.k;\\n+----\\n+logical_opt\\t<REGEX>:(?s).*Comparison Join[^~]*~8 rows.*\\n+\\n+query I\\n+SELECT count(*)\\n+FROM (SELECT first(x) FILTER (WHERE id % 2 = 0) AS k FROM unique_list_values GROUP BY id) l\\n+JOIN (SELECT first(x) FILTER (WHERE id % 2 = 0) AS k FROM unique_list_values_right GROUP BY id) r ON l.k = r.k;\\n+----\\n+16\\n+\\n+query II\\n+EXPLAIN SELECT count(*)\\n+FROM (SELECT arg_min(x, id, 2) AS k FROM unique_list_values GROUP BY id) l\\n+JOIN (SELECT arg_min(x, id, 2) AS k FROM unique_list_values_right GROUP BY id) r ON l.k = r.k;\\n+----\\n+logical_opt\\t<REGEX>:(?s).*Comparison Join[^~]*~8 rows.*\\n+\\n+query II\\n+EXPLAIN SELECT count(*)\\n+FROM (SELECT arg_max(x, id, 2) AS k FROM unique_list_values GROUP BY id) l\\n+JOIN (SELECT arg_max(x, id, 2) AS k FROM unique_list_values_right GROUP BY id) r ON l.k = r.k;\\n+----\\n+logical_opt\\t<REGEX>:(?s).*Comparison Join[^~]*~8 rows.*\\n+\\n+query I\\n+SELECT count(*)\\n+FROM (SELECT arg_min(x, id, 2) AS k FROM unique_list_values GROUP BY id) l\\n+JOIN (SELECT arg_min(x, id, 2) AS k FROM unique_list_values_right GROUP BY id) r ON l.k = r.k;\\n+----\\n+32\\n+\\n+query I\\n+SELECT count(*)\\n+FROM (SELECT arg_max(x, id, 2) AS k FROM unique_list_values GROUP BY id) l\\n+JOIN (SELECT arg_max(x, id, 2) AS k FROM unique_list_values_right GROUP BY id) r ON l.k = r.k;\\n+----\\n+32\\n+\\n+query I\\n+SELECT count(*)\\n+FROM (SELECT arg_min(x, p, 2) AS k FROM bit_pairs_left GROUP BY g) l\\n+JOIN (SELECT arg_min(x, p, 2) AS k FROM bit_pairs_right GROUP BY g) r ON l.k = r.k;\\n+----\\n+4\\n+\\n+query I\\n+SELECT count(*)\\n+FROM (SELECT arg_max(x, p, 2) AS k FROM bit_pairs_left GROUP BY g) l\\n+JOIN (SELECT arg_max(x, p, 2) AS k FROM bit_pairs_right GROUP BY g) r ON l.k = r.k;\\n+----\\n+4\\n+\\n+query I\\n+SELECT count(*)\\n+FROM (SELECT first(x ORDER BY p) AS k FROM bit_pairs_left GROUP BY g) l\\n+JOIN (SELECT first(x ORDER BY p) AS k FROM bit_pairs_right GROUP BY g) r ON l.k = r.k;\\n+----\\n+8\\n+\\n+# Aliases retain the bound single-value identity property.\\n+query II\\n+EXPLAIN SELECT count(*) FROM (SELECT arbitrary(k) AS k FROM nodes GROUP BY id) s JOIN edges e ON s.k = e.k;\\n+----\\n+logical_opt\\t<REGEX>:(?s).*Comparison Join[^~]*~100,000 rows.*\\n+\\n+query II\\n+EXPLAIN SELECT count(*) FROM (SELECT last(k) AS k FROM nodes GROUP BY id) s JOIN edges e ON s.k = e.k;\\n+----\\n+logical_opt\\t<REGEX>:(?s).*Comparison Join[^~]*~100,000 rows.*\\n+\\n+query II\\n+EXPLAIN SELECT count(*) FROM (SELECT any_value(k) AS k FROM nodes GROUP BY id) s JOIN edges e ON s.k = e.k;\\n+----\\n+logical_opt\\t<REGEX>:(?s).*Comparison Join[^~]*~100,000 rows.*\\n+\\n+# A unique grouping key does not stop arg_min from rejecting a NULL selector.\\n+query II\\n+EXPLAIN SELECT count(*)\\n+FROM (SELECT arg_min(x, p) AS k FROM unique_list_values GROUP BY id) l\\n+JOIN (SELECT arg_min(x, p) AS k FROM unique_list_values_right GROUP BY id) r ON l.k = r.k;\\n+----\\n+logical_opt\\t<REGEX>:(?s).*Comparison Join[^~]*~8 rows.*\\n+\\n+query II\\n+EXPLAIN SELECT count(*)\\n+FROM (SELECT min_by(x, p) AS k FROM unique_list_values GROUP BY id) l\\n+JOIN (SELECT min_by(x, p) AS k FROM unique_list_values_right GROUP BY id) r ON l.k = r.k;\\n+----\\n+logical_opt\\t<REGEX>:(?s).*Comparison Join[^~]*~8 rows.*\\n+\\n+query II\\n+EXPLAIN SELECT count(*)\\n+FROM (SELECT max_by(x, p) AS k FROM unique_list_values GROUP BY id) l\\n+JOIN (SELECT max_by(x, p) AS k FROM unique_list_values_right GROUP BY id) r ON l.k = r.k;\\n+----\\n+logical_opt\\t<REGEX>:(?s).*Comparison Join[^~]*~8 rows.*\\n+\\n+query I\\n+SELECT count(*)\\n+FROM (SELECT arg_min(x, p) AS k FROM unique_list_values GROUP BY id) l\\n+JOIN (SELECT arg_min(x, p) AS k FROM unique_list_values_right GROUP BY id) r ON l.k = r.k;\\n+----\\n+0\\n+\\n+# An identical CTAS input has no certified key and retains the fallback.\\n+statement ok\\n+CREATE TABLE unconstrained_values AS SELECT * FROM unique_list_values;\\n+\\n+statement ok\\n+CREATE TABLE unconstrained_values_right AS SELECT * FROM unique_list_values;\\n+\\n+statement ok\\n+ANALYZE;\\n+\\n+query II\\n+EXPLAIN SELECT count(*)\\n+FROM (SELECT first(x) AS k FROM unconstrained_values GROUP BY id) l\\n+JOIN (SELECT first(x) AS k FROM unconstrained_values_right GROUP BY id) r ON l.k = r.k;\\n+----\\n+logical_opt\\t<REGEX>:(?s).*Comparison Join[^~]*~8 rows.*\\n+\\n+query I\\n+SELECT count(*)\\n+FROM (SELECT first(x) AS k FROM unconstrained_values GROUP BY id) l\\n+JOIN (SELECT first(x) AS k FROM unconstrained_values_right GROUP BY id) r ON l.k = r.k;\\n+----\\n+32\\n+\\n+# Non-null immediate UNIQUE keys provide the same proof as a primary key.\\n+statement ok\\n+CREATE TABLE nonnull_unique(id INTEGER UNIQUE NOT NULL, x INTEGER);\\n+\\n+statement ok\\n+INSERT INTO nonnull_unique SELECT id, x FROM unique_list_values;\\n+\\n+statement ok\\n+CREATE TABLE nullable_unique(id INTEGER UNIQUE, x INTEGER);\\n+\\n+statement ok\\n+INSERT INTO nullable_unique SELECT CASE WHEN id = 7 THEN NULL ELSE id END, x FROM unique_list_values;\\n+\\n+statement ok\\n+CREATE TABLE composite_unique(a INTEGER NOT NULL, b INTEGER NOT NULL, x INTEGER, UNIQUE(a, b));\\n+\\n+statement ok\\n+INSERT INTO composite_unique SELECT id % 2, id // 2, x FROM unique_list_values;\\n+\\n+statement ok\\n+ANALYZE;\\n+\\n+query II\\n+EXPLAIN (FORMAT JSON) SELECT count(*)\\n+FROM (SELECT first(x) AS k FROM nonnull_unique GROUP BY id) l\\n+JOIN (SELECT first(x) AS k FROM nonnull_unique GROUP BY id) r ON l.k = r.k;\\n+----\\n+logical_opt\\t<REGEX>:(?s).*\\\"Join Type\\\": \\\"INNER\\\",\\\\s*\\\"Conditions\\\": \\\"\\\\(k = k\\\\)\\\",\\\\s*\\\"Estimated Cardinality\\\": \\\"32\\\".*\\n+\\n+# Nullable keys do not certify singleton groups.\\n+query II\\n+EXPLAIN (FORMAT JSON) SELECT count(*)\\n+FROM (SELECT first(x) AS k FROM nullable_unique GROUP BY id) l\\n+JOIN (SELECT first(x) AS k FROM nullable_unique GROUP BY id) r ON l.k = r.k;\\n+----\\n+logical_opt\\t<REGEX>:(?s).*\\\"Join Type\\\": \\\"INNER\\\",\\\\s*\\\"Conditions\\\": \\\"\\\\(k = k\\\\)\\\",\\\\s*\\\"Estimated Cardinality\\\": \\\"7\\\".*\\n+\\n+# A composite key must be completely covered by the grouping expressions.\\n+query II\\n+EXPLAIN (FORMAT JSON) SELECT count(*)\\n+FROM (SELECT first(x) AS k FROM composite_unique GROUP BY a, b) l\\n+JOIN (SELECT first(x) AS k FROM composite_unique GROUP BY a, b) r ON l.k = r.k;\\n+----\\n+logical_opt\\t<REGEX>:(?s).*\\\"Join Type\\\": \\\"INNER\\\",\\\\s*\\\"Conditions\\\": \\\"\\\\(k = k\\\\)\\\",\\\\s*\\\"Estimated Cardinality\\\": \\\"32\\\".*\\n+\\n+query II\\n+EXPLAIN (FORMAT JSON) SELECT count(*)\\n+FROM (SELECT first(x) AS k FROM composite_unique GROUP BY a) l\\n+JOIN (SELECT first(x) AS k FROM composite_unique GROUP BY a) r ON l.k = r.k;\\n+----\\n+logical_opt\\t<REGEX>:(?s).*\\\"Join Type\\\": \\\"INNER\\\",\\\\s*\\\"Conditions\\\": \\\"\\\\(k = k\\\\)\\\",\\\\s*\\\"Estimated Cardinality\\\": \\\"2\\\".*\\n+\\n+# Casting a key to BOOLEAN can merge distinct key values.\\n+query II\\n+EXPLAIN SELECT count(*)\\n+FROM (SELECT first(x) AS k FROM unique_list_values GROUP BY CAST(id AS BOOLEAN)) l\\n+JOIN (SELECT first(x) AS k FROM unique_list_values_right GROUP BY CAST(id AS BOOLEAN)) r ON l.k = r.k;\\n+----\\n+logical_opt\\t<REGEX>:(?s).*Comparison Join[^~]*~4 rows.*\\n+\\n+# TRY_CAST can turn distinct key values into NULL and cannot certify uniqueness.\\n+query II\\n+EXPLAIN SELECT count(*)\\n+FROM (SELECT first(x) AS k FROM unique_list_values GROUP BY TRY_CAST(id AS TINYINT)) l\\n+JOIN (SELECT first(x) AS k FROM unique_list_values_right GROUP BY TRY_CAST(id AS TINYINT)) r ON l.k = r.k;\\n+----\\n+logical_opt\\t<REGEX>:(?s).*Comparison Join[^~]*~4 rows.*\\n+\\n+# Joins can multiply a key's rows and must invalidate the singleton proof.\\n+query II\\n+EXPLAIN SELECT count(*)\\n+FROM (SELECT first(v.x) AS k FROM unique_list_values v CROSS JOIN range(2) t(i) GROUP BY v.id) l\\n+JOIN (SELECT first(v.x) AS k FROM unique_list_values_right v CROSS JOIN range(2) t(i) GROUP BY v.id) r ON l.k = r.k;\\n+----\\n+logical_opt\\t<REGEX>:(?s).*Comparison Join[^~]*~8 rows.*\\n+\\n+query I\\n+SELECT count(*)\\n+FROM (SELECT first(v.x) AS k FROM unique_list_values v CROSS JOIN range(2) t(i) GROUP BY v.id) l\\n+JOIN (SELECT first(v.x) AS k FROM unique_list_values_right v CROSS JOIN range(2) t(i) GROUP BY v.id) r ON l.k = r.k;\\n+----\\n+32\\n+\\n+# The ordered rewrite must retain identity on the bound expression only.\\n+query II\\n+EXPLAIN SELECT count(*)\\n+FROM (SELECT first(x ORDER BY p) AS k FROM unique_list_values GROUP BY id) l\\n+JOIN (SELECT last(x ORDER BY p) AS k FROM unique_list_values_right GROUP BY id) r ON l.k = r.k;\\n+----\\n+logical_opt\\t<REGEX>:(?s).*Comparison Join[^~]*~32 rows.*\\n+\\n+query II\\n+EXPLAIN SELECT count(*)\\n+FROM (SELECT any_value(x ORDER BY p) AS k FROM unique_list_values GROUP BY id) l\\n+JOIN (SELECT any_value(x ORDER BY p) AS k FROM unique_list_values_right GROUP BY id) r ON l.k = r.k;\\n+----\\n+logical_opt\\t<REGEX>:(?s).*Comparison Join[^~]*~32 rows.*\\n+\\n+query II\\n+EXPLAIN SELECT count(*)\\n+FROM (SELECT arg_min_null(x, p) AS k FROM unique_list_values GROUP BY id) l\\n+JOIN (SELECT arg_max_null(x, p) AS k FROM unique_list_values_right GROUP BY id) r ON l.k = r.k;\\n+----\\n+logical_opt\\t<REGEX>:(?s).*Comparison Join[^~]*~8 rows.*\\n+\\n+# Every group selects the common value from an otherwise larger input domain.\\n+statement ok\\n+CREATE TABLE biased_nodes AS\\n+SELECT g::INT AS g, p::INT AS p, CASE WHEN p = 0 THEN 0 ELSE 2 * g + 1 END::INT AS k\\n+FROM range(4) g_values(g), range(2) p_values(p);\\n+\\n+statement ok\\n+CREATE TABLE biased_edges AS SELECT ((i % 3) * 3)::INT AS k FROM range(12) t(i);\\n+\\n+statement ok\\n+ANALYZE;\\n+\\n+query II\\n+EXPLAIN SELECT count(*)\\n+FROM (SELECT DISTINCT ON (g) k FROM biased_nodes ORDER BY g, p) s\\n+JOIN biased_edges e ON s.k = e.k;\\n+----\\n+logical_opt\\t<REGEX>:(?s).*Comparison Join[^~]*~16 rows.*\\n+\\n+query I\\n+SELECT count(*)\\n+FROM (SELECT DISTINCT ON (g) k FROM biased_nodes ORDER BY g, p) s\\n+JOIN biased_edges e ON s.k = e.k;\\n+----\\n+16\\n+\\n+# Computed CTE keys remain ineligible even when each group contains one row.\\n+statement ok\\n+CREATE TABLE ids_left AS SELECT i::INT AS id FROM range(8) t(i);\\n+\\n+statement ok\\n+CREATE TABLE ids_right AS SELECT i::INT AS id FROM range(4) t(i);\\n+\\n+statement ok\\n+ANALYZE;\\n+\\n+query II\\n+EXPLAIN (FORMAT JSON)\\n+WITH lhs AS MATERIALIZED (SELECT id, id % 2 AS k FROM ids_left),\\n+     rhs AS MATERIALIZED (SELECT id % 2 AS k FROM ids_right)\\n+SELECT count(*) FROM (SELECT first(k) AS k FROM lhs GROUP BY id) s JOIN rhs r ON s.k = r.k;\\n+----\\n+logical_opt\\t<REGEX>:(?s).*\\\"Join Type\\\": \\\"INNER\\\",\\\\s*\\\"Conditions\\\": \\\"\\\\(k = k\\\\)\\\",\\\\s*\\\"Estimated Cardinality\\\": \\\"8\\\".*\\n+\\n+query I\\n+WITH lhs AS MATERIALIZED (SELECT id, id % 2 AS k FROM ids_left),\\n+     rhs AS MATERIALIZED (SELECT id % 2 AS k FROM ids_right)\\n+SELECT count(*) FROM (SELECT first(k) AS k FROM lhs GROUP BY id) s JOIN rhs r ON s.k = r.k;\\n+----\\n+16\\n+\\n+# Temporary tables support deferred constraints with v1 storage.\\n+# Keep this last so earlier cases can still exercise forced restarts.\\n+statement ok\\n+CREATE TEMP TABLE deferred_unique(id INTEGER PRIMARY KEY DEFERRED, x INTEGER);\\n+\\n+statement ok\\n+INSERT INTO deferred_unique SELECT id, x FROM unique_list_values;\\n+\\n+statement ok\\n+ANALYZE deferred_unique;\\n+\\n+# Deferred keys do not certify singleton groups.\\n+query II\\n+EXPLAIN (FORMAT JSON) SELECT count(*)\\n+FROM (SELECT first(x) AS k FROM deferred_unique GROUP BY id) l\\n+JOIN (SELECT first(x) AS k FROM deferred_unique GROUP BY id) r ON l.k = r.k;\\n+----\\n+logical_opt\\t<REGEX>:(?s).*\\\"Join Type\\\": \\\"INNER\\\",\\\\s*\\\"Conditions\\\": \\\"\\\\(k = k\\\\)\\\",\\\\s*\\\"Estimated Cardinality\\\": \\\"8\\\".*\\n\"}], \"contexts\": {\"incoming\": {\"title\": \"Who reaches this change?\", \"subtitle\": \"Unchanged clients, an unchanged wrapper and the modified implementation.\", \"caveat\": \"External API references are candidates. The wrapper-to-internal leg is syntax-bound.\", \"nodes\": [{\"id\": \"client\", \"title\": \"Relation manager\", \"sub\": \"relation_manager.cpp\", \"state\": \"Unchanged\", \"x\": 20, \"y\": 38, \"evidence\": \"context:255\"}, {\"id\": \"extractor\", \"title\": \"Statistics extractor\", \"sub\": \"relation_statistics_extractor.cpp\", \"state\": \"Unchanged\", \"x\": 20, \"y\": 171, \"evidence\": \"context:257\"}, {\"id\": \"wrapper\", \"title\": \"ExtractAggregationStats\", \"sub\": \"Two overloads inside modified file\", \"state\": \"Unchanged body\", \"x\": 278, \"y\": 102, \"evidence\": \"native:wrapper\"}, {\"id\": \"internal\", \"title\": \"ExtractAggregationStatsInternal\", \"sub\": \"relation_statistics_helper.cpp\", \"state\": \"Modified\", \"x\": 536, \"y\": 102, \"evidence\": \"native:helpers\"}], \"edges\": [{\"a\": \"client\", \"b\": \"wrapper\", \"label\": \"API reference\", \"evidence\": \"context:255\", \"kind\": \"candidate\"}, {\"a\": \"extractor\", \"b\": \"wrapper\", \"label\": \"API reference\", \"evidence\": \"context:257\", \"kind\": \"candidate\"}, {\"a\": \"wrapper\", \"b\": \"internal\", \"label\": \"Calls\", \"evidence\": \"native:wrapper\", \"kind\": \"syntax\"}], \"rows\": [{\"title\": \"Join-order client \\u2192 public wrapper \\u2192 changed internal\", \"body\": \"The first leg references an overloaded API. The second has a retained syntax binding.\", \"evidence\": \"context:255\", \"kind\": \"candidate\"}, {\"title\": \"Unchanged wrapper inside a modified file\", \"body\": \"Identical body bytes; moved from base lines 554 / 562 to head 688 / 696.\", \"evidence\": \"native:wrapper\", \"kind\": \"syntax\"}, {\"title\": \"Changed internal \\u2192 new uniqueness proof\", \"body\": \"Follow the new helper without treating syntax matching as compiler resolution.\", \"evidence\": \"native:helpers\", \"kind\": \"syntax\"}]}, \"dependencies\": {\"title\": \"What does this change depend on?\", \"subtitle\": \"A bounded view of retained syntax bindings from the selected statistics change.\", \"caveat\": \"These are syntax matches. C++ receiver, qualifier and overload resolution still have known gaps.\", \"nodes\": [{\"id\": \"internal\", \"title\": \"ExtractAggregationStatsInternal\", \"sub\": \"Modified statistics implementation\", \"state\": \"Modified\", \"x\": 20, \"y\": 104, \"evidence\": \"native:helpers\"}, {\"id\": \"unique\", \"title\": \"HasUniqueGroups\", \"sub\": \"New singleton-group proof\", \"state\": \"Added\", \"x\": 278, \"y\": 38, \"evidence\": \"native:helpers\"}, {\"id\": \"source\", \"title\": \"HasBaseColumnSource\", \"sub\": \"New base-column proof\", \"state\": \"Added\", \"x\": 278, \"y\": 171, \"evidence\": \"native:HasBaseColumnSource\"}, {\"id\": \"value\", \"title\": \"GetValuePreservingColumn\", \"sub\": \"New expression proof\", \"state\": \"Added\", \"x\": 536, \"y\": 104, \"evidence\": \"native:GetValuePreservingColumn\"}], \"edges\": [{\"a\": \"internal\", \"b\": \"unique\", \"label\": \"Calls\", \"evidence\": \"native:helpers\", \"kind\": \"syntax\"}, {\"a\": \"internal\", \"b\": \"source\", \"label\": \"Calls\", \"evidence\": \"native:HasBaseColumnSource\", \"kind\": \"syntax\"}, {\"a\": \"internal\", \"b\": \"value\", \"label\": \"Calls\", \"evidence\": \"native:GetValuePreservingColumn\", \"kind\": \"syntax\"}], \"rows\": [{\"title\": \"Internal implementation \\u2192 HasUniqueGroups\", \"body\": \"Retained native syntax path. Open its source and binding receipt.\", \"evidence\": \"native:helpers\", \"kind\": \"syntax\"}, {\"title\": \"Statistics reader \\u2192 aggregate identity contract\", \"body\": \"The statistics guard reads the unchanged metadata API.\", \"evidence\": \"context:276\", \"kind\": \"contract\"}, {\"title\": \"Repository dependency binding receipt\", \"body\": \"Cross-file shared Rust graph candidate; function anchor is not an exact call site.\", \"evidence\": \"context:319\", \"kind\": \"syntax\"}]}, \"contract\": {\"title\": \"Which shared contracts deserve review?\", \"subtitle\": \"The changed producer and statistics reader share metadata with unchanged code.\", \"caveat\": \"Shared contract participation does not establish runtime impact on the recursive-CTE consumer.\", \"nodes\": [{\"id\": \"rewrite\", \"title\": \"RewriteOrderedFirst\", \"sub\": \"first_last_any.cpp\", \"state\": \"Modified\", \"x\": 20, \"y\": 38, \"evidence\": \"context:268\"}, {\"id\": \"stats\", \"title\": \"Statistics helper\", \"sub\": \"relation_statistics_helper.cpp\", \"state\": \"Modified\", \"x\": 20, \"y\": 171, \"evidence\": \"context:276\"}, {\"id\": \"identity\", \"title\": \"single_value_identity\", \"sub\": \"aggregate_function.hpp\", \"state\": \"Unchanged\", \"x\": 278, \"y\": 102, \"evidence\": \"context:268\"}, {\"id\": \"cte\", \"title\": \"Recursive CTE consumer\", \"sub\": \"physical_recursive_cte_keyed.cpp\", \"state\": \"Unchanged\", \"x\": 536, \"y\": 102, \"evidence\": \"context:267\"}], \"edges\": [{\"a\": \"rewrite\", \"b\": \"identity\", \"label\": \"Writes\", \"evidence\": \"context:268\", \"kind\": \"contract\"}, {\"a\": \"stats\", \"b\": \"identity\", \"label\": \"Reads\", \"evidence\": \"context:276\", \"kind\": \"contract\"}, {\"a\": \"cte\", \"b\": \"identity\", \"label\": \"Reads\", \"evidence\": \"context:267\", \"kind\": \"contract\"}], \"rows\": [{\"title\": \"Changed rewrite \\u2192 shared identity contract\", \"body\": \"The rewrite propagates HasSingleValueIdentity into the replacement aggregate.\", \"evidence\": \"context:268\", \"kind\": \"contract\"}, {\"title\": \"Unchanged recursive-CTE consumer \\u2192 same contract\", \"body\": \"A concrete review neighbor. Runtime propagation from this PR is not established.\", \"evidence\": \"context:267\", \"kind\": \"contract\"}, {\"title\": \"Unchanged dispatcher \\u2192 possible rewrite callback\", \"body\": \"Inspect one dispatcher and all four registration witnesses.\", \"evidence\": \"context:266\", \"kind\": \"candidate\"}]}, \"config\": {\"title\": \"Where does test selection change?\", \"subtitle\": \"Two unchanged configurations inherit a changed shared skip list.\", \"caveat\": \"Declared exclusions only. This experiment did not execute the DuckDB test suite.\", \"nodes\": [{\"id\": \"disable\", \"title\": \"disable_optimizer.json\", \"sub\": \"Inherits shared list\", \"state\": \"Unchanged\", \"x\": 20, \"y\": 38, \"evidence\": \"context:237\"}, {\"id\": \"serializer\", \"title\": \"verify_serializer.json\", \"sub\": \"Inherits shared list\", \"state\": \"Unchanged\", \"x\": 20, \"y\": 171, \"evidence\": \"context:249\"}, {\"id\": \"shared\", \"title\": \"stats_dependent_tests.json\", \"sub\": \"Shared skip list\", \"state\": \"Modified\", \"x\": 278, \"y\": 102, \"evidence\": \"context:243\"}, {\"id\": \"test\", \"title\": \"distinct_on_stats_propagation.test\", \"sub\": \"New statistics regression\", \"state\": \"Added\", \"x\": 536, \"y\": 102, \"evidence\": \"context:243\"}], \"edges\": [{\"a\": \"disable\", \"b\": \"shared\", \"label\": \"Inherits\", \"evidence\": \"context:237\", \"kind\": \"config\"}, {\"a\": \"serializer\", \"b\": \"shared\", \"label\": \"Inherits\", \"evidence\": \"context:249\", \"kind\": \"config\"}, {\"a\": \"shared\", \"b\": \"test\", \"label\": \"Excludes\", \"evidence\": \"context:243\", \"kind\": \"config\"}], \"rows\": [{\"title\": \"disable_optimizer \\u2192 shared skip list \\u2192 new test\", \"body\": \"Unchanged configuration inherits the newly declared exclusion.\", \"evidence\": \"context:237\", \"kind\": \"config\"}, {\"title\": \"verify_serializer \\u2192 shared skip list \\u2192 new test\", \"body\": \"A second unchanged configuration inherits the exclusion.\", \"evidence\": \"context:249\", \"kind\": \"config\"}, {\"title\": \"verify_aggregate_state_export \\u2192 new test\", \"body\": \"The modified execution configuration directly excludes the test.\", \"evidence\": \"context:246\", \"kind\": \"config\"}]}}, \"evidence\": {\"context:255\": {\"source_path\": \"src/optimizer/join_order/relation_manager.cpp\", \"target_path\": \"src/optimizer/relation_statistics/relation_statistics_helper.cpp\", \"relation\": \"references_aggregate_stats_api\", \"evidence_class\": \"source_reference_candidate\", \"source_changed\": false, \"target_changed\": true, \"witness_path\": \"src/optimizer/join_order/relation_manager.cpp\", \"witness_line\": 623, \"witness_text\": \"auto operator_stats = RelationStatisticsHelper::ExtractAggregationStats(aggr, child_stats, extractor, true);\", \"witness_blob_oid\": \"722f7dbbc7feb7af0ad422d9cc27d496f1dc42ec\", \"revision\": \"d29b26bb92ac940a0807be0f43cc385b31623839\", \"target_api\": \"RelationStatisticsHelper::ExtractAggregationStats\", \"caveat\": \"Source reference; overload/declaration-to-definition binding is not compiler-proven\", \"id\": \"context:255\", \"location\": \"Exact source witness\", \"lines\": [{\"n\": 621, \"text\": \"\\t\\tauto &aggr = op->Cast<LogicalAggregate>();\"}, {\"n\": 622, \"text\": \"\\t\\tRelationStatsExtractor extractor(context, optimizer.GetSourceStats());\"}, {\"n\": 623, \"text\": \"\\t\\tauto operator_stats = RelationStatisticsHelper::ExtractAggregationStats(aggr, child_stats, extractor, true);\"}, {\"n\": 624, \"text\": \"\\t\\tif (!operator_stats) {\"}, {\"n\": 625, \"text\": \"\\t\\t\\tstats_complete = false;\"}, {\"n\": 626, \"text\": \"\\t\\t\\treturn false;\"}], \"artifact\": \"relationships.json\", \"artifact_index\": 255}, \"context:257\": {\"source_path\": \"src/optimizer/relation_statistics/relation_statistics_extractor.cpp\", \"target_path\": \"src/optimizer/relation_statistics/relation_statistics_helper.cpp\", \"relation\": \"references_aggregate_stats_api\", \"evidence_class\": \"source_reference_candidate\", \"source_changed\": false, \"target_changed\": true, \"witness_path\": \"src/optimizer/relation_statistics/relation_statistics_extractor.cpp\", \"witness_line\": 124, \"witness_text\": \"return RelationStatisticsHelper::ExtractAggregationStats(op.Cast<LogicalAggregate>(), child_stats[0], *this);\", \"witness_blob_oid\": \"f8f7a1eda024525a3456505fae6e7b9a82487ac7\", \"revision\": \"d29b26bb92ac940a0807be0f43cc385b31623839\", \"target_api\": \"RelationStatisticsHelper::ExtractAggregationStats\", \"caveat\": \"Source reference; overload/declaration-to-definition binding is not compiler-proven\", \"id\": \"context:257\", \"location\": \"Exact source witness\", \"lines\": [{\"n\": 122, \"text\": \"\\t}\"}, {\"n\": 123, \"text\": \"\\tif (op.type == LogicalOperatorType::LOGICAL_AGGREGATE_AND_GROUP_BY && child_stats.size() == 1) {\"}, {\"n\": 124, \"text\": \"\\t\\treturn RelationStatisticsHelper::ExtractAggregationStats(op.Cast<LogicalAggregate>(), child_stats[0], *this);\"}, {\"n\": 125, \"text\": \"\\t}\"}, {\"n\": 126, \"text\": \"\\treturn RelationStatisticsHelper::ExtractOperatorStats(op, context, child_stats);\"}, {\"n\": 127, \"text\": \"}\"}], \"artifact\": \"relationships.json\", \"artifact_index\": 257}, \"native:wrapper\": {\"id\": \"native:wrapper\", \"relation\": \"Wrapper calls changed internal helper\", \"evidence_class\": \"native_bound_syntax\", \"witness_path\": \"src/optimizer/relation_statistics/relation_statistics_helper.cpp\", \"witness_line\": 693, \"revision\": \"d29b26bb92ac940a0807be0f43cc385b31623839\", \"witness_text\": \"\\t// Join reconstruction has already published the selected input cardinalities.\\n\\tRelationStatsExtractor extractor(context);\\n\\treturn ExtractAggregationStatsInternal(aggregate, child_stats, extractor, true);\\n}\\n\\noptional<RelationStats> RelationStatisticsHelper::ExtractAggregationStats(LogicalAggregate &aggregate,\", \"lines\": [{\"n\": 691, \"text\": \"\\t// Join reconstruction has already published the selected input cardinalities.\"}, {\"n\": 692, \"text\": \"\\tRelationStatsExtractor extractor(context);\"}, {\"n\": 693, \"text\": \"\\treturn ExtractAggregationStatsInternal(aggregate, child_stats, extractor, true);\"}, {\"n\": 694, \"text\": \"}\"}, {\"n\": 695, \"text\": \"\"}, {\"n\": 696, \"text\": \"optional<RelationStats> RelationStatisticsHelper::ExtractAggregationStats(LogicalAggregate &aggregate,\"}], \"caveat\": \"Syntax binding from the retained two-file graph. The wrapper body is unchanged; compiler resolution is not claimed.\", \"artifact\": \"native-incoming-paths.json\", \"artifact_index\": 6, \"location\": \"Source context anchor; exact graph path is in the retained receipt\"}, \"native:helpers\": {\"id\": \"native:helpers\", \"relation\": \"Internal helper calls uniqueness proof\", \"evidence_class\": \"native_bound_syntax\", \"witness_path\": \"src/optimizer/relation_statistics/relation_statistics_helper.cpp\", \"witness_line\": 627, \"revision\": \"d29b26bb92ac940a0807be0f43cc385b31623839\", \"witness_text\": \"\\t\\t\\t}\\n\\t\\t}\\n\\t\\thas_unique_groups = HasUniqueGroups(*aggregate.children[0], groups);\\n\\t\\tif (has_unique_groups) {\\n\\t\\t\\tresult.cardinality = child_stats.cardinality;\\n\\t\\t}\", \"lines\": [{\"n\": 625, \"text\": \"\\t\\t\\t}\"}, {\"n\": 626, \"text\": \"\\t\\t}\"}, {\"n\": 627, \"text\": \"\\t\\thas_unique_groups = HasUniqueGroups(*aggregate.children[0], groups);\"}, {\"n\": 628, \"text\": \"\\t\\tif (has_unique_groups) {\"}, {\"n\": 629, \"text\": \"\\t\\t\\tresult.cardinality = child_stats.cardinality;\"}, {\"n\": 630, \"text\": \"\\t\\t}\"}], \"caveat\": \"Syntax binding, not compiler proof. Open the frozen path record for graph-scoped identities.\", \"artifact\": \"native-incoming-paths.json\", \"artifact_index\": 3, \"location\": \"Source context anchor; exact graph path is in the retained receipt\"}, \"native:HasBaseColumnSource\": {\"id\": \"native:HasBaseColumnSource\", \"relation\": \"Internal implementation calls HasBaseColumnSource\", \"evidence_class\": \"native_bound_syntax\", \"witness_path\": \"src/optimizer/relation_statistics/relation_statistics_helper.cpp\", \"witness_line\": 669, \"revision\": \"d29b26bb92ac940a0807be0f43cc385b31623839\", \"witness_text\": \"\\t\\t\\t\\tauto binding = agg_children[0]->Cast<BoundColumnRefExpression>().Binding();\\n\\t\\t\\t\\tauto child_column = child_columns.Find(binding);\\n\\t\\t\\t\\tif (child_column && HasBaseColumnSource(*aggregate.children[0], binding)) {\\n\\t\\t\\t\\t\\ttotal_domain = child_column->total_domain;\\n\\t\\t\\t\\t\\tcurrent_domain = child_column->current_domain;\\n\\t\\t\\t\\t}\", \"lines\": [{\"n\": 667, \"text\": \"\\t\\t\\t\\tauto binding = agg_children[0]->Cast<BoundColumnRefExpression>().Binding();\"}, {\"n\": 668, \"text\": \"\\t\\t\\t\\tauto child_column = child_columns.Find(binding);\"}, {\"n\": 669, \"text\": \"\\t\\t\\t\\tif (child_column && HasBaseColumnSource(*aggregate.children[0], binding)) {\"}, {\"n\": 670, \"text\": \"\\t\\t\\t\\t\\ttotal_domain = child_column->total_domain;\"}, {\"n\": 671, \"text\": \"\\t\\t\\t\\t\\tcurrent_domain = child_column->current_domain;\"}, {\"n\": 672, \"text\": \"\\t\\t\\t\\t}\"}], \"caveat\": \"Retained syntax binding, not compiler resolution.\", \"artifact\": \"native-incoming-paths.json\", \"artifact_index\": 0, \"location\": \"Source span with retained native path\"}, \"native:GetValuePreservingColumn\": {\"id\": \"native:GetValuePreservingColumn\", \"relation\": \"Internal implementation calls GetValuePreservingColumn\", \"evidence_class\": \"native_bound_syntax\", \"witness_path\": \"src/optimizer/relation_statistics/relation_statistics_helper.cpp\", \"witness_line\": 622, \"revision\": \"d29b26bb92ac940a0807be0f43cc385b31623839\", \"witness_text\": \"\\t\\tcolumn_binding_set_t groups;\\n\\t\\tfor (auto &group : aggregate.groups) {\\n\\t\\t\\tauto binding = GetValuePreservingColumn(*group);\\n\\t\\t\\tif (binding) {\\n\\t\\t\\t\\tgroups.insert(*binding);\\n\\t\\t\\t}\", \"lines\": [{\"n\": 620, \"text\": \"\\t\\tcolumn_binding_set_t groups;\"}, {\"n\": 621, \"text\": \"\\t\\tfor (auto &group : aggregate.groups) {\"}, {\"n\": 622, \"text\": \"\\t\\t\\tauto binding = GetValuePreservingColumn(*group);\"}, {\"n\": 623, \"text\": \"\\t\\t\\tif (binding) {\"}, {\"n\": 624, \"text\": \"\\t\\t\\t\\tgroups.insert(*binding);\"}, {\"n\": 625, \"text\": \"\\t\\t\\t}\"}], \"caveat\": \"Retained syntax binding, not compiler resolution.\", \"artifact\": \"native-incoming-paths.json\", \"artifact_index\": 10, \"location\": \"Source span with retained native path\"}, \"context:268\": {\"source_path\": \"src/function/aggregate/distributive/first_last_any.cpp\", \"target_path\": \"src/include/duckdb/function/aggregate_function.hpp\", \"relation\": \"writes_shared_identity_contract\", \"evidence_class\": \"shared_contract_reference\", \"source_changed\": true, \"target_changed\": false, \"witness_path\": \"src/function/aggregate/distributive/first_last_any.cpp\", \"witness_line\": 57, \"witness_text\": \"result->FunctionMutable().SetSingleValueIdentity(aggregate->Function().HasSingleValueIdentity());\", \"witness_blob_oid\": \"0ba568cb72b05ba00c313dfd8e6d58d158e6b656\", \"revision\": \"d29b26bb92ac940a0807be0f43cc385b31623839\", \"contract\": \"AggregateFunction.properties.single_value_identity\", \"contract_definition\": {\"path\": \"src/include/duckdb/function/aggregate_function.hpp\", \"field_line\": 353, \"reader_line\": 401, \"writer_line\": 402}, \"caveat\": \"Shared contract participation; not a proven runtime flow between these files\", \"id\": \"context:268\", \"location\": \"Exact source witness\", \"lines\": [{\"n\": 55, \"text\": \"\\t                                           aggregate->GetAggregateType());\"}, {\"n\": 56, \"text\": \"\\t// The encoded sort key is non-NULL, so a singleton still returns its input value.\"}, {\"n\": 57, \"text\": \"\\tresult->FunctionMutable().SetSingleValueIdentity(aggregate->Function().HasSingleValueIdentity());\"}, {\"n\": 58, \"text\": \"\\treturn unique_ptr_cast<BoundAggregateExpression, Expression>(std::move(result));\"}, {\"n\": 59, \"text\": \"}\"}, {\"n\": 60, \"text\": \"\"}], \"artifact\": \"relationships.json\", \"artifact_index\": 268}, \"context:276\": {\"source_path\": \"src/optimizer/relation_statistics/relation_statistics_helper.cpp\", \"target_path\": \"src/include/duckdb/function/aggregate_function.hpp\", \"relation\": \"reads_shared_identity_contract\", \"evidence_class\": \"shared_contract_reference\", \"source_changed\": true, \"target_changed\": false, \"witness_path\": \"src/optimizer/relation_statistics/relation_statistics_helper.cpp\", \"witness_line\": 664, \"witness_text\": \"if (bound_aggregate.Function().HasSingleValueIdentity() && !bound_aggregate.GetFilter() &&\", \"witness_blob_oid\": \"6eac019d1e36990aae2d28be02ad8657892cb8d5\", \"revision\": \"d29b26bb92ac940a0807be0f43cc385b31623839\", \"contract\": \"AggregateFunction.properties.single_value_identity\", \"contract_definition\": {\"path\": \"src/include/duckdb/function/aggregate_function.hpp\", \"field_line\": 353, \"reader_line\": 401, \"writer_line\": 402}, \"caveat\": \"Shared contract participation; not a proven runtime flow between these files\", \"id\": \"context:276\", \"location\": \"Exact source witness\", \"lines\": [{\"n\": 662, \"text\": \"\\t\\t\\tauto &bound_aggregate = expression->Cast<BoundAggregateExpression>();\"}, {\"n\": 663, \"text\": \"\\t\\t\\tauto &agg_children = bound_aggregate.GetChildren();\"}, {\"n\": 664, \"text\": \"\\t\\t\\tif (bound_aggregate.Function().HasSingleValueIdentity() && !bound_aggregate.GetFilter() &&\"}, {\"n\": 665, \"text\": \"\\t\\t\\t    !agg_children.empty() && expression->GetReturnType() == agg_children[0]->GetReturnType() &&\"}, {\"n\": 666, \"text\": \"\\t\\t\\t    agg_children[0]->GetExpressionClass() == ExpressionClass::BOUND_COLUMN_REF) {\"}, {\"n\": 667, \"text\": \"\\t\\t\\t\\tauto binding = agg_children[0]->Cast<BoundColumnRefExpression>().Binding();\"}], \"artifact\": \"relationships.json\", \"artifact_index\": 276}, \"context:267\": {\"source_path\": \"src/execution/operator/set/physical_recursive_cte_keyed.cpp\", \"target_path\": \"src/include/duckdb/function/aggregate_function.hpp\", \"relation\": \"reads_shared_identity_contract\", \"evidence_class\": \"shared_contract_reference\", \"source_changed\": false, \"target_changed\": false, \"witness_path\": \"src/execution/operator/set/physical_recursive_cte_keyed.cpp\", \"witness_line\": 42, \"witness_text\": \"aggregate.Function().HasSingleValueIdentity() && !children.empty() &&\", \"witness_blob_oid\": \"69a0f6255be6e0485ba15440a261a67ea3db85e1\", \"revision\": \"d29b26bb92ac940a0807be0f43cc385b31623839\", \"contract\": \"AggregateFunction.properties.single_value_identity\", \"contract_definition\": {\"path\": \"src/include/duckdb/function/aggregate_function.hpp\", \"field_line\": 353, \"reader_line\": 401, \"writer_line\": 402}, \"caveat\": \"Shared contract participation; not a proven runtime flow between these files\", \"id\": \"context:267\", \"location\": \"Exact source witness\", \"lines\": [{\"n\": 40, \"text\": \"\\t\\tauto &children = aggregate.GetChildren();\"}, {\"n\": 41, \"text\": \"\\t\\tresult.can_reuse_new_group_candidates =\"}, {\"n\": 42, \"text\": \"\\t\\t    aggregate.Function().HasSingleValueIdentity() && !children.empty() &&\"}, {\"n\": 43, \"text\": \"\\t\\t    children[0]->GetExpressionClass() == ExpressionClass::BOUND_REF &&\"}, {\"n\": 44, \"text\": \"\\t\\t    children[0]->Cast<BoundReferenceExpression>().Index() == op.payload_idx[payload_idx];\"}, {\"n\": 45, \"text\": \"\\t}\"}], \"artifact\": \"relationships.json\", \"artifact_index\": 267}, \"context:266\": {\"source_path\": \"src/optimizer/aggregate_rewrite.cpp\", \"target_path\": \"src/function/aggregate/distributive/first_last_any.cpp\", \"relation\": \"dispatches_possible_registered_rewrite\", \"evidence_class\": \"callback_dispatch_candidate\", \"source_changed\": false, \"target_changed\": true, \"witness_path\": \"src/optimizer/aggregate_rewrite.cpp\", \"witness_line\": 58, \"witness_text\": \"auto rewrite = input.aggregate.Function().GetDirectRewriteCallback()(input);\", \"witness_blob_oid\": \"862ba08cef7ef508a700d24350465dda41a770f7\", \"revision\": \"d29b26bb92ac940a0807be0f43cc385b31623839\", \"registration_witnesses\": [{\"path\": \"src/function/aggregate/distributive/first_last_any.cpp\", \"line\": 492, \"symbol\": \"RewriteOrderedFirst\", \"text\": \"function.SetDirectRewriteCallback(RewriteOrderedFirst<LAST, SKIP_NULLS>);\"}, {\"path\": \"src/function/aggregate/distributive/first_last_any.cpp\", \"line\": 517, \"symbol\": \"RewriteOrderedFirst\", \"text\": \"function.SetDirectRewriteCallback(RewriteOrderedFirst<LAST, SKIP_NULLS>);\"}, {\"path\": \"src/function/aggregate/distributive/first_last_any.cpp\", \"line\": 542, \"symbol\": \"RewriteOrderedFirst\", \"text\": \"fun.SetDirectRewriteCallback(RewriteOrderedFirst<false, false>);\"}, {\"path\": \"src/function/aggregate/distributive/first_last_any.cpp\", \"line\": 551, \"symbol\": \"RewriteOrderedFirst\", \"text\": \"fun.SetDirectRewriteCallback(RewriteOrderedFirst<true, false>);\"}], \"caveat\": \"Possible callback flow from registration and dispatcher; not a direct bound call or runtime trace\", \"id\": \"context:266\", \"location\": \"Exact source witness\", \"lines\": [{\"n\": 56, \"text\": \"\\t\\treturn nullptr;\"}, {\"n\": 57, \"text\": \"\\t}\"}, {\"n\": 58, \"text\": \"\\tauto rewrite = input.aggregate.Function().GetDirectRewriteCallback()(input);\"}, {\"n\": 59, \"text\": \"\\tif (!rewrite) {\"}, {\"n\": 60, \"text\": \"\\t\\treturn nullptr;\"}, {\"n\": 61, \"text\": \"\\t}\"}], \"artifact\": \"relationships.json\", \"artifact_index\": 266}, \"context:243\": {\"source_path\": \"test/configs/stats_dependent_tests.json\", \"target_path\": \"test/sql/optimizer/statistics/distinct_on_stats_propagation.test\", \"relation\": \"skips_test_in_mode\", \"evidence_class\": \"config_value\", \"source_changed\": true, \"target_changed\": true, \"witness_path\": \"test/configs/stats_dependent_tests.json\", \"witness_line\": 40, \"witness_text\": \"\\\"test/sql/optimizer/statistics/distinct_on_stats_propagation.test\\\",\", \"witness_blob_oid\": \"7457b6bce5591a0984135f9d551b794441f64609\", \"revision\": \"d29b26bb92ac940a0807be0f43cc385b31623839\", \"reason\": \"Statistics are not set without optimizers.\", \"on_init\": null, \"id\": \"context:243\", \"location\": \"Exact source witness\", \"lines\": [{\"n\": 38, \"text\": \"        \\\"test/sql/storage/list_length_stats_nested_optimistic.test\\\",\"}, {\"n\": 39, \"text\": \"        \\\"test/sql/optimizer/statistics/list_length_stats_propagation.test\\\",\"}, {\"n\": 40, \"text\": \"        \\\"test/sql/optimizer/statistics/distinct_on_stats_propagation.test\\\",\"}, {\"n\": 41, \"text\": \"        \\\"test/sql/storage/list_length_stats_old_version.test\\\",\"}, {\"n\": 42, \"text\": \"        \\\"test/sql/types/nested/array/array_statistics.test\\\",\"}, {\"n\": 43, \"text\": \"        \\\"test/sql/vacuum/test_analyze.test\\\",\"}], \"artifact\": \"relationships.json\", \"artifact_index\": 243}, \"context:237\": {\"source_path\": \"test/configs/disable_optimizer.json\", \"target_path\": \"test/configs/stats_dependent_tests.json\", \"relation\": \"inherits_skip_tests\", \"evidence_class\": \"config_value\", \"source_changed\": false, \"target_changed\": true, \"witness_path\": \"test/configs/disable_optimizer.json\", \"witness_line\": 5, \"witness_text\": \"\\\"inherit_skip_tests\\\": \\\"test/configs/stats_dependent_tests.json\\\",\", \"witness_blob_oid\": \"9e2330e8854ff5de38dcdc0a491b5c17885ee6ca\", \"revision\": \"d29b26bb92ac940a0807be0f43cc385b31623839\", \"id\": \"context:237\", \"location\": \"Exact source witness\", \"lines\": [{\"n\": 3, \"text\": \"  \\\"on_init\\\": \\\"SET debug_disable_optimizer=true;\\\",\"}, {\"n\": 4, \"text\": \"  \\\"skip_compiled\\\": \\\"true\\\",\"}, {\"n\": 5, \"text\": \"  \\\"inherit_skip_tests\\\": \\\"test/configs/stats_dependent_tests.json\\\",\"}, {\"n\": 6, \"text\": \"  \\\"skip_tests\\\": [\"}, {\"n\": 7, \"text\": \"    {\"}, {\"n\": 8, \"text\": \"      \\\"reason\\\": \\\"Checks the plan of an executed query through the profiler output.\\\",\"}], \"artifact\": \"relationships.json\", \"artifact_index\": 237}, \"context:249\": {\"source_path\": \"test/configs/verify_serializer.json\", \"target_path\": \"test/configs/stats_dependent_tests.json\", \"relation\": \"inherits_skip_tests\", \"evidence_class\": \"config_value\", \"source_changed\": false, \"target_changed\": true, \"witness_path\": \"test/configs/verify_serializer.json\", \"witness_line\": 5, \"witness_text\": \"\\\"inherit_skip_tests\\\": \\\"test/configs/stats_dependent_tests.json\\\",\", \"witness_blob_oid\": \"e410713085d6a974b1c326d9adda9ba15e7a1104\", \"revision\": \"d29b26bb92ac940a0807be0f43cc385b31623839\", \"id\": \"context:249\", \"location\": \"Exact source witness\", \"lines\": [{\"n\": 3, \"text\": \"  \\\"on_init\\\": \\\"SET debug_verify_serializer=true\\\",\"}, {\"n\": 4, \"text\": \"  \\\"skip_compiled\\\": \\\"true\\\",\"}, {\"n\": 5, \"text\": \"  \\\"inherit_skip_tests\\\": \\\"test/configs/stats_dependent_tests.json\\\",\"}, {\"n\": 6, \"text\": \"  \\\"skip_tests\\\": [\"}, {\"n\": 7, \"text\": \"    {\"}, {\"n\": 8, \"text\": \"      \\\"reason\\\": \\\"Measures CTE restrictions using planner dependencies deliberately discarded by serialization\\\",\"}], \"artifact\": \"relationships.json\", \"artifact_index\": 249}, \"context:246\": {\"source_path\": \"test/configs/verify_aggregate_state_export.json\", \"target_path\": \"test/sql/optimizer/statistics/distinct_on_stats_propagation.test\", \"relation\": \"skips_test_in_mode\", \"evidence_class\": \"config_value\", \"source_changed\": true, \"target_changed\": true, \"witness_path\": \"test/configs/verify_aggregate_state_export.json\", \"witness_line\": 50, \"witness_text\": \"\\\"test/sql/optimizer/statistics/distinct_on_stats_propagation.test\\\",\", \"witness_blob_oid\": \"e59f36a380661403a63cf36bc89c0712b8e00cb3\", \"revision\": \"d29b26bb92ac940a0807be0f43cc385b31623839\", \"reason\": \"Explain/profiling output is different due to verification\", \"on_init\": \"SET debug_verify_aggregate_state_export=true;\", \"id\": \"context:246\", \"location\": \"Exact source witness\", \"lines\": [{\"n\": 48, \"text\": \"        \\\"test/sql/copy/parquet/read_single_parquet_file.test\\\",\"}, {\"n\": 49, \"text\": \"        \\\"test/sql/optimizer/aggregate_stats_propagation.test\\\",\"}, {\"n\": 50, \"text\": \"        \\\"test/sql/optimizer/statistics/distinct_on_stats_propagation.test\\\",\"}, {\"n\": 51, \"text\": \"        \\\"test/sql/optimizer/scalar_aggregate_merge_files_plan.test\\\",\"}, {\"n\": 52, \"text\": \"        \\\"test/sql/optimizer/scalar_aggregate_merge_plan.test\\\",\"}, {\"n\": 53, \"text\": \"        \\\"test/sql/optimizer/scalar_aggregate_merge_tpcds_plan.test\\\",\"}], \"artifact\": \"relationships.json\", \"artifact_index\": 246}, \"context:319\": {\"source_path\": \"src/optimizer/relation_statistics/relation_statistics_helper.cpp\", \"target_path\": \"src/include/duckdb/parser/base_expression.hpp\", \"relation\": \"calls_dependency_in_shared_syntax_graph\", \"evidence_class\": \"native_bound_syntax\", \"source_changed\": true, \"target_changed\": false, \"witness_path\": \"src/optimizer/relation_statistics/relation_statistics_helper.cpp\", \"witness_line\": 558, \"witness_text\": \"src/optimizer/relation_statistics/relation_statistics_helper.cpp :: ExtractAggregationStatsInternal -> src/include/duckdb/parser/base_expression.hpp :: GetExpressionClass [bind=Some(\\\"method_crate_singleton\\\"), cross_file=true]\", \"witness_blob_oid\": \"6eac019d1e36990aae2d28be02ad8657892cb8d5\", \"revision\": \"d29b26bb92ac940a0807be0f43cc385b31623839\", \"source_symbol\": \"ExtractAggregationStatsInternal\", \"target_symbol\": \"GetExpressionClass\", \"changed_symbol_seed\": true, \"bind_method\": \"Some(\\\"method_crate_singleton\\\")\", \"evidence_artifact\": \"docs/research/artifacts/2026-11-17-cpp-extraction-gap-lab/rust-full-repository-seed-witnesses.txt\", \"evidence_artifact_line\": 52, \"witness_location\": \"Core receipt is the binding witness; repository line is function start when unique, not the call site\", \"id\": \"context:319\", \"location\": \"Exact source witness\", \"lines\": [{\"n\": 556, \"text\": \"}\"}, {\"n\": 557, \"text\": \"\"}, {\"n\": 558, \"text\": \"static optional<RelationStats> ExtractAggregationStatsInternal(LogicalAggregate &aggregate,\"}, {\"n\": 559, \"text\": \"                                                               const RelationStats &child_stats,\"}, {\"n\": 560, \"text\": \"                                                               RelationStatsExtractor &extractor,\"}, {\"n\": 561, \"text\": \"                                                               bool use_operator_cardinality) {\"}], \"artifact\": \"relationships.json\", \"artifact_index\": 319}, \"context:114\": {\"source_path\": \"src/function/aggregate/distributive/first_last_any.cpp\", \"target_path\": \"src/function/aggregate/distributive/CMakeLists.txt\", \"relation\": \"listed_in_build_target\", \"evidence_class\": \"build_declaration\", \"source_changed\": true, \"target_changed\": false, \"witness_path\": \"src/function/aggregate/distributive/CMakeLists.txt\", \"witness_line\": 1, \"witness_text\": \"add_library_unity(duckdb_aggr_distr OBJECT count.cpp decimal_avg.cpp\\n                  first_last_any.cpp minmax.cpp)\", \"witness_blob_oid\": \"d7e51e1411270b494afa438d1328aa169ae66ae6\", \"revision\": \"d29b26bb92ac940a0807be0f43cc385b31623839\", \"component\": \"duckdb_aggr_distr\", \"id\": \"context:114\", \"location\": \"Exact source witness\", \"lines\": [{\"n\": 1, \"text\": \"add_library_unity(duckdb_aggr_distr OBJECT count.cpp decimal_avg.cpp\"}, {\"n\": 2, \"text\": \"                  first_last_any.cpp minmax.cpp)\"}, {\"n\": 3, \"text\": \"set(ALL_OBJECT_FILES\"}, {\"n\": 4, \"text\": \"    ${ALL_OBJECT_FILES} $<TARGET_OBJECTS:duckdb_aggr_distr>\"}], \"artifact\": \"relationships.json\", \"artifact_index\": 114}, \"context:157\": {\"source_path\": \"src/optimizer/relation_statistics/relation_statistics_extractor.cpp\", \"target_path\": \"src/optimizer/relation_statistics/CMakeLists.txt\", \"relation\": \"listed_in_build_target\", \"evidence_class\": \"build_declaration\", \"source_changed\": false, \"target_changed\": false, \"witness_path\": \"src/optimizer/relation_statistics/CMakeLists.txt\", \"witness_line\": 1, \"witness_text\": \"add_library_unity(\\n  duckdb_optimizer_relation_statistics\\n  OBJECT\\n  relation_statistics.cpp\\n  relation_statistics_extractor.cpp\\n  relation_statistics_helper.cpp\\n  relation_statistics_operator.cpp\\n  relation_statistics_scan.cpp)\", \"witness_blob_oid\": \"94d04dd6cf092a944afb6ee89389ef475422af69\", \"revision\": \"d29b26bb92ac940a0807be0f43cc385b31623839\", \"component\": \"duckdb_optimizer_relation_statistics\", \"id\": \"context:157\", \"location\": \"Exact source witness\", \"lines\": [{\"n\": 1, \"text\": \"add_library_unity(\"}, {\"n\": 2, \"text\": \"  duckdb_optimizer_relation_statistics\"}, {\"n\": 3, \"text\": \"  OBJECT\"}, {\"n\": 4, \"text\": \"  relation_statistics.cpp\"}], \"artifact\": \"relationships.json\", \"artifact_index\": 157}, \"context:156\": {\"source_path\": \"src/optimizer/join_order/relation_manager.cpp\", \"target_path\": \"src/optimizer/join_order/CMakeLists.txt\", \"relation\": \"listed_in_build_target\", \"evidence_class\": \"build_declaration\", \"source_changed\": false, \"target_changed\": false, \"witness_path\": \"src/optimizer/join_order/CMakeLists.txt\", \"witness_line\": 1, \"witness_text\": \"add_library_unity(\\n  duckdb_optimizer_join_order\\n  OBJECT\\n  query_graph.cpp\\n  filter_info.cpp\\n  join_predicate.cpp\\n  join_relation_set.cpp\\n  join_node.cpp\\n  join_order_operator.cpp\\n  join_order_optimizer.cpp\\n  join_order_source_stats.cpp\\n  cardinality_estimator.cpp\\n  cost_model.cpp\\n  plan_enumerator.cpp\\n  relation_manager.cpp\\n  query_graph_manager.cpp)\", \"witness_blob_oid\": \"94eb6ce5fe48fb20cc162b06b09e39ee44cffcf5\", \"revision\": \"d29b26bb92ac940a0807be0f43cc385b31623839\", \"component\": \"duckdb_optimizer_join_order\", \"id\": \"context:156\", \"location\": \"Exact source witness\", \"lines\": [{\"n\": 1, \"text\": \"add_library_unity(\"}, {\"n\": 2, \"text\": \"  duckdb_optimizer_join_order\"}, {\"n\": 3, \"text\": \"  OBJECT\"}, {\"n\": 4, \"text\": \"  query_graph.cpp\"}], \"artifact\": \"relationships.json\", \"artifact_index\": 156}, \"context:147\": {\"source_path\": \"src/optimizer/aggregate_rewrite.cpp\", \"target_path\": \"src/optimizer/CMakeLists.txt\", \"relation\": \"listed_in_build_target\", \"evidence_class\": \"build_declaration\", \"source_changed\": false, \"target_changed\": false, \"witness_path\": \"src/optimizer/CMakeLists.txt\", \"witness_line\": 10, \"witness_text\": \"add_library_unity(\\n  duckdb_optimizer\\n  OBJECT\\n  aggregate_rewrite.cpp\\n  aggregate_function_rewriter.cpp\\n  aggregate_reuse.cpp\\n  apply_functional_dependencies.cpp\\n  materialized_aggregate_reuse.cpp\\n  aggregate_rewrite_helper.cpp\\n  build_probe_side_optimizer.cpp\\n  column_binding_replacer.cpp\\n  column_lifetime_analyzer.cpp\\n  constant_or_null_simplification.cpp\\n  empty_result_pullup.cpp\\n  common_aggregate_optimizer.cpp\\n  common_subplan_optimizer.cpp\\n  compressed_materialization.cpp\\n  cse_optimizer.cpp\\n  cte_filter_pusher.cpp\\n  cte_join_filter_collector.cpp\\n  cte_join_filter_pusher.cpp\\n  cte_runtime_filter_pushdown.cpp\\n  cte_inlining.cpp\\n  deliminator.cpp\\n  duplicate_eliminated_domain_optimizer.cpp\\n  multi_stage_aggregate_rewriter.cpp\\n  expression_heuristics.cpp\\n  expression_rewriter.cpp\\n  filter_statistics.cpp\\n  filter_combiner.cpp\\n  filter_pullup.cpp\\n  filter_pushdown.cpp\\n  grouping_sets_optimizer.cpp\\n  in_clause_rewriter.cpp\\n  join_filter_pushdown_optimizer.cpp\\n  runtime_filter_cast.cpp\\n  late_materialization.cpp\\n  late_materialization_helper.cpp\\n  optimizer.cpp\\n  outer_join_simplification.cpp\\n  partial_aggregate_pushdown.cpp\\n  partitioned_execution.cpp\\n  join_elimination.cpp\\n  join_build_properties.cpp\\n  recursive_join_planning.cpp\\n  regex_range_filter.cpp\\n  remove_duplicate_groups.cpp\\n  remove_redundant_order.cpp\\n  remove_redundant_order_keys.cpp\\n  remove_unused_columns.cpp\\n  row_group_pruner.cpp\\n  statistics_propagator.cpp\\n  limit_pushdown.cpp\\n  topn_optimizer.cpp\\n  topn_window_elimination.cpp\\n  unnest_rewriter.cpp\\n  window_join_pushdown.cpp\\n  window_self_join.cpp\\n  row_number_rewriter.cpp\\n  sampling_pushdown.cpp\\n  scalar_aggregate_merger.cpp\\n  type_pushdown.cpp\\n  scalar_fn_pushdown.cpp\\n  self_join_aggregate.cpp\\n  projection_pullup.cpp\\n  projection_placement.cpp\\n  remote_pushdown_optimizer.cpp\\n  disjunctive_join_rewriter.cpp)\", \"witness_blob_oid\": \"f408695f8f87cac0b83cf931d5c2db9119b1d1f9\", \"revision\": \"d29b26bb92ac940a0807be0f43cc385b31623839\", \"component\": \"duckdb_optimizer\", \"id\": \"context:147\", \"location\": \"Exact source witness\", \"lines\": [{\"n\": 8, \"text\": \"add_subdirectory(statistics)\"}, {\"n\": 9, \"text\": \"\"}, {\"n\": 10, \"text\": \"add_library_unity(\"}, {\"n\": 11, \"text\": \"  duckdb_optimizer\"}, {\"n\": 12, \"text\": \"  OBJECT\"}, {\"n\": 13, \"text\": \"  aggregate_rewrite.cpp\"}], \"artifact\": \"relationships.json\", \"artifact_index\": 147}, \"context:105\": {\"source_path\": \"src/execution/operator/set/physical_recursive_cte_keyed.cpp\", \"target_path\": \"src/execution/operator/set/CMakeLists.txt\", \"relation\": \"listed_in_build_target\", \"evidence_class\": \"build_declaration\", \"source_changed\": false, \"target_changed\": false, \"witness_path\": \"src/execution/operator/set/CMakeLists.txt\", \"witness_line\": 1, \"witness_text\": \"add_library_unity(\\n  duckdb_operator_set\\n  OBJECT\\n  physical_union.cpp\\n  physical_recursive_cte.cpp\\n  physical_recursive_cte_delta.cpp\\n  physical_recursive_cte_keyed.cpp\\n  physical_recursive_cte_runtime.cpp\\n  physical_recursive_cte_state.cpp\\n  physical_recursive_cte_union.cpp\\n  physical_cte.cpp)\", \"witness_blob_oid\": \"70700bfd2a47745adbb578f91d761c0e84708379\", \"revision\": \"d29b26bb92ac940a0807be0f43cc385b31623839\", \"component\": \"duckdb_operator_set\", \"id\": \"context:105\", \"location\": \"Exact source witness\", \"lines\": [{\"n\": 1, \"text\": \"add_library_unity(\"}, {\"n\": 2, \"text\": \"  duckdb_operator_set\"}, {\"n\": 3, \"text\": \"  OBJECT\"}, {\"n\": 4, \"text\": \"  physical_union.cpp\"}], \"artifact\": \"relationships.json\", \"artifact_index\": 105}, \"context:221\": {\"source_path\": \"test/optimizer/relation_statistics.cpp\", \"target_path\": \"test/optimizer/CMakeLists.txt\", \"relation\": \"listed_in_build_target\", \"evidence_class\": \"build_declaration\", \"source_changed\": false, \"target_changed\": false, \"witness_path\": \"test/optimizer/CMakeLists.txt\", \"witness_line\": 1, \"witness_text\": \"add_library_unity(\\n  test_optimizer\\n  OBJECT\\n  aggregate_rewrite.cpp\\n  aggregate_statistics.cpp\\n  bound_function_identity.cpp\\n  build_probe_side_optimizer.cpp\\n  column_binding_rewrite.cpp\\n  common_subplan.cpp\\n  common_subplan_nonserializable.cpp\\n  dynamic_filter_groups.cpp\\n  expression_type_serialization.cpp\\n  filter_pushdown_alias.cpp\\n  function_invocation_serialization.cpp\\n  join_filter_optimizer_hooks.cpp\\n  join_order_conflict_detector.cpp\\n  join_order_enumeration.cpp\\n  logical_plan_verification.cpp\\n  multi_file_metadata.cpp\\n  multi_file_repeatability.cpp\\n  relation_statistics.cpp\\n  remove_unused_columns.cpp\\n  runtime_filter_schedule.cpp\\n  source_cardinality_bounds.cpp\\n  unchecked_cast.cpp\\n  union_alls.cpp)\", \"witness_blob_oid\": \"58d6dae3ffd2a0a5a88fd8a1283f355976edded7\", \"revision\": \"d29b26bb92ac940a0807be0f43cc385b31623839\", \"component\": \"test_optimizer\", \"id\": \"context:221\", \"location\": \"Exact source witness\", \"lines\": [{\"n\": 1, \"text\": \"add_library_unity(\"}, {\"n\": 2, \"text\": \"  test_optimizer\"}, {\"n\": 3, \"text\": \"  OBJECT\"}, {\"n\": 4, \"text\": \"  aggregate_rewrite.cpp\"}], \"artifact\": \"relationships.json\", \"artifact_index\": 221}}, \"components\": [{\"target\": \"duckdb_aggr_distr\", \"caption\": \"Aggregate functions\", \"members\": [\"src/function/aggregate/distributive/first_last_any.cpp\", \"src/function/aggregate/distributive/minmax.cpp\"], \"evidence\": \"context:114\"}, {\"target\": \"duckdb_optimizer_relation_statistics\", \"caption\": \"Relation statistics\", \"members\": [\"src/optimizer/relation_statistics/relation_statistics_extractor.cpp\", \"src/optimizer/relation_statistics/relation_statistics_helper.cpp\", \"src/optimizer/relation_statistics/relation_statistics_operator.cpp\"], \"evidence\": \"context:157\"}, {\"target\": \"duckdb_optimizer_join_order\", \"caption\": \"Join ordering\", \"members\": [\"src/optimizer/join_order/relation_manager.cpp\"], \"evidence\": \"context:156\"}, {\"target\": \"duckdb_optimizer\", \"caption\": \"Optimizer\", \"members\": [\"src/optimizer/aggregate_rewrite.cpp\"], \"evidence\": \"context:147\"}, {\"target\": \"duckdb_operator_set\", \"caption\": \"Set execution\", \"members\": [\"src/execution/operator/set/physical_recursive_cte_keyed.cpp\"], \"evidence\": \"context:105\"}, {\"target\": \"test_optimizer\", \"caption\": \"Optimizer tests\", \"members\": [\"test/optimizer/relation_statistics.cpp\"], \"evidence\": \"context:221\"}], \"people\": [{\"file_path\": \"src/function/aggregate/distributive/first_last_any.cpp\", \"author_name\": \"Denis Hirn\", \"base_region_lines\": 38, \"github_login\": \"kryonix\", \"evidence_url\": \"https://github.com/duckdb/duckdb/commit/3a5d0c26c254ee4521b504e03a0f1315b821fcdc\", \"history_commits\": 7, \"role\": \"historical_expertise_candidate\", \"formal_owner_verified\": false}, {\"file_path\": \"src/function/aggregate/distributive/first_last_any.cpp\", \"author_name\": \"Mytherin\", \"base_region_lines\": 2, \"github_login\": null, \"evidence_url\": null, \"history_commits\": 13, \"role\": \"historical_expertise_candidate\", \"formal_owner_verified\": false}, {\"file_path\": \"src/function/aggregate/distributive/first_last_any.cpp\", \"author_name\": \"Pedro Holanda\", \"base_region_lines\": 1, \"github_login\": null, \"evidence_url\": null, \"history_commits\": 1, \"role\": \"historical_expertise_candidate\", \"formal_owner_verified\": false}, {\"file_path\": \"src/function/aggregate/distributive/first_last_any.cpp\", \"author_name\": \"Maxxen\", \"base_region_lines\": 1, \"github_login\": null, \"evidence_url\": null, \"history_commits\": 1, \"role\": \"historical_expertise_candidate\", \"formal_owner_verified\": false}, {\"file_path\": \"src/optimizer/relation_statistics/relation_statistics_helper.cpp\", \"author_name\": \"Laurens Kuiper\", \"base_region_lines\": 108, \"github_login\": \"lnkuiper\", \"evidence_url\": \"https://github.com/duckdb/duckdb/commit/255d752c99831484aa675d496d3934767c5ac0cb\", \"history_commits\": 22, \"role\": \"historical_expertise_candidate\", \"formal_owner_verified\": false}, {\"file_path\": \"src/optimizer/relation_statistics/relation_statistics_helper.cpp\", \"author_name\": \"xproblematique\", \"base_region_lines\": 5, \"github_login\": null, \"evidence_url\": null, \"history_commits\": 1, \"role\": \"historical_expertise_candidate\", \"formal_owner_verified\": false}, {\"file_path\": \"src/optimizer/relation_statistics/relation_statistics_helper.cpp\", \"author_name\": \"Robinho-MR\", \"base_region_lines\": 3, \"github_login\": null, \"evidence_url\": null, \"history_commits\": 3, \"role\": \"historical_expertise_candidate\", \"formal_owner_verified\": false}], \"input_refs\": [\"2026-11-18-pr-repository-context-map/relationships.json\", \"2026-11-18-pr-repository-context-map/native-incoming-paths.json\", \"2026-11-17-review-extensions-poc/stakeholder-candidates.json\"]};\nfunction App(){\n const [s,set]=useState(initial());\n const act=(type,value)=>set(old=>{\n  let next={...old,notice:''};\n  if(type==='file'){\n   next.file=value;\n   next.symbol=d.files.find(f=>f.id===value).symbols[0]||'';\n   next.question=value==='stats'?'incoming':value==='rewrite'?'contract':'config';\n   next.evidence=d.contexts[next.question].rows[0].evidence;\n   next.lens='Changes + context'; next.mode='map';\n  }else if(type==='symbol'){next.symbol=value;next.question=old.file==='stats'?'dependencies':'contract';next.evidence=value==='HasBaseColumnSource'?'native:HasBaseColumnSource':value==='GetValuePreservingColumn'?'native:GetValuePreservingColumn':d.contexts[next.question].rows[0].evidence;next.notice='Selected '+value;\n  }else if(type==='coverage'){\n   next.notice='Coverage: 18,342 tracked head entries, 17,289 scanned UTF-8 blobs, 1,053 binary metadata entries. The selected context map and C++ linking are partial.';\n  }else if(type==='copy'){\n   const ref='duckdb/duckdb#28731@'+d.head+':'+value;\n   if(navigator.clipboard?.writeText) navigator.clipboard.writeText(ref).catch(()=>{});\n   next.notice='Reference: '+ref;\n  }else {next[type]=value;if(type==='question')next.evidence=d.contexts[value].rows[0].evidence;}\n  return next;\n });\n return h(View,{s,d,act});\n}\nhydrate(h(App),document.getElementById('app'));\ndocument.documentElement.dataset.interactive='ready';\n";
const buildDir=await Deno.makeTempDir({prefix:'spur-review-design-'});
await Deno.writeTextFile(buildDir+'/view.ts',viewSource);
await Deno.writeTextFile(buildDir+'/client.ts',clientSource);
const view=await import('file://'+buildDir+'/view.ts');
const baseline=ssr(h(view.View,{s:view.initial(),d:data}));
const httpPlugin={name:'http',setup(b){
 b.onResolve({filter:/^https?:\/\//},a=>({path:a.path,namespace:'http-url'}));
 b.onResolve({filter:/.*/,namespace:'http-url'},a=>({path:new URL(a.path,a.importer).toString(),namespace:'http-url'}));
 b.onLoad({filter:/.*/,namespace:'http-url'},async a=>{const r=await fetch(a.path);if(!r.ok)throw Error('Build dependency '+r.status);return {contents:await r.text(),loader:'js'};});
}};
const bundle=await esbuild.build({plugins:[httpPlugin],entryPoints:[buildDir+'/client.ts'],bundle:true,minify:true,format:'iife',write:false,legalComments:'none',supported:{'template-literal':false}});
await esbuild.stop();
const doc='<!doctype html><html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>Spur Graph Review | interactive design</title><style>'+css+'</style></head><body><div id="app">'+baseline+'</div><script>'+bundle.outputFiles[0].text.replaceAll('</script','<\/script')+'</script></body></html>';
await Deno.writeTextFile('/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-18-spur-review-ux/prototype.html',doc);
({[Symbol.for('Jupyter.display')](){return {'text/html':doc}}});


<!doctype html> Spur Graph Review | interactive design S Spur Graph Review Review Inbox Repository Interactive design Captured evidence from PR #28731. Navigation is interactive; collection, AI execution and publication are not connected. DUCKDB / PULL REQUEST #28731 Preserve aggregate domains for proven singleton groups 629e395 → d29b26b 5 changed files Partial linking Review packet Changes + context Architecture People AI review Changed files 5 Select a change to explore repository context. relation_statistics_helper.cpp Modified ExtractAggregationStatsInternal HasUniqueGroups HasBaseColumnSource GetValuePreservingColumn first_last_any.cpp Modified stats_dependent_tests.json Modified verify_aggregate_state_export.json Modified distinct_on_stats_propagation.test Added Context includes unchanged files Tracked repository catalog. Semantic linking remains partial. Inspect coverage SELECTED CHANGE ExtractAggregationStatsInternal src/optimizer/relation_statistics/relation_statistics_helper.cpp Modified Context map Path list Source diff Incoming clients Dependencies Shared contracts Tests + config Who reaches this change? Unchanged clients, an unchanged wrapper and the modified implementation. Include candidates Captured PR neighborhood · frozen head API ref. API ref. Calls Relation manager relation_manager.cpp Unchanged Statistics extractor relation_statistics_extractor.cpp Unchanged ExtractAggregationStats Two overloads inside modified file Unchanged body ExtractAggregationStatsInternal relation_statistics_helper.cpp Modified Evidence limit. External API references are candidates. The wrapper-to-internal leg is syntax-bound. Review paths Select a path to inspect its witness Join-order client → public wrapper → changed internal The first leg references an overloaded API. The second has a retained syntax binding. Candidate Unchanged wrapper inside a modified file Identical body bytes; moved from base lines 554 / 562 to head 688 / 696. Syntax-bound Changed internal → new uniqueness proof Follow the new helper without treating syntax matching as compiler resolution. Syntax-bound Evidence Frozen head Syntax-bound Wrapper calls changed internal helper src/optimizer/relation_statistics/relation_statistics_helper.cpp Line 693 Source context anchor; exact graph path is in the retained receipt 691 // Join reconstruction has already published the selected input cardinalities. 
 692 RelationStatsExtractor extractor(context); 
 693 return ExtractAggregationStatsInternal(aggregate, child_stats, extractor, true); 
 694 } 
 695 
 696 optional<RelationStats> RelationStatisticsHelper::ExtractAggregationStats(LogicalAggregate &aggregate, 
 What this establishes Syntax binding from the retained two-file graph. The wrapper body is unchanged; compiler resolution is not claimed. Provenance and query scope Revision d29b26bb92ac940a0807be0f43cc385b31623839 Evidence ID native:wrapper Receipt native-incoming-paths.json [6] Graph identities and source anchors remain scoped to the originating experiment. Copy reference Source and path selection share this review run. Selection persists across review lenses. Solid: syntax / declarations · Dashed: candidates · Dotted: shared contracts Static design preview. Enable active content in the notebook to try navigation and evidence selection.

# Spur Graph Review: interactive notebook app specification

Status: design draft grounded in executed experiments. Native app implementation is a subsequent step. The accompanying interactive artifact demonstrates navigation over captured evidence; it does not collect GitHub data, run an AI review or publish a review.

## Product decision

Spur Graph is a view inside Spur Notebook. Its primary review workflow starts with a PR's changed files and symbols, then explains their relationships to the entire tracked repository. Unchanged code remains visible whenever it supplies a client, contract, callback, test, configuration or component relationship.

The reviewer should be able to answer:

1. What behavior changed, and where should I inspect its effects?
2. Which paths connect this change to surrounding code, and how well established is each connection?
3. Which components and contracts does it cross? Does an explicit architecture policy permit those changes?
4. Who is participating, who has declared responsibility, and who has relevant historical context?
5. What review questions or findings follow from the available evidence?

The default is a focused review workspace. Repository exploration is an explicit scope that can grow from a selection. A whole-repository graph canvas is available as a later exploration mode, without replacing the changed-file review path.

## Grounding and scope

The design uses DuckDB PR #28731, “Preserve aggregate domains for proven singleton groups.” Its captured comparison is base `629e395a854d72dcb1f414e1a0451c68ac0e9f1f` to head `d29b26bb92ac940a0807be0f43cc385b31623839`.

The completed [repository context experiment](../../research/2026-11-18-pr28731-change-context.md) inventories 18,342 tracked head files and establishes five changed files, five semantic function changes, unchanged API clients, unchanged wrapper bodies, callback registration/dispatch candidates, shared aggregate metadata, CMake membership and inherited test exclusions. The context map is selected, typed evidence over the tracked tree. It is not a complete semantic or runtime impact graph.

The [C++ resolution investigation](2026-11-18-cpp-repository-resolution.md) shows that the installed SQL extraction contract remains single-file. A shared Rust graph recovers additional cross-file relationships, but has known qualifier, receiver, shadowing and overload errors. The UX must preserve those limits. A node count or numeric confidence cannot turn syntax matching into compiler proof.

DuckTails, Spur Graph and DuckPGQ loaded together in the [extension POC](../../research/2026-11-17-spur-review-extensions-poc.md). The optional graph engine traverses established tables; it cannot infer relationships missing from extraction. GitHub data is a captured resource snapshot, with incomplete current reviewer/check information. Historical people evidence and AI readiness are supported; an AI review has not been executed.

## People and jobs

| Person | Main job | Success signal |
|---|---|---|
| Contributor | Check consequences before requesting review | Explains affected contracts, callers and test modes with citations |
| Maintainer | Find the risky assumptions efficiently | Opens an exact witness, distinguishes a defect from a question, records a decision |
| New contributor | Learn the architecture around a PR | Moves from a changed function to its build component, clients and shared interfaces |
| Reviewing agent | Retrieve bounded, revision-correct evidence | Gets typed records and source references identical to those shown to the human |

## Information architecture

```text
Spur Notebook
  Notebook | App
  Spur Graph Review
    Repository setup
    GitHub inbox: Pull requests | Issues | Saved reviews
    Review workspace
      Changes + context
      Architecture
      People
      AI review
    Evidence and jobs
```

The notebook owns the app and its evidence pipeline. App mode hides code cells from the review flow. “Open in notebook” selects the producing cell and returns to the same app selection afterward. The Notebook/App switch is host navigation; the review app never draws a second notebook tab bar.

## Journey A: connect and collect

Repository setup is a resumable page, not a modal with many mandatory fields.

1. Choose a local Git directory. Show canonical root, remote candidates, dirty status, shallow state and object availability.
2. Confirm upstream identity and optional fork independently. A remote named `origin` is not assumed to be upstream.
3. Select the saved GitHub connection (`@github-pat` in this workspace). Keep only its reference in the browser. Public/offline/cached use remains possible.
4. Validate the actual DuckDB runtime and the three provider probes. Show each provider's status and its consequence, such as “History unavailable” or “SQL path queries available.” Advanced details contain versions and receipts.
5. Choose index scope. The tracked-file inventory is the context baseline; language extraction, binary metadata, generated/vendor exclusions and missing submodules are separate coverage categories. Show excluded paths before collection.
6. Save configuration, then explicitly start collection or open the captured example. Saving does not switch branches, stash local edits or fetch all history.

The PR inbox supports search, state/author/label filters, Issues/PRs tabs, a pasted URL and saved review runs. Each row shows title, number, author, collection time and completeness. Missing review/check fields read “Not collected”; a successful empty collection reads “None.” Cached rows remain readable during refresh.

Opening a PR resolves target base, merge base and head, then freezes source blobs and GitHub resources into a review run. Default comparison is merge-base to head. Working-tree review, target-branch drift and an experimental merge-result comparison are separately named contexts.

## Journey B: understand a change in context

### Workspace layout

At a wide desktop viewport, use three panes below a persistent PR header and lens tabs:

```text
duckdb / #28731   Preserve aggregate domains...      [Snapshot] [Review packet]
629e395 -> d29b26b  Captured source   Partial linking  New revision if available
[Changes + context] [Architecture] [People] [AI review]

Changed files + symbols | Selected change and question              | Evidence
                       | [Context map] [Paths] [Source / diff]      | Relation, status
5 file entries         | [Incoming clients] [Dependencies]         | Exact source
expand to 5 functions  | [Shared contracts] [Tests + config]        | Revision and scope
                       |                                           |
unchanged context in   | Small typed neighborhood around selection | Why related?
separate result list   | Ordered paths below map                    | Limits / next step
```

The header is compact. Counts establish PR scope, not a dashboard of graph statistics. No risk score, centrality leaderboard or percentage-complete blast radius appears without an agreed meaning and measurement.

The changed list groups code and tests/configuration, with path suffixes and change badges. Files expand to normalized symbols. A modified file can contain an unchanged function: the function badge must say “Unchanged body” even when its file says “Modified.” Rename, deletion, duplicate/ambiguous identity and unsupported source remain explicit states.

### Selection contract

The shared selection contains repository, review run, comparison side, file identity, optional scoped symbol identity, lens and optional evidence ID. Selection is independent of lens filters and presentation mode.

- Clicking a file selects its change summary and context. Clicking a symbol narrows context without discarding the file.
- Clicking a path, graph edge or node opens its evidence in the right pane. Nodes expose all matching records; a default witness is shown with an “Other witnesses” count.
- Changing a lens retains the selection. People and architecture initially filter to that scope; “Whole PR” is an explicit escape.
- Back/forward restores selection, lens and expansion state within the same run. A refresh never jumps the reviewer to another file.
- Copy reference copies a review-scoped citation. Agent exports use the same identity; raw node IDs never join graphs from different snapshots.
- A node outside the changed files has an “Unchanged context” badge. Opening it keeps the changed seed pinned. “Explore from here” makes a new exploration root and leaves a breadcrumb back to the PR selection.

### Context questions and expansion

Use named questions rather than only a depth slider:

| Control | Meaning |
|---|---|
| Incoming clients | Who references or calls the selected API? Arrows point client to callee |
| Dependencies | What does the selection call/read/use? Arrows point selection to dependency |
| Shared contracts | Who writes/reads/registers the same contract? Participation is not automatically runtime propagation |
| Tests + config | What tests, inherited configuration and build declarations relate to the change? |

Default neighborhood is bounded to two relationships, with a smaller readable subset rendered and the returned total listed. Production node/row budgets are configuration backed by a usability/performance probe, not fixed completeness claims. “Show more” paginates; “Expand from this node” displays the added scope and frontier. Cancellation preserves previously loaded results.

The map uses stable layered positions: upstream clients at left, selected change at center, dependencies/contracts at right. It avoids continuous force motion. A sibling path list is always available and encodes the same relations, direction, uncertainty and selected witness. Filtering candidates removes their paths from both views and reports how many were hidden. It does not upgrade remaining paths to verified runtime behavior.

If no path is found, display “No path found within this query's scope” plus the missing/excluded frontier and available next actions. Never display “No impact” from an empty result.

### Evidence grammar

Change state and relation quality are independent channels. Use text badges, line patterns and icons/shapes in addition to color.

| Evidence | Label in app | Map treatment | Permitted conclusion |
|---|---|---|---|
| Source span changed | Modified / Added / Deleted | Selected node has strong outline; added node has plus badge | Bytes or normalized symbol changed |
| Identical symbol body | Unchanged body | Neutral node, explicit body badge | Body is identical although line/file may change |
| Syntax binder result | Syntax-bound call | Solid arrow, method in inspector | Extractor selected this endpoint; C++ semantics may remain unverified |
| Compiler-backed binding, if later supplied | Compiler-resolved call | Solid arrow plus provider label | Binding under recorded build configuration |
| Textual API reference | Reference candidate | Dashed arrow and “Candidate” text | Source references a name; overload/definition may be unresolved |
| Registration + dispatcher | Possible callback flow | Dashed arrow, both witness groups | A possible flow, not an observed dispatch |
| Shared metadata | Shared contract | Dotted links labeled reads/writes | A review neighborhood around a common contract |
| Parsed config inheritance | Inherits / Excludes test | Solid typed configuration arrow | Effective declared test selection, not an executed test result |
| CMake membership | Build-declared component | Containment or membership edge | Declared target membership, not policy compliance |
| Historical co-change | Historical association | Separate opt-in layer | Past co-editing, not causal dependency |
| Missing linkage | Unresolved frontier | Open-ended stub and readable message | Analysis is incomplete at that frontier |

The inspector shows relation direction, before/head status, method, exact witness file/span and source bytes. Advanced details expose blob hashes, graph identity, query bounds and originating artifacts. A function-start location is labeled as a function anchor, never presented as an exact call site. Syntax-bound paths carry the weakest relevant limitation even when one leg is well established.

## Required PR #28731 interactions

### 1. Unchanged clients through wrappers

Select `ExtractAggregationStatsInternal` in `relation_statistics_helper.cpp`. Incoming context shows unchanged `relation_manager.cpp` and `relation_statistics_extractor.cpp`, plus existing optimizer test references. Their reference-candidate edges reach the public API. The two overloaded wrapper bodies at head lines 688 and 696 are unchanged, although their positions moved. Their syntax-bound links reach the changed internal helper, which calls the new helpers.

The inspector must show the seam: source-reference candidate outside the retained graph, syntax binding inside it. Selecting an unchanged wrapper shows its identical body hashes and before/head positions. The reviewer can compare wrapper behavior without seeing a false modification caused by movement.

### 2. Shared identity contract

Select `RewriteOrderedFirst`, then Shared contracts. Show its setter at line 57, the unchanged `AggregateFunction` contract declaration/read/set methods, the statistics reader at line 664, unchanged `minmax.cpp` producers and the unchanged recursive-CTE consumer at line 42.

The heading reads “Shared contract neighborhood.” Selecting the recursive-CTE consumer explains why it is present and says runtime impact has not been established. The four callback registrations can be opened alongside the unchanged dispatcher at `aggregate_rewrite.cpp:58`; the connection is labeled possible callback flow.

### 3. Non-code propagation

Select the added `distinct_on_stats_propagation.test` or `stats_dependent_tests.json`, then Tests + config. Show:

- The changed shared skip list excludes the new regression.
- Two unchanged configurations, `disable_optimizer.json` and `verify_serializer.json`, inherit the list.
- The changed `verify_aggregate_state_export.json` also excludes it directly.
- The unchanged Makefile lists those three execution/verification configurations; the unchanged loader implements the inheritance.

There are four configuration records: one shared list and three execution/verification configurations. Only two inherit the shared list. Status reads “Declared exclusion; suite not run.” A test result can only replace that statement when a recorded execution exists.

## Journey C: architecture alignment

Architecture presents a component membership table and a small boundary map derived from the same selection. Initial DuckDB components use actual CMake targets: `duckdb_aggr_distr`, `duckdb_optimizer`, `duckdb_optimizer_join_order`, `duckdb_optimizer_relation_statistics`, `duckdb_operator_set` and `test_optimizer`.

Each component row expands to changed and unchanged members plus the build declaration witness. Crossing a contract connects review context; it is not automatically a new dependency or violation.

A separate policy panel reads “Alignment not evaluated: no approved policy.” Later policy setup proposes path/symbol membership, permitted dependency direction, public interfaces and exceptions. Human confirmation versions the policy. A PR cannot silently amend the trusted baseline used to approve itself.

Rule evaluations require rule ID/version, a before/head edge comparison and evidence. Distinguish Aligned, Potential violation, Needs mapping, Exception and Not evaluated. Existing base dependencies are not reported as newly introduced violations. Candidate links produce investigation items rather than definitive violations.

## Journey D: people

People has three independent sections: Current participants, Declared responsibility and Historical expertise. Missing GitHub fields show Unknown with refresh/permission details. A successful CODEOWNERS search with no match says No declared match; missing collection says Not collected.

Captured examples are `lnkuiper` for statistics and `kryonix` for aggregate rewriting. Their evidence comes from base-region blame and sampled file history, not the author's new PR lines. The detail view shows region, denominator, revision, time/sample bounds, commit witnesses and identity mapping. It does not sum incompatible samples into a synthetic ownership score.

Selecting a person highlights their relevant files and suggests review questions. “Draft request” creates an editable local note in a future implementation. Sending or assigning is a separate explicit action. The prototype has no contact action.

## Journey E: AI review and human decisions

The initial AI page says “Not run.” A readiness checklist shows diff, typed paths, architecture mapping/policy, people context, checks and known gaps. Partial evidence is allowed with visible scope and limits.

1. Select Whole PR, Current selection or Component; choose correctness, compatibility, architecture and test focus.
2. Prepare an immutable packet. Show included evidence and omissions before execution.
3. Select a saved provider and run the review explicitly. Record packet hash, model/provider, prompt version, scope and job state.
4. Present defects separately from questions and coverage gaps. Each defect includes trigger, consequence, severity, supporting evidence, counter-evidence, confidence rationale and validation steps. An unsupported citation is rejected or shown as ungrounded.
5. Keep, dismiss with reason or request verification. Dispositions bind to a finding version. Export an editable draft packet; posting to GitHub requires an explicit action and a preview of the actual destination/text.

People evidence helps route a question. It never establishes correctness. A reachable node is not itself a defect. Architecture alignment cannot be approved without a policy. PR prose and source comments are data rather than instructions to the reviewing agent.

For this prototype, the action is “Preview evidence packet.” It opens actual captured source evidence and proposed review questions labeled as design examples. It cannot imply an AI job executed.

## Shared state, jobs and failure recovery

| State/event | Visible behavior | Recovery |
|---|---|---|
| Initial inventory/index | Per-stage progress with usable captured data | Cancel job or open an earlier run |
| Partial extraction/linking | Scope message near context controls; inspectable frontier | Retry failed partition or inspect source |
| New PR head/context | “New revision available”; old run remains fixed | Create new run, compare changes, explicitly carry notes |
| GitHub rate/permission failure | Resource-specific unavailable status; cached records remain | Retry or change saved connection |
| Missing Git object | Name missing revision/object and affected comparison | Offer targeted fetch with declared refs |
| Provider mismatch | Explain disabled capability before load/query | Select correct runtime or retain SQL fallback |
| Canceled/failed AI job | Saved evidence and prior findings remain visible | Retry against same packet or prepare a new one |
| Stale asynchronous response | Result saved under originating run | Never overwrite active selection/run |
| Binary/generated/deleted file | File stays in changed list; source/extraction status explicit | View metadata, before-side source or adjust exclusions |
| Invalid or stale citation | Source unavailable/stale state, no silent relocation | Resolve exact blob or open the referenced run |

Notes and decisions persist separately from analysis results. Refresh does not erase them. Carrying a note to another run requires a stable identity match or human confirmation; a line offset alone is insufficient.

## Human and agent interaction parity

The agent interface offers the same proposed read contracts as the app: list changes, retrieve context by selection/question, retrieve exact evidence, inspect coverage, prepare packet and record a draft decision. These names describe required capabilities, not existing endpoints.

Responses include `review_run_id`, selection identity, typed records, source references, coverage, query bounds, truncation cursor and unresolved frontier. Expansion is bounded and cancelable. Every narrative conclusion can cite an evidence ID that the human opens in the inspector. A human can replay an agent's selection/path without rerunning collection.

Writes use explicit intents with expected run/version. A backend allowlist resolves paths, credentials and provider calls. Browser controls never execute arbitrary SQL/shell or read a local repository directly. Agent publication follows the same review/authorization boundary as human publication.

## Proposed data contracts and provider boundaries

| Contract | Required fields / responsibility |
|---|---|
| ReviewRun | repo/upstream/fork, comparison semantics, target/merge base/head, resource snapshot IDs, source manifests, provider versions, policy version |
| Change | before/head file/blob identity, change status, normalized symbol identity, match method and ambiguity |
| ContextNode | run-scoped identity, symbol/file/component kind, change state, source span, component memberships |
| ContextRelation | source/target, direction, type, evidence class, method, source-side status, witness IDs, limitations |
| ContextQueryResult | root, question, bounds, returned/hidden/truncated counts, cursor, unresolved frontier, coverage |
| Evidence | exact revision/blob, source span/anchor kind, content hash, origin, observed time, artifact/query reference |
| PersonEvidence | role, identity mapping, file/region, base revision, sample/date policy, source witnesses |
| ReviewPacket | run and policy IDs, ordered evidence IDs, scope, omissions, packet hash |
| Finding / Decision | packet/finding version, type, citations/counter-evidence, validation state, human disposition/reason |

Git/GitHub supply identities and resource snapshots. DuckTails supplies diff/history/blame. Spur Graph supplies extraction, matching, diagnostics and whatever binding scope its contract actually reports. A repository linker is still needed to automate reliable cross-file relationships. SQL/DuckPGQ traverse typed edges. An app-layer architecture evaluator supplies versioned membership/policy evaluations. A configured AI provider consumes the review packet. No extension is assigned a capability merely because its name sounds relevant.

## Visual design and interaction details

Design assumption: reuse the existing notebook brief's **Tech Utility** direction. This draft has not been recorded as a user-approved production visual system.

| Token | Proposed value / behavior |
|---|---|
| Canvas / panel / subtle fill | `#f6f7f6` / `#ffffff` / `#eff2ef` |
| Main / secondary text | `#1b2723` / `#53615b` |
| Border / strong border | `#d8dfda` / `#87998f` |
| Action / selected | `#176b48` / pale green `#e6f2e9` |
| Candidate / warning | Dark brown `#805416`, text and dash pattern always accompany it |
| Typography | System sans at 14px body; system mono at 12px source metadata; 22px PR title |
| Spacing | 4px rhythm; 8px compact gap, 12-16px panel padding, 24px major separation |
| Shape | 4-6px control radius, 1px dividers; avoid decorative card grids |
| Motion | Brief selection/expansion transition, no graph drift; honor reduced motion |

The changed seed and primary action use the green accent. Unchanged context is neutral. Semantic added/removed source backgrounds are subdued and carry plus/minus prefixes. Candidate state uses text and line patterns. Main evidence labels are readable at normal zoom; long symbols wrap and full paths are available on focus as well as hover.

### Accessibility and responsive behavior

- All controls have names and visible focus rings. Tabs use roving focus/arrow keys; Enter/Space selects. File/symbol rows follow tree/list keyboard conventions in implementation.
- Path list rows are keyboard reachable; every graph edge and node has an equivalent list action. SVG does not become the only way to retrieve evidence.
- Selection changes announce a concise status through a polite live region. Do not announce every progress tick. Source lines have accessible line numbers and text; evidence order is logical.
- Escape closes a sheet/dialog and restores the invoking control. Opening exact source focuses its heading. Keyboard shortcuts are discoverable and avoid overriding typing fields.
- At wide desktop width show all three panes. At intermediate width collapse the evidence inspector to a named drawer. At narrow width switch between Changes, Context and Evidence as full-width panels; preserve selection and back behavior.
- Production transitions between layouts must use content-fit testing at 200% zoom and long C++ symbols. Graph can scroll within a labeled region; the whole document must not require horizontal scrolling.
- Dark theme follows host tokens in implementation. This prototype covers the light direction only.

## Acceptance journeys

| ID | Given / action | Required result |
|---|---|---|
| UX-01 | Open captured PR | Five files, pinned comparison and partial linking visible; no full-coverage claim |
| UX-02 | Select internal stats function, inspect incoming path | Unchanged caller, unchanged wrapper and changed internal visible; candidate seam labeled |
| UX-03 | Open unchanged wrapper | Identical-body evidence and moved before/head positions; no false modified badge |
| UX-04 | Select rewrite and shared contract | Unchanged declaration and recursive-CTE reader appear with read/write witnesses and explicit runtime limit |
| UX-05 | Inspect callback | Four registrations and dispatcher shown; possible flow label retained |
| UX-06 | Select test/config | Two inherited and one direct execution/verification exclusions, shared list separate, suite not run |
| UX-07 | Open architecture | Actual CMake membership; alignment not evaluated without a policy |
| UX-08 | Open people | Historical candidates separate from unknown assignments/declared ownership |
| UX-09 | Preview AI packet | Same run/evidence; no generated defect or completed AI job implied |
| UX-10 | Change lens then return | File/symbol selection persists; source witness can be reopened |
| UX-11 | Filter candidates / no known path | Hidden candidates explained; absence never becomes no-impact claim |
| UX-12 | Move head, fail collector, receive old response | New run offered; old evidence/notes remain and no mixed snapshot shown |
| UX-13 | Keyboard / narrow / scripts disabled | Readable default design; path list equivalent; no graph-only task |
| UX-14 | Agent sends a citation | Human opens exact run/blob/span; invalid/stale IDs visibly rejected |

## Delivery boundary and next implementation slice

This delivery is the specification, captured-data interactive design artifact, notebook source and verification record. It does not report a native app doctor result or live integration readiness.

The first implementation slice should load a typed captured review run in a native notebook frontend, with shared selection, unchanged-wrapper context, path list and source inspector. Validate this one journey through the actual notebook DAG and doctor before wiring live collectors. Then add config/contract context, architecture and people, followed by explicitly triggered AI review. Reliable C++ repository linking remains a parallel capability requirement; it must not be concealed by a polished graph.


## Design verification and handoff

The final Deno artifact cell executed successfully and saves exactly one `text/html` MIME payload. Its HTML is self-contained and server-rendered. An isolated local Chrome session passed **60 checks** over actual interactions, source provenance, candidate filtering, shared selection, callbacks, CMake details, config inheritance, packet citations, keyboard activation, narrow/intermediate layout and scripts-disabled rendering. Screenshots were visually inspected.

The design fixture contains five changed files, four context questions, 21 source-backed evidence records and six declared components. Presentation is curated from the recorded experiments. The artifact does not run new context queries, collect current GitHub state, evaluate an architecture policy or generate AI findings. No native App mode integration or app doctor result is claimed.

The next native implementation slice is one complete journey: captured typed review run → changed-symbol selection → unchanged-wrapper context → accessible path list → exact source inspector. Then connect live collection and repository linking under the same evidence contract.

Review package: `/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-18-spur-review-ux/`. Standalone preview: `prototype.html`. Verification: `verification.json`.